In [ ]:
#5%-7%

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
import re
from itertools import product
warnings.filterwarnings('ignore')

from scipy import stats
from scipy.optimize import minimize, minimize_scalar, differential_evolution
from scipy.linalg import solve
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, HuberRegressor, Lasso
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.model_selection import TimeSeriesSplit

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import acf, pacf, adfuller
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
from statsmodels.tsa.statespace.kalman_filter import KalmanFilter

try:
    import lightgbm as lgb
    HAS_LGB = True
    print("  ✓ LightGBM tersedia")
except ImportError:
    HAS_LGB = False
    print("  ℹ LightGBM tidak tersedia")

try:
    from prophet import Prophet
    HAS_PROPHET = True
    print("  ✓ Prophet tersedia")
except ImportError:
    HAS_PROPHET = False
    print("  ℹ Prophet tidak tersedia")

plt.rcParams.update({
    'figure.dpi': 100, 'font.family': 'DejaVu Sans',
    'axes.spines.top': False, 'axes.spines.right': False,
})
COLORS = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#3B1F2B', '#44BBA4',
          '#E84855', '#3D405B', '#81B29A', '#F2CC8F']

# ─── Path Data ───────────────────────────────────────────────────────────────
DATA_KLAIM  = r"Data_Klaim (1).csv"
DATA_POLIS  = r"Data_Polis.csv"
DATA_SAMPLE = r"sample_submission.csv"

print("=" * 80)
print("  DSC MCF ITB — INSURANCE CLAIM PREDICTION v22.0")
print("  ROOT CAUSE FIX: Kalman OOS eval pakai phi optimal (bukan default 0.98)")
print("  BASE: v21 | Target: AVG MAPE < 3.5%")
print("=" * 80)


# ========================== HELPER FUNCTIONS =================================

def calc_mape(y_true, y_pred, eps=1.0):
    yt = np.maximum(np.array(y_true, dtype=float), eps)
    yp = np.maximum(np.array(y_pred, dtype=float), 0)
    return float(np.mean(np.abs((yt - yp) / yt)) * 100)

def calc_rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.array(y_true) - np.array(y_pred))**2)))

def safe_kde(ax, data, color, linestyle='-', linewidth=2.0):
    d = np.array(data, dtype=float)
    d = d[np.isfinite(d)]
    if len(d) < 2 or len(np.unique(d)) < 2:
        return
    try:
        kde = stats.gaussian_kde(d)
        xr = np.linspace(d.min(), d.max(), 200)
        ax.plot(xr, kde(xr), color=color, linewidth=linewidth, linestyle=linestyle)
    except:
        pass

def _title(ax, t, fs=11):
    ax.set_title(t, fontsize=fs, fontweight='bold')

def _no_data(ax, msg='Tidak tersedia'):
    ax.text(0.5, 0.5, msg, transform=ax.transAxes,
            ha='center', va='center', fontsize=10, color='gray')

def parse_numeric_safe(series):
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors='coerce')
    def _parse_one(val):
        if pd.isna(val): return np.nan
        s = str(val).strip()
        s = re.sub(r'[^\d.,\-]', '', s)
        if not s: return np.nan
        if ',' in s and '.' in s:
            last_dot, last_comma = s.rfind('.'), s.rfind(',')
            if last_comma > last_dot:
                s = s.replace('.', '').replace(',', '.')
            else:
                s = s.replace(',', '')
        elif ',' in s:
            parts = s.split(',')
            if len(parts) == 2 and len(parts[1]) <= 2:
                s = s.replace(',', '.')
            else:
                s = s.replace(',', '')
        elif '.' in s:
            parts = s.split('.')
            if len(parts) > 2:
                s = s.replace('.', '')
        try: return float(s)
        except: return np.nan
    return series.apply(_parse_one)

def parse_date_safe(series):
    if pd.api.types.is_integer_dtype(series) or pd.api.types.is_float_dtype(series):
        return pd.to_datetime(series.astype(str).str.split('.').str[0],
                              format='%Y%m%d', errors='coerce')
    result = pd.to_datetime(series, errors='coerce', dayfirst=True)
    for fmt in ['%d/%m/%Y', '%m/%d/%Y', '%Y/%m/%d', '%d-%m-%Y',
                '%Y%m%d', '%d %B %Y', '%B %d, %Y', '%Y-%m-%d']:
        missing = result.isna()
        if not missing.any(): break
        try:
            result[missing] = pd.to_datetime(series[missing], format=fmt, errors='coerce')
        except: pass
    return result

def find_col(df, keywords, required=False):
    cols_lower = {c.lower().strip(): c for c in df.columns}
    for kw in keywords:
        kw_l = kw.lower()
        if kw_l in cols_lower: return cols_lower[kw_l]
        for cl, corig in cols_lower.items():
            if kw_l in cl or cl in kw_l: return corig
    if required:
        raise KeyError(f"Kolom tidak ditemukan. Keywords: {keywords}")
    return None

def read_csv_robust(path):
    for enc in ['utf-8', 'utf-8-sig', 'latin-1', 'cp1252', 'iso-8859-1']:
        for sep in [',', ';', '\t', '|']:
            try:
                df = pd.read_csv(path, encoding=enc, sep=sep, low_memory=False)
                if df.shape[1] > 1:
                    df.columns = df.columns.str.strip()
                    print(f"  → '{path}' (encoding={enc}, sep='{sep}')")
                    return df
            except: continue
    raise ValueError(f"Tidak bisa membaca: {path}")


# ========================== [FIX-1] KALMAN FILTER ===========================

class SimpleKalmanFilter:
    """
    [FIX-1] Local Level Kalman Filter (Structural Time Series Model)
    
    State equation:  level_t = level_{t-1} + noise_level ~ N(0, σ²_level)
    Obs equation:    y_t = level_t + noise_obs ~ N(0, σ²_obs)
    
    This is optimal for tracking slowly-changing unobserved level.
    Signal-to-noise ratio q = σ²_level / σ²_obs determines smoothing.
    q→0: static (heavily smoothed), q→∞: noisy/track every point.
    
    Optimal q estimated via MLE on training data.
    """
    def __init__(self, q=0.1):
        self.q = q  # signal-to-noise ratio
        self.level_ = None
        self.P_ = None
        self.fitted_values_ = None
        self.resid_ = None
        
    def fit(self, y, optimize_q=True):
        y = np.array(y, dtype=float)
        n = len(y)
        
        if optimize_q:
            def neg_ll(log_q):
                q_ = np.exp(log_q[0])
                _, ll = self._run_filter(y, q_)
                return -ll
            try:
                res = minimize(neg_ll, [np.log(0.1)], method='Nelder-Mead',
                               options={'xatol': 1e-6, 'fatol': 1e-6, 'maxiter': 200})
                self.q = np.exp(res.x[0])
            except:
                self.q = 0.1
        
        levels, Ps, innov = self._run_filter(y, self.q, return_levels=True)
        self.level_      = levels[-1]
        self.P_          = Ps[-1]
        self.fitted_values_ = levels
        self.resid_      = y - np.array(levels)
        self.y_          = y
        return self
    
    def _run_filter(self, y, q, return_levels=False):
        n = len(y)
        # Initialize
        level = y[0]
        P = 1.0
        
        levels = []
        Ps = []
        log_likelihood = 0.0
        
        for t in range(n):
            # Prediction
            level_pred = level
            P_pred = P + q
            
            # Innovation
            innov = y[t] - level_pred
            S = P_pred + 1.0  # obs noise = 1 (normalized)
            
            # Log-likelihood contribution
            if S > 0:
                log_likelihood += -0.5 * (np.log(2 * np.pi * S) + innov**2 / S)
            
            # Update
            K = P_pred / S  # Kalman gain
            level = level_pred + K * innov
            P = (1 - K) * P_pred
            
            levels.append(level)
            Ps.append(P)
        
        if return_levels:
            return levels, Ps, None
        return None, log_likelihood
    
    def forecast(self, h):
        """Forecast h steps ahead"""
        # Level forecast: constant (local level model)
        # Variance grows linearly with horizon
        forecasts = [self.level_] * h
        return np.array(forecasts)
    
    def forecast_with_trend(self, h, trend_estimate):
        """Forecast with external trend estimate"""
        forecasts = [self.level_ + trend_estimate * i for i in range(1, h+1)]
        return np.array(forecasts)


class KalmanTrendFilter:
    """
    Local Linear Trend Kalman Filter
    State: [level, trend]
    
    Level equation: level_t = level_{t-1} + trend_{t-1} + w1
    Trend equation: trend_t = phi * trend_{t-1} + w2  (phi=damping)
    Obs equation:   y_t = level_t + v
    
    This naturally implements DAMPED TREND forecasting!
    phi < 1 → damped, phi = 1 → random walk trend (Holt linear)
    """
    def __init__(self, phi=0.98, q_level=0.01, q_trend=0.001, q_obs=1.0):
        self.phi = phi
        self.q_level = q_level
        self.q_trend = q_trend
        self.q_obs   = q_obs
        self.level_  = None
        self.trend_  = None
        self.fitted_values_ = None
        
    def fit(self, y, optimize=True):
        y = np.array(y, dtype=float)
        
        if optimize:
            def neg_ll(params):
                phi = np.clip(params[0], 0.85, 1.00)  # [v21] min 0.85, bukan 0.50
                q_l = np.exp(params[1])
                q_t = np.exp(params[2])
                try:
                    ll = self._run_filter_ll(y, phi, q_l, q_t, self.q_obs)
                    return -ll
                except:
                    return 1e10
            
            try:
                res = minimize(
                    neg_ll,
                    [0.95, np.log(0.01), np.log(0.001)],
                    method='Nelder-Mead',
                    options={'xatol': 1e-5, 'fatol': 1e-5, 'maxiter': 500}
                )
                self.phi     = np.clip(res.x[0], 0.85, 1.00)  # [v21] min 0.85
                self.q_level = np.exp(res.x[1])
                self.q_trend = np.exp(res.x[2])
            except:
                pass
        
        levels, trends = self._run_filter(y, self.phi, self.q_level, self.q_trend, self.q_obs)
        self.level_ = levels[-1]
        self.trend_ = trends[-1]
        self.fitted_values_ = np.array(levels)
        self.resid_ = y - self.fitted_values_
        print(f"    [Kalman] phi={self.phi:.3f} q_l={self.q_level:.5f} q_t={self.q_trend:.5f}")
        return self
    
    def _run_filter_ll(self, y, phi, q_l, q_t, q_obs):
        n = len(y)
        # State: [level, trend]
        state = np.array([y[0], 0.0])
        P = np.eye(2) * 10.0
        
        F = np.array([[1, 1], [0, phi]])  # transition
        Q = np.diag([q_l, q_t])           # process noise
        H = np.array([[1, 0]])             # observation
        R = np.array([[q_obs]])            # obs noise
        
        ll = 0.0
        for t in range(n):
            # Predict
            state_pred = F @ state
            P_pred     = F @ P @ F.T + Q
            
            # Innovation
            innov = y[t] - float(H @ state_pred)
            S     = float(H @ P_pred @ H.T + R)
            
            if S <= 0: return -1e10
            ll += -0.5 * (np.log(2*np.pi*S) + innov**2/S)
            
            # Update
            K     = (P_pred @ H.T) / S
            state = state_pred + K.flatten() * innov
            P     = (np.eye(2) - K @ H) @ P_pred
        
        return ll
    
    def _run_filter(self, y, phi, q_l, q_t, q_obs):
        n = len(y)
        state = np.array([y[0], 0.0])
        P = np.eye(2) * 10.0
        
        F = np.array([[1, 1], [0, phi]])
        Q = np.diag([q_l, q_t])
        H = np.array([[1, 0]])
        R = np.array([[q_obs]])
        
        levels = []
        trends = []
        
        for t in range(n):
            state_pred = F @ state
            P_pred     = F @ P @ F.T + Q
            
            innov = y[t] - float(H @ state_pred)
            S     = float(H @ P_pred @ H.T + R)
            
            K = (P_pred @ H.T) / S
            state = state_pred + K.flatten() * innov
            P = (np.eye(2) - K @ H) @ P_pred
            
            levels.append(float(state[0]))
            trends.append(float(state[1]))
        
        return levels, trends
    
    def forecast(self, h):
        """Damped trend forecast h steps ahead"""
        phi   = self.phi
        level = self.level_
        trend = self.trend_
        
        forecasts = []
        curr_level = level
        curr_trend = trend
        
        for i in range(h):
            curr_level = curr_level + curr_trend
            curr_trend = phi * curr_trend  # damping
            forecasts.append(curr_level)
        
        return np.array(forecasts)


# ========================== [FIX-3] EXTERNAL DATA ===========================

def build_external_data(year_start=2024, month_start=1, n_months=None,
                        forecast_months=None, forecast_year=2025):
    """
    [FIX-3] Build external data regressors for Indonesia insurance market.
    
    Based on publicly available knowledge (no internet needed):
    1. Medical inflation Indonesia: ~6-8% annual (BPS/BPJS data)
    2. Ramadan effect: reduced inpatient, increased outpatient
    3. Lebaran effect: spike in claims after Eid (exhaustion effect)
    4. Year-end effect: claim spike in Nov-Dec (deductible exhaustion)
    5. Q1 effect: low claims Jan (reset deductible)
    6. USD/IDR trend: affects imported medical equipment costs
    
    Note: These are PRIOR KNOWLEDGE based features - they don't require
    internet access and are based on well-documented Indonesian healthcare patterns.
    """
    # Generate date range for training + forecast
    all_periods = []
    yr, mo = year_start, month_start
    max_iter = 60
    while max_iter > 0:
        all_periods.append((yr, mo))
        mo += 1
        if mo > 12:
            mo = 1
            yr += 1
        max_iter -= 1
        if n_months and len(all_periods) >= n_months:
            break
    
    if forecast_months:
        for fm in forecast_months:
            all_periods.append((forecast_year, fm))
    
    records = []
    base_inflation = 0.007  # ~8.4% annual medical inflation = 0.7% monthly
    
    # Ramadan/Lebaran calendar 2024-2025 (approximate, well-known dates)
    # Ramadan 2024: March 11 - April 9 → month 3
    # Eid ul-Fitr 2024: April 10 → month 4
    # Ramadan 2025: March 1 - March 30 → month 3
    # Eid ul-Fitr 2025: March 31 → month 3/4
    # Eid ul-Adha 2024: June 17 → month 6
    # Eid ul-Adha 2025: June 7 → month 6
    ramadan_months  = {(2024, 3): 1, (2025, 3): 1}  # Ramadan peak
    eid_months      = {(2024, 4): 1, (2025, 4): 1}  # Post-Eid claim surge
    adha_months     = {(2024, 6): 0.5, (2025, 6): 0.5}
    
    for i, (yr, mo) in enumerate(all_periods):
        t = i  # time index
        
        # Medical inflation index (compound monthly)
        # Jan 2024 = base 1.0
        months_since_start = (yr - year_start) * 12 + (mo - month_start)
        inflation_idx = (1 + base_inflation) ** months_since_start
        
        # Seasonal patterns (Indonesia-specific insurance claims)
        # Based on empirical knowledge of BPJS/private insurance patterns:
        # - Jan: Low (new deductible year starts)
        # - Mar-Apr: Ramadan/Lebaran → mixed effects
        # - Jun: Eid ul-Adha spike
        # - Jul-Ags: Mid-year, school holiday → lower inpatient
        # - Nov-Des: Year-end deductible exhaustion → high claims
        monthly_seasonality = {
            1: 0.85,   # Low - new deductible year
            2: 0.90,   # Slightly low
            3: 1.00,   # Average
            4: 1.05,   # Post-Ramadan spike
            5: 0.95,   # Return to normal
            6: 1.10,   # Eid ul-Adha spike + mid-year
            7: 1.00,   # Normal
            8: 0.95,   # School holiday
            9: 1.00,   # Normal
            10: 1.05,  # Pre-year-end
            11: 1.10,  # Year-end claim rush
            12: 1.15,  # Highest - deductible exhaustion
        }
        
        record = {
            'year': yr,
            'month': mo,
            'time_idx': months_since_start,
            'inflation_idx': inflation_idx,
            'inflation_mom': base_inflation,  # monthly inflation
            
            # Calendar effects
            'is_ramadan':   int((yr, mo) in ramadan_months),
            'is_eid_fitr':  int((yr, mo) in eid_months),
            'is_eid_adha':  int((yr, mo) in adha_months),
            'is_yearend':   int(mo in [11, 12]),
            'is_yearstart': int(mo == 1),
            'is_q1':        int(mo in [1, 2, 3]),
            'is_q4':        int(mo in [10, 11, 12]),
            'is_holiday':   int(mo in [1, 4, 6, 7, 12]),
            
            # Seasonal index (prior knowledge)
            'seasonal_prior': monthly_seasonality.get(mo, 1.0),
            
            # Trend features
            'linear_trend': months_since_start / 12.0,  # years elapsed
            'trend_sq':     (months_since_start / 12.0) ** 2,
            
            # Cyclic encoding
            'sin_month': np.sin(2 * np.pi * mo / 12),
            'cos_month': np.cos(2 * np.pi * mo / 12),
            'sin_qtr':   np.sin(2 * np.pi * ((mo-1)//3 + 1) / 4),
            'cos_qtr':   np.cos(2 * np.pi * ((mo-1)//3 + 1) / 4),
        }
        records.append(record)
    
    df_ext = pd.DataFrame(records)
    return df_ext


# ========================== [FIX-10] OPTIMAL DAMPING ========================

def find_optimal_damping(series, phi_grid=None):
    """
    [FIX-10] Find optimal damping factor via Walk-Forward Cross-Validation.
    
    Damped Holt-Winters: forecast = level + phi/(1-phi) * trend (as h→∞)
    phi=1.0 → no damping (linear extrapolation)
    phi=0.98 → mild damping (industry standard for business forecasting)
    phi=0.80 → strong damping (quickly reverts to mean)
    
    Key insight: For insurance with ~19 months data, phi ≈ 0.90-0.95 is often best.
    """
    if phi_grid is None:
        phi_grid = [0.80, 0.85, 0.88, 0.90, 0.92, 0.95, 0.97, 0.98, 0.99, 1.00]
    
    series = np.array(series, dtype=float)
    n = len(series)
    min_train = max(8, n - 8)
    
    best_phi  = 0.95
    best_mape = np.inf
    phi_mapes = {}
    
    for phi in phi_grid:
        wf_mapes = []
        for test_idx in range(min_train, n):
            y_train = series[:test_idx]
            y_true  = series[test_idx]
            
            if len(y_train) < 6: continue
            
            try:
                if phi < 1.0:
                    model = ExponentialSmoothing(
                        y_train, trend='add', seasonal=None,
                        damped_trend=True, initialization_method='estimated'
                    )
                    fit = model.fit(
                        damping_trend=phi, optimized=True, use_brute=False
                    )
                else:
                    model = ExponentialSmoothing(
                        y_train, trend='add', seasonal=None,
                        damped_trend=False, initialization_method='estimated'
                    )
                    fit = model.fit(optimized=True, use_brute=False)
                
                pred = float(fit.forecast(1).iloc[0])
                pred = max(0, pred)
                wf_mapes.append(calc_mape([y_true], [pred]))
            except:
                continue
        
        if wf_mapes:
            avg_mape = np.mean(wf_mapes)
            phi_mapes[phi] = avg_mape
            if avg_mape < best_mape:
                best_mape = avg_mape
                best_phi  = phi
    
    return best_phi, best_mape, phi_mapes


# ========================== LOAD & PREPROCESS ===============================
print("\n[1/10] LOADING DATA...")

df_klaim  = read_csv_robust(DATA_KLAIM)
df_polis  = read_csv_robust(DATA_POLIS)
df_sample = read_csv_robust(DATA_SAMPLE)

print(f"  ✓ Data Klaim  : {df_klaim.shape[0]:,} × {df_klaim.shape[1]}")
print(f"  ✓ Data Polis  : {df_polis.shape[0]:,} × {df_polis.shape[1]}")
print(f"  ✓ Sample IDs  : {df_sample['id'].tolist()}")

print("\n[2/10] PRE-PROCESSING...")

COL_POLIS_K    = find_col(df_klaim, ['nomor polis', 'nopolis', 'polis_id', 'polis'], required=True)
COL_STATUS     = find_col(df_klaim, ['status klaim', 'status claim', 'status'], required=True)
COL_TARGET     = find_col(df_klaim, ['nominal klaim yang disetujui', 'klaim disetujui',
                                     'nominal disetujui', 'approved claim', 'disetujui'])
COL_TGL_MASUK  = find_col(df_klaim, ['tanggal pasien masuk rs', 'tanggal pasien masuk',
                                     'tgl masuk', 'masuk rs', 'admission date', 'masuk'])
COL_TGL_KELUAR = find_col(df_klaim, ['tanggal pasien keluar rs', 'tanggal pasien keluar',
                                     'tgl keluar', 'keluar rs', 'discharge date', 'keluar'])
COL_TGL_BAYAR  = find_col(df_klaim, ['tanggal pembayaran klaim', 'tanggal pembayaran',
                                     'tgl bayar', 'payment date', 'pembayaran'])
COL_ICD        = find_col(df_klaim, ['icd diagnosis', 'icd_diagnosis', 'icd code', 'icd'])
COL_IP_OP      = find_col(df_klaim, ['inpatient/outpatient', 'rawat inap', 'ip_op', 'rawat'])
COL_LOKASI_RS  = find_col(df_klaim, ['lokasi rs', 'lokasi rumah sakit', 'lokasi'])
COL_BIAYA_RS   = find_col(df_klaim, ['nominal biaya rs yang terjadi', 'nominal biaya rs',
                                     'biaya rs', 'biaya'])
COL_REIMBURSE  = find_col(df_klaim, ['reimburse/cashless', 'reimburse', 'cashless'])

COL_POLIS_P   = find_col(df_polis, ['nomor polis', 'nopolis', 'polis_id', 'polis'], required=True)
COL_TGL_LAHIR = find_col(df_polis, ['tanggal lahir', 'tgl lahir', 'birth date', 'dob'])
COL_TGL_EFF   = find_col(df_polis, ['tanggal efektif polis', 'tanggal efektif', 'efektif'])
COL_PLAN      = find_col(df_polis, ['plan code', 'plancode', 'plan'])
COL_GENDER    = find_col(df_polis, ['gender', 'jenis kelamin', 'sex'])

# Parse numeric & dates
df_klaim[COL_TARGET] = parse_numeric_safe(df_klaim[COL_TARGET])
if COL_BIAYA_RS and COL_BIAYA_RS in df_klaim.columns:
    df_klaim[COL_BIAYA_RS] = parse_numeric_safe(df_klaim[COL_BIAYA_RS])

for col in [c for c in [COL_TGL_MASUK, COL_TGL_KELUAR, COL_TGL_BAYAR]
            if c and c in df_klaim.columns]:
    df_klaim[col] = parse_date_safe(df_klaim[col])

for col in [c for c in [COL_TGL_LAHIR, COL_TGL_EFF]
            if c and c in df_polis.columns]:
    df_polis[col] = parse_date_safe(df_polis[col])

df = df_klaim.merge(df_polis, left_on=COL_POLIS_K, right_on=COL_POLIS_P, how='left')
if COL_POLIS_K != COL_POLIS_P and COL_POLIS_P in df.columns:
    df = df.drop(columns=[COL_POLIS_P])

# Impute payment date
if (COL_TGL_BAYAR and COL_TGL_MASUK and
        COL_TGL_BAYAR in df.columns and COL_TGL_MASUK in df.columns):
    known = df[COL_TGL_BAYAR].notna() & df[COL_TGL_MASUK].notna()
    med_lag = 14
    if known.sum() > 0:
        lag = (df.loc[known, COL_TGL_BAYAR] - df.loc[known, COL_TGL_MASUK]).dt.days
        if not lag.isna().all():
            med_lag = int(np.clip(lag.median(), 0, 90))
    paid_na = df[COL_STATUS].isin(['PAID']) & df[COL_TGL_BAYAR].isna()
    df.loc[paid_na, COL_TGL_BAYAR] = (
        df.loc[paid_na, COL_TGL_MASUK] + pd.Timedelta(days=med_lag))

for col in df.select_dtypes(include=[np.number]).columns:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())
for col in df.select_dtypes(include='object').columns:
    if df[col].isna().any():
        mode_ = df[col].mode()
        df[col] = df[col].fillna(mode_.iloc[0] if not mode_.empty else 'Unknown')

# Filter PAID
status_counts = df[COL_STATUS].value_counts()
PAID_VALUES = [v for v in status_counts.index
               if any(kw in str(v).lower()
                      for kw in ['paid', 'dibayar', 'approved', 'disetujui', 'settled'])]
if not PAID_VALUES:
    PAID_VALUES = [status_counts.index[0]]

df_paid = df[df[COL_STATUS].isin(PAID_VALUES)].copy().reset_index(drop=True)
df_paid[COL_TARGET] = parse_numeric_safe(df_paid[COL_TARGET])

masuk_col = (COL_TGL_MASUK if COL_TGL_MASUK and COL_TGL_MASUK in df_paid.columns
             else COL_TGL_BAYAR)
df_paid[masuk_col] = pd.to_datetime(df_paid[masuk_col], errors='coerce')

valid_dates = df_paid[masuk_col].dropna().sort_values()
if df_paid[masuk_col].isna().any() and len(valid_dates) > 0:
    med_date = valid_dates.iloc[len(valid_dates) // 2]
    df_paid[masuk_col] = df_paid[masuk_col].fillna(med_date)

df_paid['tahun']      = df_paid[masuk_col].dt.year.fillna(2024).astype(int)
df_paid['bulan']      = df_paid[masuk_col].dt.month.fillna(1).astype(int)
df_paid['year_month'] = df_paid[masuk_col].dt.to_period('M')

TARGET_COL = COL_TARGET

# ─── Adaptive Winsorization (from v15 insight) ───────────────────────────────
# Choose best winsorization level based on F×S ratio closest to 1.0
for pct in [(0.01, 0.99), (0.05, 0.95)]:
    col_name = f'target_w{int(pct[1]*100)}'
    df_paid[col_name] = df_paid[TARGET_COL].clip(
        lower=df_paid[TARGET_COL].quantile(pct[0]),
        upper=df_paid[TARGET_COL].quantile(pct[1])
    )

# Quick check: which winsorization gives ratio closest to 1.0
ratios = {}
for win_col in ['target_w99', 'target_w95']:
    tmp = df_paid.groupby('year_month').agg(
        freq=(TARGET_COL, 'count'),
        sev_w=(win_col, 'mean'),
        total=(TARGET_COL, 'sum')
    )
    ratio_ = (tmp['total'] / (tmp['freq'] * tmp['sev_w'] + 1e-10)).median()
    ratios[win_col] = abs(ratio_ - 1.0)

best_win_col = min(ratios, key=ratios.get)
df_paid['target_wins'] = df_paid[best_win_col].copy()
TARGET_WINS_COL = 'target_wins'

print(f"  ✓ df_paid     : {len(df_paid):,} baris")
print(f"  Best winsor.  : {best_win_col} (|ratio-1|={ratios[best_win_col]:.3f})")
print(f"  PAID values   : {PAID_VALUES}")


# ========================== 3. EDA ==========================================
print("\n[3/10] EXPLORATORY DATA ANALYSIS...")

mf_eda = df_paid.groupby('year_month').agg(
    freq     = (TARGET_COL, 'count'),
    sev_mean = (TARGET_COL, 'mean'),
    sev_wins = ('target_wins', 'mean'),
    sev_med  = (TARGET_COL, 'median'),
    total    = (TARGET_COL, 'sum'),
    p25      = (TARGET_COL, lambda x: x.quantile(.25)),
    p75      = (TARGET_COL, lambda x: x.quantile(.75))
).reset_index()
mf_eda['ym_str'] = mf_eda['year_month'].astype(str)
mf_eda['ym_dt']  = pd.to_datetime(mf_eda['ym_str'])

# Key diagnostic from v15 analysis:
ratio_train = (mf_eda['total'] / (mf_eda['freq'] * mf_eda['sev_wins'] + 1e-10)).median()
print(f"  F×S ratio (global)   : {ratio_train:.3f}")

# ─── External Data for EDA period ────────────────────────────────────────────
start_yr = int(str(mf_eda['year_month'].min())[:4])
start_mo = int(str(mf_eda['year_month'].min())[5:7])
df_ext_train = build_external_data(
    year_start=start_yr, month_start=start_mo, n_months=len(mf_eda)
)
print(f"  External data shape  : {df_ext_train.shape}")
print(f"  Inflation idx range  : {df_ext_train['inflation_idx'].min():.3f} - {df_ext_train['inflation_idx'].max():.3f}")

# ─── EDA Dashboard ───────────────────────────────────────────────────────────
fig = plt.figure(figsize=(22, 28))
fig.patch.set_facecolor('#F8F9FA')
fig.suptitle('DSC MCF ITB — Insurance Claim EDA Dashboard v16.0\n'
             'Kalman Filter + Optimal Damping + External Data | Target: <4.0',
             fontsize=14, fontweight='bold', y=0.998)
gs = gridspec.GridSpec(5, 3, figure=fig, hspace=0.60, wspace=0.40)

# Plot 1: Frequency Trend
ax1 = fig.add_subplot(gs[0, :2])
try:
    bar_colors = ['#C73E1D' if '2025' in str(x) else '#2E86AB' for x in mf_eda['year_month']]
    ax1.bar(range(len(mf_eda)), mf_eda['freq'], color=bar_colors, alpha=0.7, edgecolor='white')
    ax1.plot(range(len(mf_eda)), mf_eda['freq'], color='#1a1a2e', lw=2, marker='o', ms=4, zorder=5)
    x_arr = np.arange(len(mf_eda))
    z = np.polyfit(x_arr, mf_eda['freq'], 1)
    ax1.plot(x_arr, np.poly1d(z)(x_arr), 'r--', lw=2, alpha=0.7, label=f'Trend ({z[0]:+.1f}/bln)')
    
    # [v16] Quick Kalman fit
    kf_quick = SimpleKalmanFilter()
    kf_quick.fit(mf_eda['freq'].values, optimize_q=True)
    ax1.plot(range(len(mf_eda)), kf_quick.fitted_values_, color='#F18F01',
             lw=2.5, ls='-.', label=f'Kalman Level (q={kf_quick.q:.3f})')
    
    ax1.set_xticks(range(len(mf_eda)))
    ax1.set_xticklabels(mf_eda['ym_str'], rotation=45, ha='right', fontsize=7.5)
    ax1.set_ylabel('Frekuensi Klaim / Bulan')
    ax1.legend(fontsize=9)
    
    # Annotate last point (key for understanding v15 underforecast)
    last_val = float(mf_eda['freq'].iloc[-1])
    ax1.annotate(f'Last: {last_val:.0f}\n[v15: 198?]',
                 xy=(len(mf_eda)-1, last_val),
                 xytext=(len(mf_eda)-3, last_val+15),
                 arrowprops=dict(arrowstyle='->', color='red'),
                 fontsize=9, color='red')
except Exception as e:
    _no_data(ax1, f'{e}')
_title(ax1, '[v16 KEY] Frekuensi Klaim + Kalman Level  (Merah = 2025)', fs=12)

# Plot 2: ACF/PACF
ax2 = fig.add_subplot(gs[0, 2])
try:
    freq_series = mf_eda['freq'].values
    if len(freq_series) >= 6:
        nlags     = min(len(freq_series) - 2, 15)
        acf_vals  = acf(freq_series, nlags=nlags, fft=True)
        pacf_vals = pacf(freq_series, nlags=min(nlags, len(freq_series)//2-1))
        ax2.bar(range(len(acf_vals)), acf_vals, color=COLORS[1], alpha=0.75, label='ACF')
        ax2.step(range(len(pacf_vals)), pacf_vals, color=COLORS[0], lw=2, label='PACF', where='mid')
        ci = 1.96 / np.sqrt(len(freq_series))
        ax2.axhline(y=ci, color='red', ls='--', lw=1.5, alpha=0.7)
        ax2.axhline(y=-ci, color='red', ls='--', lw=1.5, alpha=0.7)
        ax2.axhline(y=0, color='black', lw=0.8)
        ax2.legend(fontsize=8)
except Exception as e:
    _no_data(ax2, f'{e}')
_title(ax2, 'ACF + PACF (ARIMA Order Guidance)')

# Plot 3: Damping phi analysis (NEW in v16)
ax3 = fig.add_subplot(gs[1, 0])
try:
    phi_grid_eda = [0.80, 0.85, 0.90, 0.92, 0.95, 0.97, 0.98, 0.99, 1.00]
    best_phi_eda, best_mape_phi, phi_mapes = find_optimal_damping(
        mf_eda['freq'].values, phi_grid=phi_grid_eda
    )
    phis = list(phi_mapes.keys())
    mapes_ = [phi_mapes[p] for p in phis]
    best_idx = np.argmin(mapes_)
    bar_colors_phi = ['#C73E1D' if i == best_idx else '#2E86AB' for i in range(len(phis))]
    ax3.bar([str(p) for p in phis], mapes_, color=bar_colors_phi, alpha=0.85)
    ax3.set_ylabel('WF-MAPE (%)')
    ax3.set_xticklabels([str(p) for p in phis], rotation=45, ha='right', fontsize=8)
    ax3.text(0.05, 0.97, f'Optimal φ={best_phi_eda}\nMAPE={best_mape_phi:.2f}%',
             transform=ax3.transAxes, fontsize=9, va='top',
             bbox=dict(boxstyle='round', facecolor='#FFF3CD', alpha=0.9))
    OPTIMAL_PHI_FREQ = best_phi_eda
except Exception as e:
    _no_data(ax3, f'{e}')
    OPTIMAL_PHI_FREQ = 0.95
_title(ax3, '[v16 NEW] Optimal Damping Factor (φ) Search')

# Plot 4: Kalman Filter vs HW comparison
ax4 = fig.add_subplot(gs[1, 1])
try:
    freq_vals = mf_eda['freq'].values
    # Kalman Trend filter
    kf_trend = KalmanTrendFilter()
    kf_trend.fit(freq_vals, optimize=True)
    
    ax4.plot(range(len(mf_eda)), freq_vals, color='gray', lw=1.5, alpha=0.7,
             marker='o', ms=4, label='Actual')
    ax4.plot(range(len(mf_eda)), kf_trend.fitted_values_, color=COLORS[0], lw=2.5,
             label=f'Kalman (φ={kf_trend.phi:.2f})')
    
    # Forecast next 5 months
    fc_kal = kf_trend.forecast(5)
    last_x = len(mf_eda) - 1
    ax4.plot(range(last_x, last_x + 6), 
             [freq_vals[-1]] + fc_kal.tolist(), 
             color=COLORS[0], lw=2.5, ls='--', marker='D', ms=6, alpha=0.8,
             label='Kalman Forecast')
    ax4.set_ylabel('Frekuensi')
    ax4.legend(fontsize=8)
    ax4.text(0.05, 0.05,
             f'[v16] Kalman φ={kf_trend.phi:.2f}\nLevel={kf_trend.level_:.0f} Trend={kf_trend.trend_:.1f}',
             transform=ax4.transAxes, fontsize=8, va='bottom',
             bbox=dict(boxstyle='round', facecolor='#E3F9E3', alpha=0.9))
except Exception as e:
    _no_data(ax4, f'{e}')
_title(ax4, '[v16 NEW] Kalman Trend Filter + Damped Forecast')

# Plot 5: External Data impact
ax5 = fig.add_subplot(gs[1, 2])
try:
    ext_months = df_ext_train['month'].values
    ext_seas   = df_ext_train['seasonal_prior'].values
    ext_infl   = df_ext_train['inflation_idx'].values
    
    ax5a = ax5
    ax5b = ax5a.twinx()
    
    bars5 = ax5a.bar(range(len(mf_eda)), ext_seas[:len(mf_eda)], 
                     color=COLORS[2], alpha=0.5, label='Seasonal prior')
    line5 = ax5b.plot(range(len(mf_eda)), ext_infl[:len(mf_eda)],
                      color=COLORS[3], lw=2.5, label='Inflation idx')
    ax5a.set_ylabel('Seasonal Index', color=COLORS[2])
    ax5b.set_ylabel('Inflation Index', color=COLORS[3])
    ax5a.axhline(y=1.0, color='black', lw=1, ls='--')
    ax5a.set_xticks(range(0, len(mf_eda), 2))
    ax5a.set_xticklabels(mf_eda['ym_str'].iloc[::2], rotation=45, ha='right', fontsize=7)
    lines1, labels1 = ax5a.get_legend_handles_labels()
    lines2, labels2 = ax5b.get_legend_handles_labels()
    ax5a.legend(lines1+lines2, labels1+labels2, fontsize=7)
except Exception as e:
    _no_data(ax5, f'{e}')
_title(ax5, '[v16 NEW] External Data: Inflation + Seasonal Prior')

# Plot 6: Severity comparison
ax6 = fig.add_subplot(gs[2, :2])
try:
    xr = range(len(mf_eda))
    ax6.plot(xr, mf_eda['sev_mean']/1e6, color='#636363', lw=1.5, ls='--',
             marker='o', ms=4, label=f'Mean orig')
    ax6.plot(xr, mf_eda['sev_wins']/1e6, color=COLORS[2], lw=2.5,
             marker='s', ms=5, label=f'Wins ({best_win_col})')
    ax6.plot(xr, mf_eda['sev_med']/1e6, color=COLORS[3], lw=1.5,
             ls=':', marker='^', ms=5, label=f'Median (← v10 problem)')
    
    # Severity floor from v16 analysis
    sev_recent_6m = mf_eda['sev_wins'].tail(6).mean()
    sev_floor = sev_recent_6m * 0.85
    ax6.axhline(y=sev_floor/1e6, color='red', lw=2, ls='--',
                label=f'[v16] Severity floor: {sev_floor/1e6:.1f}M')
    
    ax6.set_xticks(range(len(mf_eda)))
    ax6.set_xticklabels(mf_eda['ym_str'], rotation=45, ha='right', fontsize=7.5)
    ax6.set_ylabel('Severitas (Juta IDR)')
    ax6.legend(fontsize=8)
    ax6.text(0.02, 0.97,
             f'[v16] Severity FLOOR = recent 6m avg × 0.85 = {sev_floor/1e6:.1f}M\n'
             f'Prevents v15 problem: forecast dropping below floor',
             transform=ax6.transAxes, fontsize=8.5, va='top',
             bbox=dict(boxstyle='round', facecolor='#FFE5E5', alpha=0.9))
except Exception as e:
    _no_data(ax6, f'{e}')
_title(ax6, '[v16 KEY] Severity: Adaptive Winsor + Floor Protection', fs=11)

# Plot 7: v15 vs v16 forecast comparison preview (from diagnosis)
ax7 = fig.add_subplot(gs[2, 2])
try:
    months_2025 = [8, 9, 10, 11, 12]
    # v15 predictions (from visualization)
    v15_freq = [198, 190.5, 226.0, 228.1, 201.5]
    # v16 expectation: anchored to Jul 2025 level (~260)
    # With damping: freq should be ~250-260 range
    v16_freq_est = [252, 248, 255, 258, 250]  # placeholder for visualization
    
    ax7.bar([str(m) for m in months_2025], v15_freq, color='#636363', alpha=0.6, label='v15 pred')
    ax7.plot(range(len(months_2025)), v16_freq_est, color=COLORS[0],
             lw=2.5, marker='D', ms=8, label='v16 target range')
    ax7.axhline(y=260, color='red', lw=2, ls='--', label='Jul 2025 actual')
    ax7.axhline(y=np.mean(mf_eda['freq'].tail(6).values), color=COLORS[2],
                lw=1.5, ls=':', label='Recent 6m avg')
    ax7.set_ylabel('Frekuensi Klaim')
    ax7.legend(fontsize=7)
    ax7.text(0.05, 0.97,
             '[v16] Freq should NOT drop\nbelow recent average\nv15 bug: 198 vs 260 last actual',
             transform=ax7.transAxes, fontsize=8, va='top',
             bbox=dict(boxstyle='round', facecolor='#FFE5E5', alpha=0.9))
except Exception as e:
    _no_data(ax7, f'{e}')
_title(ax7, '[v16 DIAG] v15 Frequency Underforecast Fix', fs=9)

# Plot 8: Correlation heatmap
ax8 = fig.add_subplot(gs[3, :2])
try:
    hm_cols = [TARGET_COL]
    if COL_BIAYA_RS and COL_BIAYA_RS in df_paid.columns:
        hm_cols.append(COL_BIAYA_RS)
    if COL_TGL_LAHIR and COL_TGL_LAHIR in df_paid.columns:
        df_paid['usia_plot'] = (
            (pd.to_datetime(df_paid[masuk_col]) -
             pd.to_datetime(df_paid[COL_TGL_LAHIR])).dt.days / 365.25
        ).clip(0, 100)
        hm_cols.append('usia_plot')
    if COL_TGL_KELUAR and COL_TGL_KELUAR in df_paid.columns:
        df_paid['lama_rawat_plot'] = (
            (pd.to_datetime(df_paid[COL_TGL_KELUAR]) -
             pd.to_datetime(df_paid[masuk_col])).dt.days
        ).clip(0, 365).fillna(0)
        hm_cols.append('lama_rawat_plot')
    hm_df = df_paid[[c for c in hm_cols if c in df_paid.columns]].dropna()
    rename_map = {TARGET_COL: 'Klaim', COL_BIAYA_RS: 'Biaya RS',
                  'usia_plot': 'Usia', 'lama_rawat_plot': 'Lama Rawat'}
    hm_df = hm_df.rename(columns={k: v for k, v in rename_map.items() if k in hm_df.columns})
    if hm_df.shape[0] > 1 and hm_df.shape[1] >= 2:
        corr_ = hm_df.corr()
        mask_ = np.triu(np.ones_like(corr_, dtype=bool), k=1)
        sns.heatmap(corr_, annot=True, fmt='.3f', cmap='RdYlGn_r', center=0,
                    ax=ax8, mask=mask_, linewidths=0.8,
                    annot_kws={'size': 11, 'weight': 'bold'})
except Exception as e:
    _no_data(ax8, f'{e}')
_title(ax8, 'Heatmap Korelasi Fitur Numerik', fs=12)

# Plot 9: Per-year pattern
ax9 = fig.add_subplot(gs[3, 2])
try:
    mby = df_paid.groupby(['tahun', 'bulan']).size().reset_index(name='freq')
    mnames = ['J','F','M','A','M','J','J','A','S','O','N','D']
    for i, yr in enumerate(sorted(mby['tahun'].unique())):
        sub_ = mby[mby['tahun'] == yr]
        ax9.plot(sub_['bulan'], sub_['freq'], marker='o', lw=2.5,
                 label=str(yr), color=COLORS[i % len(COLORS)])
    ax9.set_xticks(range(1, 13))
    ax9.set_xticklabels(mnames, fontsize=7.5)
    ax9.legend(fontsize=9)
    ax9.set_ylabel('Frekuensi')
except Exception as e:
    _no_data(ax9, f'{e}')
_title(ax9, 'Pola Musiman per Tahun')

# Plot 10: STL Decomposition
ax10a = fig.add_subplot(gs[4, 0])
ax10b = fig.add_subplot(gs[4, 1])
ax10c = fig.add_subplot(gs[4, 2])
try:
    if len(mf_eda) >= 12:
        freq_ts = pd.Series(mf_eda['freq'].values,
                            index=pd.date_range('2024-01', periods=len(mf_eda), freq='MS'))
        stl_r = STL(freq_ts, period=6, robust=True).fit()
        xr5 = range(len(mf_eda))
        ax10a.plot(xr5, stl_r.trend, color=COLORS[0], lw=2.5, label='STL trend')
        ax10a.plot(xr5, mf_eda['freq'], color='gray', lw=1, alpha=0.5, label='Actual')
        ax10a.legend(fontsize=7)
        ax10b.bar(xr5, stl_r.seasonal, color=COLORS[2], alpha=0.75)
        ax10b.axhline(y=0, color='black', lw=0.8)
        ax10c.scatter(xr5, stl_r.resid, color=COLORS[3], s=40, alpha=0.8)
        ax10c.axhline(y=0, color='black', lw=1)
except Exception as e:
    for a in [ax10a, ax10b, ax10c]:
        _no_data(a, f'{e}')
_title(ax10a, 'STL: Trend')
_title(ax10b, 'STL: Seasonal')
_title(ax10c, 'STL: Residual')

plt.savefig('eda_dashboard_v16.png', dpi=100, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.close()
print("  ✓ eda_dashboard_v16.png tersimpan")


# ========================== 4. FEATURE ENGINEERING ==========================
print("\n[4/10] FEATURE ENGINEERING...")

df_fe = df_paid.copy()
tgl_masuk_s = pd.to_datetime(df_fe[masuk_col], errors='coerce')

# Usia
if COL_TGL_LAHIR and COL_TGL_LAHIR in df_fe.columns:
    df_fe['usia_saat_klaim'] = (
        (tgl_masuk_s - pd.to_datetime(df_fe[COL_TGL_LAHIR], errors='coerce')).dt.days / 365.25
    ).clip(0, 100).fillna(35)
else:
    df_fe['usia_saat_klaim'] = 35.0

# Tenure
if COL_TGL_EFF and COL_TGL_EFF in df_fe.columns:
    df_fe['tenure_polis_tahun'] = (
        (tgl_masuk_s - pd.to_datetime(df_fe[COL_TGL_EFF], errors='coerce')).dt.days / 365.25
    ).clip(0).fillna(1)
else:
    df_fe['tenure_polis_tahun'] = 1.0

# Lama rawat
if COL_TGL_KELUAR and COL_TGL_KELUAR in df_fe.columns:
    df_fe['lama_rawat'] = (
        (pd.to_datetime(df_fe[COL_TGL_KELUAR], errors='coerce') - tgl_masuk_s).dt.days
    ).clip(0, 365).fillna(0)
else:
    df_fe['lama_rawat'] = 1.0

df_fe['bulan_masuk']       = tgl_masuk_s.dt.month.fillna(1).astype(int)
df_fe['kuartal_masuk']     = tgl_masuk_s.dt.quarter.fillna(1).astype(int)
df_fe['hari_dalam_minggu'] = tgl_masuk_s.dt.dayofweek.fillna(0).astype(int)
df_fe['is_weekend']        = df_fe['hari_dalam_minggu'].isin([5, 6]).astype(int)
df_fe['tahun']             = tgl_masuk_s.dt.year.fillna(2024).astype(int)
df_fe['day_of_month']      = tgl_masuk_s.dt.day.fillna(15).astype(int)

# Cyclic
df_fe['sin_month'] = np.sin(2 * np.pi * df_fe['bulan_masuk'] / 12)
df_fe['cos_month'] = np.cos(2 * np.pi * df_fe['bulan_masuk'] / 12)
df_fe['sin_qtr']   = np.sin(2 * np.pi * df_fe['kuartal_masuk'] / 4)
df_fe['cos_qtr']   = np.cos(2 * np.pi * df_fe['kuartal_masuk'] / 4)

# [v16 NEW] External data features per claim
# Attach inflation index per claim month
ext_lookup = df_ext_train.set_index(['year', 'month'])
def get_ext_feat(yr, mo, feat):
    try: return float(ext_lookup.loc[(yr, mo), feat])
    except: return 1.0

for feat_ in ['inflation_idx', 'seasonal_prior', 'is_ramadan', 'is_eid_fitr',
              'is_yearend', 'is_yearstart', 'linear_trend']:
    df_fe[f'ext_{feat_}'] = [
        get_ext_feat(int(r['tahun']), int(r['bulan_masuk']), feat_)
        for _, r in df_fe[['tahun', 'bulan_masuk']].iterrows()
    ]

# IP/OP, Gender
if COL_IP_OP and COL_IP_OP in df_fe.columns:
    df_fe['is_ip'] = df_fe[COL_IP_OP].astype(str).str.lower().str.contains('in').astype(int)
else:
    df_fe['is_ip'] = 0

if COL_GENDER and COL_GENDER in df_fe.columns:
    df_fe['is_female'] = df_fe[COL_GENDER].astype(str).str.upper().str.contains('F').astype(int)
else:
    df_fe['is_female'] = 0

if COL_REIMBURSE and COL_REIMBURSE in df_fe.columns:
    df_fe['is_reimburse'] = df_fe[COL_REIMBURSE].astype(str).str.upper().str.contains('R').astype(int)
else:
    df_fe['is_reimburse'] = 0

# ICD features
global_med = df_fe[TARGET_COL].median()
if pd.isna(global_med) or global_med == 0:
    global_med = 1.0

if COL_ICD and COL_ICD in df_fe.columns:
    df_fe['icd_chapter'] = df_fe[COL_ICD].astype(str).str[0].str.upper().fillna('Z')
    df_fe['icd_3char']   = df_fe[COL_ICD].astype(str).str[:3].str.upper().fillna('Z00')

    med_cost = df_fe.groupby('icd_3char')[TARGET_COL].median()
    freq_icd = df_fe.groupby('icd_3char').size()
    pctiles  = med_cost.quantile([.20, .40, .60, .80])

    def _tier(icd):
        c = med_cost.get(icd, global_med)
        if pd.isna(c): c = global_med
        if c <= pctiles[.20]:   return 0
        elif c <= pctiles[.40]: return 1
        elif c <= pctiles[.60]: return 2
        elif c <= pctiles[.80]: return 3
        return 4

    df_fe['icd_cost_tier']   = df_fe['icd_3char'].map(_tier).fillna(2).astype(int)
    df_fe['icd_freq_rank']   = df_fe['icd_3char'].map(freq_icd).fillna(1).astype(int)
    chapter_sev = {
        'A': 1, 'B': 1, 'C': 5, 'D': 4, 'E': 3, 'F': 2, 'G': 3, 'H': 1,
        'I': 4, 'J': 2, 'K': 2, 'L': 1, 'M': 2, 'N': 2, 'O': 3, 'P': 2,
        'Q': 3, 'R': 1, 'S': 2, 'T': 3, 'Z': 1
    }
    df_fe['icd_disease_sev'] = df_fe['icd_chapter'].map(chapter_sev).fillna(1).astype(int)
    chronic_ch = {'C', 'D', 'E', 'I', 'J', 'K', 'M', 'N'}
    df_fe['is_chronic']      = df_fe['icd_chapter'].isin(chronic_ch).astype(int)
    df_fe['is_catastrophic'] = (df_fe['icd_cost_tier'] == 4).astype(int)

    mean_cost_icd = df_fe.groupby('icd_3char')[TARGET_WINS_COL].mean()
    df_fe['icd_mean_cost']  = df_fe['icd_3char'].map(mean_cost_icd).fillna(global_med)
    df_fe['icd_cost_ratio'] = df_fe['icd_mean_cost'] / (global_med + 1e-10)
else:
    for col_ in ['icd_chapter', 'icd_3char', 'icd_cost_tier', 'icd_disease_sev',
                 'is_chronic', 'is_catastrophic', 'icd_freq_rank', 'icd_mean_cost', 'icd_cost_ratio']:
        df_fe[col_] = 2 if 'cost' in col_ or 'tier' in col_ else (1 if 'sev' in col_ else 0)

if COL_BIAYA_RS and COL_BIAYA_RS in df_fe.columns:
    df_fe['ratio_klaim_biaya'] = (
        df_fe[TARGET_COL] / df_fe[COL_BIAYA_RS].replace(0, np.nan)
    ).clip(0, 5).fillna(0.8)
else:
    df_fe['ratio_klaim_biaya'] = 0.8

if COL_PLAN and COL_PLAN in df_fe.columns:
    plan_sev   = df_fe.groupby(COL_PLAN)[TARGET_COL].median()
    plan_count = df_fe.groupby(COL_PLAN).size()
    df_fe['plan_sev_median'] = df_fe[COL_PLAN].map(plan_sev).fillna(global_med)
    df_fe['plan_count']      = df_fe[COL_PLAN].map(plan_count).fillna(0)
    plan_rank = {'M-001': 3, 'M-002': 2, 'M-003': 1}
    df_fe['plan_rank']       = df_fe[COL_PLAN].map(plan_rank).fillna(1)
else:
    df_fe['plan_sev_median'] = 0
    df_fe['plan_count']      = 0
    df_fe['plan_rank']       = 1

# Interactions
df_fe['usia_x_chronic']  = df_fe['usia_saat_klaim'] * df_fe['is_chronic']
df_fe['usia_x_tier']     = df_fe['usia_saat_klaim'] * df_fe['icd_cost_tier']
df_fe['rawat_x_tier']    = df_fe['lama_rawat']      * df_fe['icd_cost_tier']
df_fe['rawat_x_ip']      = df_fe['lama_rawat']      * df_fe['is_ip']
df_fe['plan_x_tier']     = df_fe['plan_rank']       * df_fe['icd_cost_tier']
df_fe['cost_x_ip']       = df_fe['icd_cost_ratio']  * df_fe['is_ip']
df_fe['cost_x_chronic']  = df_fe['icd_cost_ratio']  * df_fe['is_chronic']

# [v16] Inflation-adjusted severity interaction
df_fe['sev_adj_infl'] = df_fe[TARGET_WINS_COL] / df_fe['ext_inflation_idx']

print(f"  ✓ Features: {df_fe.shape[1]} kolom | Records: {len(df_fe):,}")


# ========================== 5. AGREGASI BULANAN ==============================
print("\n[5/10] AGREGASI BULANAN...")

monthly = df_fe.groupby(['tahun', 'bulan_masuk']).agg(
    frequency     = (TARGET_COL,           'count'),
    severity      = (TARGET_COL,           'mean'),
    severity_w    = (TARGET_WINS_COL,      'mean'),
    severity_med  = (TARGET_COL,           'median'),
    total_claim   = (TARGET_COL,           'sum'),
    usia_mean     = ('usia_saat_klaim',    'mean'),
    tenure_mean   = ('tenure_polis_tahun', 'mean'),
    rawat_mean    = ('lama_rawat',         'mean'),
    pct_chronic   = ('is_chronic',         'mean'),
    pct_catast    = ('is_catastrophic',    'mean'),
    avg_tier      = ('icd_cost_tier',      'mean'),
    avg_dis_sev   = ('icd_disease_sev',    'mean'),
    pct_ip        = ('is_ip',              'mean'),
    pct_female    = ('is_female',          'mean'),
    plan_sev_m    = ('plan_sev_median',    'mean'),
    plan_rank_m   = ('plan_rank',          'mean'),
    ratio_kb_m    = ('ratio_klaim_biaya',  'mean'),
    icd_cost_r    = ('icd_cost_ratio',     'mean'),
    cost_ip_m     = ('cost_x_ip',          'mean'),
    # [v16] External regressors
    ext_inflation = ('ext_inflation_idx',  'mean'),
    ext_seasonal  = ('ext_seasonal_prior', 'mean'),
    ext_ramadan   = ('ext_is_ramadan',     'mean'),
    ext_eid       = ('ext_is_eid_fitr',    'mean'),
    ext_yearend   = ('ext_is_yearend',     'mean'),
    ext_trend     = ('ext_linear_trend',   'mean'),
).reset_index()

monthly['year_month'] = pd.to_datetime(
    monthly['tahun'].astype(str) + '-' +
    monthly['bulan_masuk'].astype(str).str.zfill(2)
)
monthly = monthly.sort_values('year_month').reset_index(drop=True)
monthly['fs_wins']       = monthly['frequency'] * monthly['severity_w']
monthly['fs_correction'] = monthly['total_claim'] / (monthly['fs_wins'] + 1e-10)

# Global correction factor (use median - more robust)
GLOBAL_FS_CORRECTION = float(monthly['fs_correction'].median())
print(f"  Global F×S correction: {GLOBAL_FS_CORRECTION:.3f}")

# Lag features
for lag in [1, 2, 3]:
    monthly[f'mlag_freq_{lag}'] = monthly['frequency'].shift(lag)
    monthly[f'mlag_sevw_{lag}'] = monthly['severity_w'].shift(lag)
monthly['mlag_tot_1']  = monthly['total_claim'].shift(1)
monthly['mlag_tot_2']  = monthly['total_claim'].shift(2)
monthly['mlag_corr_1'] = monthly['fs_correction'].shift(1)

monthly['mroll_freq_3m']  = monthly['frequency'].rolling(3, min_periods=1).mean().shift(1)
monthly['mroll_freq_6m']  = monthly['frequency'].rolling(6, min_periods=1).mean().shift(1)
monthly['mroll_sevw_3m']  = monthly['severity_w'].rolling(3, min_periods=1).mean().shift(1)
monthly['mroll_tot_3m']   = monthly['total_claim'].rolling(3, min_periods=1).sum().shift(1)
monthly['mroll_tot_6m']   = monthly['total_claim'].rolling(6, min_periods=1).sum().shift(1)

monthly['yoy_freq']  = monthly['frequency'].shift(12)
monthly['yoy_sevw']  = monthly['severity_w'].shift(12)
monthly['yoy_total'] = monthly['total_claim'].shift(12)
monthly['yoy_growth_freq']  = (monthly['frequency']   / (monthly['yoy_freq'] + 1e-10) - 1).clip(-1, 5)
monthly['yoy_growth_sevw']  = (monthly['severity_w']  / (monthly['yoy_sevw'] + 1e-10) - 1).clip(-1, 5)
monthly['yoy_growth_total'] = (monthly['total_claim'] / (monthly['yoy_total']+ 1e-10) - 1).clip(-1, 5)

monthly['sin_month'] = np.sin(2 * np.pi * monthly['bulan_masuk'] / 12)
monthly['cos_month'] = np.cos(2 * np.pi * monthly['bulan_masuk'] / 12)
monthly['freq_mom']  = monthly['frequency'].pct_change().clip(-2, 2).shift(1).fillna(0)
monthly['sevw_mom']  = monthly['severity_w'].pct_change().clip(-2, 2).shift(1).fillna(0)

all_lag_cols = [c for c in monthly.columns if any(x in c for x in ['mlag', 'mroll', 'yoy', 'mom'])]
monthly[all_lag_cols] = monthly[all_lag_cols].bfill().ffill().fillna(0)

N = len(monthly)
print(f"  ✓ Monthly data: {N} bulan")
print(f"  frequency  : mean={monthly['frequency'].mean():.1f}, last={monthly['frequency'].iloc[-1]:.0f}")
print(f"  severity_w : mean={monthly['severity_w'].mean()/1e6:.2f}M")
print(f"  total_claim: mean={monthly['total_claim'].mean()/1e9:.3f}B IDR")

# YoY growth rates
yoy_mask = monthly['yoy_freq'].notna() & (monthly['yoy_freq'] > 0)
if yoy_mask.sum() >= 3:
    ACTUAL_GROWTH_FREQ  = float(monthly.loc[yoy_mask, 'yoy_growth_freq'].median())
    ACTUAL_GROWTH_SEV   = float(monthly.loc[yoy_mask, 'yoy_growth_sevw'].median())
    ACTUAL_GROWTH_TOTAL = float(monthly.loc[yoy_mask, 'yoy_growth_total'].median())
else:
    ACTUAL_GROWTH_FREQ  = 0.027
    ACTUAL_GROWTH_SEV   = 0.05
    ACTUAL_GROWTH_TOTAL = 0.10

print(f"  YoY growth: Freq={ACTUAL_GROWTH_FREQ*100:+.1f}% Sev={ACTUAL_GROWTH_SEV*100:+.1f}% "
      f"Total={ACTUAL_GROWTH_TOTAL*100:+.1f}%")

# Historical bounds
HIST_FREQ_P5,  HIST_FREQ_P95  = monthly['frequency'].quantile(0.05), monthly['frequency'].quantile(0.95)
HIST_SEV_P5,   HIST_SEV_P95   = monthly['severity_w'].quantile(0.05), monthly['severity_w'].quantile(0.95)
HIST_TOT_P5,   HIST_TOT_P95   = monthly['total_claim'].quantile(0.05), monthly['total_claim'].quantile(0.95)

# [v16 KEY] Frequency anchor - recent 3-month average
FREQ_RECENT_3M = float(monthly['frequency'].tail(3).mean())
FREQ_RECENT_6M = float(monthly['frequency'].tail(6).mean())
SEV_RECENT_3M  = float(monthly['severity_w'].tail(3).mean())
SEV_RECENT_6M  = float(monthly['severity_w'].tail(6).mean())
TOT_RECENT_3M  = float(monthly['total_claim'].tail(3).mean())
TOT_RECENT_6M  = float(monthly['total_claim'].tail(6).mean())

print(f"\n  [v16] Anchors (from recent data):")
print(f"    Freq recent 3m={FREQ_RECENT_3M:.1f}, 6m={FREQ_RECENT_6M:.1f}")
print(f"    Sev  recent 3m={SEV_RECENT_3M/1e6:.2f}M, 6m={SEV_RECENT_6M/1e6:.2f}M")
print(f"    Total recent 3m={TOT_RECENT_3M/1e9:.3f}B, 6m={TOT_RECENT_6M/1e9:.3f}B")


# ========================== 6. FEATURE IMPORTANCE ===========================
print("\n[6/10] FEATURE IMPORTANCE (v16)...")

FEATS_CANDIDATES = [
    'bulan_masuk', 'tahun', 'sin_month', 'cos_month',
    'usia_mean', 'tenure_mean', 'rawat_mean',
    'pct_chronic', 'pct_catast', 'avg_tier', 'avg_dis_sev', 'pct_ip',
    'pct_female', 'plan_sev_m', 'plan_rank_m', 'ratio_kb_m',
    'icd_cost_r', 'cost_ip_m',
    'mlag_freq_1', 'mlag_freq_2', 'mlag_freq_3',
    'mlag_sevw_1', 'mlag_sevw_2',
    'mlag_tot_1', 'mlag_tot_2', 'mlag_corr_1',
    'mroll_freq_3m', 'mroll_freq_6m',
    'mroll_sevw_3m', 'mroll_tot_3m', 'mroll_tot_6m',
    'freq_mom', 'sevw_mom',
    'yoy_freq', 'yoy_sevw', 'yoy_total',
    # [v16 NEW] External regressors
    'ext_inflation', 'ext_seasonal', 'ext_ramadan', 'ext_eid',
    'ext_yearend', 'ext_trend',
]
FEATS = list(dict.fromkeys([f for f in FEATS_CANDIDATES if f in monthly.columns]))
monthly[FEATS] = monthly[FEATS].fillna(0)

train_data = monthly[
    monthly['frequency'].notna() &
    monthly['severity_w'].notna() &
    monthly['total_claim'].notna()
].copy()
train_data[FEATS] = train_data[FEATS].fillna(0)

print(f"  ✓ Features: {len(FEATS)} | Training: {len(train_data)} bulan")

fi_models = {}
for tgt_name, tgt_col in [
    ('Frequency',  'frequency'),
    ('Severity',   'severity_w'),
    ('Total',      'total_claim')
]:
    try:
        fi_y = train_data[tgt_col].fillna(0).values
        if np.all(fi_y == 0):
            fi_models[tgt_name] = pd.Series(np.ones(len(FEATS))/len(FEATS), index=FEATS)
            continue
        m = RandomForestRegressor(n_estimators=200, max_depth=5,
                                  min_samples_leaf=2, random_state=42, n_jobs=-1)
        m.fit(train_data[FEATS], np.log1p(np.maximum(fi_y, 0)))
        fi_models[tgt_name] = pd.Series(m.feature_importances_, index=FEATS).sort_values(ascending=False)
    except Exception as e:
        print(f"  ⚠ FI {tgt_name}: {e}")
        fi_models[tgt_name] = pd.Series(np.ones(len(FEATS)), index=FEATS)

try:
    fig_fi, axes_fi = plt.subplots(1, 3, figsize=(22, 9))
    fig_fi.suptitle('Feature Importance v16.0 — RandomForest Top 15 per Target\n'
                    '[v16] Kalman + Optimal Damping + External Data',
                    fontsize=13, fontweight='bold')
    for ax, (tgt_name, fi), color in zip(axes_fi, fi_models.items(), COLORS):
        top = fi.head(15)
        max_imp = top.values.max() if top.values.max() > 0 else 1.0
        top_norm = top / max_imp
        yp = range(len(top_norm))
        ax.barh(list(yp), top_norm.values[::-1], color=color, alpha=0.85)
        ax.set_yticks(list(yp))
        ax.set_yticklabels(top_norm.index[::-1], fontsize=9)
        ax.set_title(f'{tgt_name}', fontsize=11, fontweight='bold')
        ax.set_xlabel('Importance (normalized)')
        ax.set_xlim(0, 1.15)
        for bar, val in zip(ax.patches, top_norm.values[::-1]):
            ax.text(val + 0.02, bar.get_y() + bar.get_height()/2,
                    f'{val:.3f}', va='center', fontsize=8)
    plt.tight_layout()
    plt.savefig('feature_importance_v16.png', dpi=100, bbox_inches='tight', facecolor='white')
    plt.close()
    print("  ✓ feature_importance_v16.png tersimpan")
except Exception as e:
    print(f"  ⚠ FI plot: {e}")
    plt.close('all')


# ========================== 7. PEMODELAN v16 ================================
print("\n[7/10] PEMODELAN v16.0...")
print("  [v16] Kalman Trend + Optimal Damping + SARIMAX(exog) + STL + Stacking")
print("  [v16] Frequency ANCHOR + Severity FLOOR + F×S Direct Enforcement")

TARGET_CONFIG = {
    'frequency':   {'col': 'frequency',   'log': False, 'use_exog': True},
    'severity':    {'col': 'severity_w',  'log': True,  'use_exog': True},
    'total_claim': {'col': 'total_claim', 'log': True,  'use_exog': True},
}

# External regressors for training period
EXT_COLS_MONTHLY = ['ext_inflation', 'ext_seasonal', 'ext_ramadan', 'ext_eid',
                    'ext_yearend', 'ext_trend']
EXT_COLS_AVAIL = [c for c in EXT_COLS_MONTHLY if c in monthly.columns]

# Forecast external regressors for 2025 Aug-Dec
PRED_MONTHS_2025 = [8, 9, 10, 11, 12]
df_ext_forecast = build_external_data(
    year_start=2025, month_start=8,
    n_months=5
)
# Fix: only take the first 5 months
df_ext_forecast = df_ext_forecast.head(5)

print(f"  External regressors for forecast: {df_ext_forecast[['year','month']].values.tolist()}")

def find_best_sarimax(series, exog=None, max_p=2, max_d=1, max_q=2):
    """Find best SARIMAX order via AIC"""
    best_aic   = np.inf
    best_order = (1, 1, 1)
    for p, d, q in product(range(max_p+1), range(max_d+1), range(max_q+1)):
        if p == 0 and q == 0: continue
        try:
            model = SARIMAX(
                series, exog=exog, order=(p, d, q),
                seasonal_order=(0,0,0,0),
                enforce_stationarity=False, enforce_invertibility=False
            )
            result = model.fit(disp=False, maxiter=100)
            if result.aic < best_aic:
                best_aic   = result.aic
                best_order = (p, d, q)
        except: continue
    return best_order, best_aic


# ─── Walk-Forward Cross-Validation ───────────────────────────────────────────
print("\n  [Walk-Forward CV - v16]")
MIN_TRAIN_WF = max(12, N - 8)

wf_results = {}

for tgt_name, cfg in TARGET_CONFIG.items():
    col     = cfg['col']
    use_log = cfg['log']
    series  = monthly[col].values.copy()
    mapes_wf, preds_wf, actuals_wf = [], [], []

    for test_idx in range(MIN_TRAIN_WF, N):
        y_train     = series[:test_idx]
        y_true      = series[test_idx]
        ext_train   = monthly[EXT_COLS_AVAIL].values[:test_idx] if EXT_COLS_AVAIL else None
        ext_test    = monthly[EXT_COLS_AVAIL].values[test_idx:test_idx+1] if EXT_COLS_AVAIL else None
        
        if len(y_train) < 6 or np.all(y_train == 0): continue
        y_fit = np.log1p(np.maximum(y_train, 0)) if use_log else y_train.copy()

        # [v16-A] Kalman Trend Model
        try:
            kf = KalmanTrendFilter(phi=0.95)  # [v22] stable default, bukan 0.98
            kf.fit(y_fit, optimize=False)
            kf_pred_log = float(kf.forecast(1)[0])
            kf_pred = max(0, np.expm1(kf_pred_log) if use_log else kf_pred_log)
        except:
            kf_pred = float(y_train.mean())

        # [v16-B] Damped HW (with optimal phi)
        try:
            hw_model = ExponentialSmoothing(
                y_fit, trend='add', seasonal=None, damped_trend=True,
                initialization_method='estimated'
            )
            hw_fit  = hw_model.fit(optimized=True, use_brute=False)
            hw_pred_log = float(hw_fit.forecast(1).iloc[0])
            hw_pred = max(0, np.expm1(hw_pred_log) if use_log else hw_pred_log)
        except:
            hw_pred = float(y_train.mean())

        # [v16-C] SARIMAX with exog
        try:
            if ext_train is not None and len(ext_train) >= 6:
                sarima_model = SARIMAX(
                    y_fit, exog=ext_train,
                    order=(1, 1, 1), seasonal_order=(0,0,0,0),
                    enforce_stationarity=False, enforce_invertibility=False
                )
                sarima_fit  = sarima_model.fit(disp=False, maxiter=100)
                sarima_pred_log = float(sarima_fit.forecast(steps=1, exog=ext_test).iloc[-1])
            else:
                sarima_model = SARIMAX(
                    y_fit, order=(1, 1, 1), seasonal_order=(0,0,0,0),
                    enforce_stationarity=False, enforce_invertibility=False
                )
                sarima_fit  = sarima_model.fit(disp=False, maxiter=100)
                sarima_pred_log = float(sarima_fit.forecast(1).iloc[-1])
            sarima_pred = max(0, np.expm1(sarima_pred_log) if use_log else sarima_pred_log)
        except:
            sarima_pred = hw_pred

        # [v16] Ensemble: simple average (stacking weights computed analytically later)
        preds_list = [kf_pred, hw_pred, sarima_pred]
        ens_pred = np.mean(preds_list)
        ens_pred = max(0, ens_pred)

        mapes_wf.append(calc_mape([y_true], [ens_pred]))
        preds_wf.append(ens_pred)
        actuals_wf.append(y_true)

    wf_results[tgt_name] = {
        'mapes':     mapes_wf,
        'mean_mape': float(np.mean(mapes_wf)) if mapes_wf else 99.0,
        'preds':     preds_wf,
        'actuals':   actuals_wf,
        'bias':      float(np.mean([(a-p)/max(abs(a),1) for a,p in zip(actuals_wf,preds_wf)]))
                     if preds_wf else 0.0
    }
    print(f"  {tgt_name:<15}: WF-MAPE={wf_results[tgt_name]['mean_mape']:.2f}%"
          f"  Bias={wf_results[tgt_name]['bias']*100:+.1f}%  (n={len(mapes_wf)})")


# ─── Build Final Models ───────────────────────────────────────────────────────
print("\n  [Building Final Models - v16]")
final_models = {}
final_mapes  = {}

for tgt_name, cfg in TARGET_CONFIG.items():
    col     = cfg['col']
    use_log = cfg['log']
    series_raw = monthly[col].values.copy()
    n_series   = len(series_raw)
    
    print(f"\n  ─── {tgt_name} ────────────────────────────────────────────")
    
    y_fit   = np.log1p(np.maximum(series_raw, 0)) if use_log else series_raw.copy()
    ext_all = monthly[EXT_COLS_AVAIL].values if EXT_COLS_AVAIL else None

    # ── [v16-A] Kalman Trend Filter ─────────────────────────────────────────
    kf_model = None
    mape_kf  = 99.0
    try:
        print(f"    [A] Kalman Trend Filter...")
        kf_model = KalmanTrendFilter()
        kf_model.fit(y_fit, optimize=True)

        # [v22-FIX] OOS evaluation pakai phi yang sudah dioptimasi (bukan default)
        # v21 bug: optimize=False → pakai phi=0.98 default → OOS-MAPE tidak berubah
        # v22 fix: simpan phi optimal dari full fit, pakai untuk semua OOS evaluation
        optimal_phi_kf    = kf_model.phi
        optimal_ql_kf     = kf_model.q_level
        optimal_qt_kf     = kf_model.q_trend

        n_test = min(6, n_series // 3)
        kf_oos_preds = []
        for i_oos in range(n_series - n_test, n_series):
            try:
                kf_tmp = KalmanTrendFilter(
                    phi=optimal_phi_kf,
                    q_level=optimal_ql_kf,
                    q_trend=optimal_qt_kf
                )
                # [v22] optimize=False tapi pakai phi yang BENAR (bukan default 0.98)
                kf_tmp.fit(y_fit[:i_oos], optimize=False)
                kf_fc_log = float(kf_tmp.forecast(1)[0])
                kf_fc_raw = max(0, np.expm1(kf_fc_log) if use_log else kf_fc_log)
                kf_oos_preds.append(kf_fc_raw)
            except:
                kf_oos_preds.append(float(series_raw[:i_oos].mean()))
        mape_kf = calc_mape(series_raw[-n_test:], kf_oos_preds)
        print(f"    [A] Kalman phi={kf_model.phi:.3f} OOS-MAPE={mape_kf:.2f}%")
    except Exception as e:
        print(f"    [A] Kalman failed: {e}")

    # ── [v16-B] Optimal Damped HW ──────────────────────────────────────────
    hw_model  = None
    mape_hw   = 99.0
    best_phi_hw = 0.95
    try:
        print(f"    [B] Optimal Damped HW...")
        best_phi_hw, best_mape_phi, phi_mapes_tgt = find_optimal_damping(
            series_raw, phi_grid=[0.80, 0.85, 0.90, 0.95, 0.97, 0.98, 1.00]
        )
        print(f"    [B] Optimal φ={best_phi_hw} (WF-MAPE={best_mape_phi:.2f}%)")

        if best_phi_hw < 1.0:
            hw_m = ExponentialSmoothing(
                y_fit, trend='add', seasonal=None, damped_trend=True,
                initialization_method='estimated'
            )
            hw_model = hw_m.fit(damping_trend=best_phi_hw, optimized=True, use_brute=True)
        else:
            hw_m = ExponentialSmoothing(
                y_fit, trend='add', seasonal=None, damped_trend=False,
                initialization_method='estimated'
            )
            hw_model = hw_m.fit(optimized=True, use_brute=True)

        # [v17-FIX-4] Gunakan WF MAPE (OOS) untuk weighting, bukan in-sample
        # best_mape_phi sudah merupakan walk-forward MAPE dari find_optimal_damping
        mape_hw = best_mape_phi
        print(f"    [B] Damped HW OOS-MAPE={mape_hw:.2f}%")
    except Exception as e:
        print(f"    [B] HW failed: {e}")

    # ── [v16-C] SARIMAX with External Regressors ───────────────────────────
    sarima_fitted = None
    mape_sarima   = 99.0
    try:
        print(f"    [C] SARIMAX + external regressors...")
        if ext_all is not None and len(ext_all) >= 8:
            best_ord, best_aic = find_best_sarimax(y_fit, exog=ext_all)
            print(f"    [C] Best SARIMAX{best_ord} AIC={best_aic:.2f}")
            sarima_model  = SARIMAX(
                y_fit, exog=ext_all, order=best_ord, seasonal_order=(0,0,0,0),
                enforce_stationarity=False, enforce_invertibility=False
            )
            sarima_fitted = sarima_model.fit(disp=False, maxiter=100)
        else:
            best_ord = (1,1,1)
            sarima_model  = SARIMAX(
                y_fit, order=best_ord, seasonal_order=(0,0,0,0),
                enforce_stationarity=False, enforce_invertibility=False
            )
            sarima_fitted = sarima_model.fit(disp=False, maxiter=100)
        
        n_test = min(6, n_series // 3)
        s_insample = sarima_fitted.predict(start=n_series-n_test, end=n_series-1)
        if use_log:
            s_insample_raw = np.expm1(np.array(s_insample))
        else:
            s_insample_raw = np.array(s_insample)
        mape_sarima = calc_mape(series_raw[-n_test:], np.maximum(s_insample_raw, 0))
        print(f"    [C] SARIMAX MAPE={mape_sarima:.2f}%")
    except Exception as e:
        print(f"    [C] SARIMAX failed: {e}")

    # ── [v16-D] STL + Ridge ────────────────────────────────────────────────
    stl_ml_components = None
    mape_stl  = 99.0
    try:
        print(f"    [D] STL + Ridge...")
        if n_series >= 12:
            ts_index  = pd.date_range(start='2024-01-01', periods=n_series, freq='MS')
            ts_series = pd.Series(y_fit, index=ts_index)
            stl_d = STL(ts_series, period=6, robust=True)
            stl_r = stl_d.fit()
            trend_vals = stl_r.trend.values
            t_arr = np.arange(n_series)
            trend_poly = np.polyfit(t_arr, trend_vals, deg=1)  # linear trend (not poly2)
            seasonal_pattern = np.zeros(6)
            for m_ in range(6):
                idxs = [i for i in range(n_series) if i % 6 == m_]
                if idxs:
                    seasonal_pattern[m_] = np.mean(stl_r.seasonal.values[idxs])
            
            resid = stl_r.resid.values
            
            # [v16] Include external regressors in STL residual model
            if ext_all is not None:
                X_stl = np.column_stack([
                    np.arange(n_series),
                    np.sin(2*np.pi*np.arange(n_series)/6),
                    np.cos(2*np.pi*np.arange(n_series)/6),
                    ext_all[:n_series, :min(3, ext_all.shape[1])]  # first 3 ext features
                ])
            else:
                X_stl = np.column_stack([
                    np.arange(n_series),
                    np.sin(2*np.pi*np.arange(n_series)/6),
                    np.cos(2*np.pi*np.arange(n_series)/6),
                ])
            
            scaler_stl = RobustScaler()
            X_stl_sc   = scaler_stl.fit_transform(X_stl)
            ridge_resid = Ridge(alpha=10)
            ridge_resid.fit(X_stl_sc[:-1], resid[1:])

            stl_ml_components = {
                'trend_poly': trend_poly, 'seasonal_pattern': seasonal_pattern,
                'ridge': ridge_resid, 'scaler': scaler_stl,
                'n_train': n_series, 'use_log': use_log,
                'n_ext': X_stl.shape[1] - 3  # how many ext cols were added
            }
            
            n_test = min(6, n_series // 3)
            stl_preds_test = []
            for i in range(n_series - n_test, n_series):
                t_next = i
                trend_next    = np.polyval(trend_poly, t_next)
                seasonal_next = seasonal_pattern[t_next % 6]
                
                if ext_all is not None:
                    X_next = np.array([[t_next,
                                        np.sin(2*np.pi*t_next/6), np.cos(2*np.pi*t_next/6)] +
                                        ext_all[i, :min(3, ext_all.shape[1])].tolist()])
                else:
                    X_next = np.array([[t_next,
                                        np.sin(2*np.pi*t_next/6), np.cos(2*np.pi*t_next/6)]])
                X_next_sc = scaler_stl.transform(X_next)
                resid_next = float(ridge_resid.predict(X_next_sc)[0])
                pv = trend_next + seasonal_next + resid_next
                stl_preds_test.append(max(0, np.expm1(pv) if use_log else pv))
            
            mape_stl = calc_mape(series_raw[-n_test:], stl_preds_test)
            print(f"    [D] STL+Ridge MAPE={mape_stl:.2f}%")
    except Exception as e:
        print(f"    [D] STL failed: {e}")

    # ── [v16-E] Prophet ────────────────────────────────────────────────────
    prophet_fitted = None
    mape_prophet   = 99.0
    try:
        if HAS_PROPHET:
            print(f"    [E] Prophet...")
            series_for_prophet = np.expm1(y_fit) if use_log else series_raw
            dates = pd.date_range(end=pd.Timestamp('2025-07-01'), periods=n_series, freq='MS')
            df_p  = pd.DataFrame({
                'ds': dates,
                'y': np.maximum(series_for_prophet, 0.01)
            })
            
            # Add external regressors if available
            if ext_all is not None:
                for j, col_name in enumerate(EXT_COLS_AVAIL[:3]):
                    df_p[col_name] = ext_all[:n_series, j]
            
            model_p = Prophet(
                yearly_seasonality=False, weekly_seasonality=False, daily_seasonality=False,
                changepoint_prior_scale=0.30, seasonality_prior_scale=0.01,
                changepoint_range=0.85, interval_width=0.8
            )
            
            if ext_all is not None:
                for col_name in EXT_COLS_AVAIL[:3]:
                    model_p.add_regressor(col_name)
            
            model_p.fit(df_p, iter=500)
            
            future = model_p.make_future_dataframe(periods=0, freq='MS')
            if ext_all is not None:
                for j, col_name in enumerate(EXT_COLS_AVAIL[:3]):
                    future[col_name] = ext_all[-len(future):, j]
            
            forecast_p = model_p.predict(future)
            prophet_fitted = model_p
            
            n_test = min(6, n_series // 3)
            p_preds_raw = forecast_p['yhat'].values[-n_test:]
            mape_prophet = calc_mape(series_raw[-n_test:], np.maximum(p_preds_raw, 0))
            print(f"    [E] Prophet MAPE={mape_prophet:.2f}%")
    except Exception as e:
        print(f"    [E] Prophet error: {e}")

    # ── [v16 FIX-9] Stacking Weights via Ridge (Analytic) ──────────────────
    mapes_all = {'Kalman': mape_kf, 'HW': mape_hw, 'SARIMAX': mape_sarima,
                 'STL': mape_stl, 'Prophet': mape_prophet}
    
    valid_models = {k: v for k, v in mapes_all.items() if v < 60.0}
    if not valid_models:
        valid_models = {'HW': mape_hw}
    
    # Inverse-MAPE^2 weights (more stable than optimization)
    inv_mape2_sum = sum(1.0 / (v**2 + 1e-6) for v in valid_models.values())
    weights_e = {}
    for k in mapes_all.keys():
        if k in valid_models:
            weights_e[k] = (1.0 / (mapes_all[k]**2 + 1e-6)) / inv_mape2_sum
        else:
            weights_e[k] = 0.0
    
    print(f"    MAPEs: {' '.join([f'{k}={v:.1f}%' for k,v in mapes_all.items()])}")
    print(f"    Weights: {' '.join([f'{k}={v:.3f}' for k,v in weights_e.items()])}")
    
    # Bias correction
    bias_cf = 1.0 + wf_results[tgt_name]['bias']
    print(f"    Bias correction: ×{bias_cf:.4f}")
    
    final_models[tgt_name] = {
        'kalman':    kf_model,
        'hw':        hw_model,
        'hw_phi':    best_phi_hw,
        'sarima':    sarima_fitted,
        'stl':       stl_ml_components,
        'prophet':   prophet_fitted,
        'weights':   weights_e,
        'use_log':   use_log,
        'col':       col,
        'n_train':   n_series,
        'series_raw': series_raw,
        'series_log': y_fit,
        'bias_cf':   bias_cf,
        'ext_all':   ext_all,
    }
    final_mapes[tgt_name] = min(valid_models.values()) if valid_models else 99.0

print(f"\n  ─────────────────────────────────────────────────────────────────")
for tgt in ['frequency', 'severity', 'total_claim']:
    print(f"  {tgt:<15}: Best MAPE={final_mapes[tgt]:.2f}%")
print(f"  AVG MAPE: {np.mean(list(final_mapes.values())):.2f}%")


# ─── Model Evaluation Dashboard ──────────────────────────────────────────────
print("\n  [Generating Model Evaluation Dashboard...]")
tgt_labels = ['Frequency', 'Severity (Wins)', 'Total Claim']
tgt_keys   = ['frequency', 'severity', 'total_claim']

try:
    fig_ev = plt.figure(figsize=(22, 18))
    fig_ev.suptitle(
        'Model Evaluation v16.0 — Kalman + Damped HW + SARIMAX(exog) + STL + Prophet\n'
        '[v16] Optimal φ Damping | Frequency Anchor | Severity Floor | Target: <4.0',
        fontsize=12, fontweight='bold'
    )
    gs_ev = gridspec.GridSpec(3, 4, figure=fig_ev, hspace=0.55, wspace=0.42)

    for row, (label, key) in enumerate(zip(tgt_labels, tgt_keys)):
        cfg     = TARGET_CONFIG[key]
        use_log = cfg['log']
        mdl     = final_models[key]
        series  = mdl['series_raw']
        n_s     = mdl['n_train']
        n_test  = min(6, n_s // 3)
        
        actuals_eval = series[-n_test:]
        preds_kf, preds_hw, preds_sarima, preds_ens = [], [], [], []

        for i in range(n_s - n_test, n_s):
            # Kalman - use OOS forecast (not in-sample fitted_values_)
            try:
                y_fit_tmp = mdl['series_log'][:i]
                kf_tmp = KalmanTrendFilter()
                kf_tmp.fit(y_fit_tmp, optimize=False)
                kf_fc_log = float(kf_tmp.forecast(1)[0])
                use_log_ev = mdl['use_log']
                pk = max(0, np.expm1(kf_fc_log) if use_log_ev else kf_fc_log)
            except:
                pk = float(series.mean())
            preds_kf.append(pk)

            # HW
            try:
                ph = float(mdl['hw'].fittedvalues[i])
                ph = max(0, np.expm1(ph) if use_log else ph)
            except:
                ph = float(series.mean())
            preds_hw.append(ph)

            # SARIMAX
            try:
                if mdl['sarima'] is not None:
                    ps = float(mdl['sarima'].predict(start=i, end=i)[0])
                    ps = max(0, np.expm1(ps) if use_log else ps)
                else:
                    ps = ph
            except:
                ps = ph
            preds_sarima.append(ps)

            # Ensemble
            w = mdl['weights']
            pred_list_, weight_list_ = [], []
            for mn, pv_m in [('Kalman', pk), ('HW', ph), ('SARIMAX', ps)]:
                wt = w.get(mn, 0)
                if wt > 0 and pv_m > 0:
                    pred_list_.append(pv_m)
                    weight_list_.append(wt)
            if pred_list_:
                log_preds = np.log(np.maximum(pred_list_, 1e-10))
                wts = np.array(weight_list_) / sum(weight_list_)
                pe  = float(np.exp(np.dot(wts, log_preds)))
            else:
                pe = float(series.mean())
            pe = max(0, pe * mdl['bias_cf'])
            preds_ens.append(pe)

        mape_kf_  = calc_mape(actuals_eval, preds_kf)
        mape_hw_  = calc_mape(actuals_eval, preds_hw)
        mape_s_   = calc_mape(actuals_eval, preds_sarima)
        mape_e_   = calc_mape(actuals_eval, preds_ens)

        ax = fig_ev.add_subplot(gs_ev[row, 0])
        model_names = ['Kalman', 'Damped HW', 'SARIMAX', 'Ensemble']
        mape_vals   = [mape_kf_, mape_hw_, mape_s_, mape_e_]
        best = np.argmin(mape_vals)
        bars = ax.bar(model_names, mape_vals,
                      color=[COLORS[0], COLORS[1], COLORS[2], COLORS[3]], alpha=0.85)
        bars[best].set_edgecolor('gold'); bars[best].set_linewidth(2.5)
        ax.set_title(f'{label} MAPE', fontsize=10, fontweight='bold')
        ax.set_ylabel('MAPE (%)')
        ax.set_xticklabels(model_names, rotation=30, ha='right', fontsize=8)
        for b, v in zip(bars, mape_vals):
            ax.text(b.get_x() + b.get_width()/2, b.get_height() + max(mape_vals)*0.02,
                    f'{v:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

        ax2 = fig_ev.add_subplot(gs_ev[row, 1])
        if len(actuals_eval) > 0 and len(preds_ens) > 0:
            ax2.scatter(actuals_eval, preds_ens, alpha=0.8, color=COLORS[row], s=80, edgecolors='white', zorder=3)
            mn = min(min(actuals_eval), min(preds_ens))
            mx = max(max(actuals_eval), max(preds_ens))
            ax2.plot([mn, mx], [mn, mx], 'r--', lw=2, label='Perfect')
            r2 = (np.corrcoef(actuals_eval, preds_ens)[0,1]**2 if len(np.unique(actuals_eval)) > 1 else 0)
            ax2.text(0.05, 0.95, f'R²={r2:.3f}\nMAPE={mape_e_:.1f}%',
                     transform=ax2.transAxes, fontsize=9, va='top',
                     bbox=dict(boxstyle='round', facecolor='#FFF9C4', alpha=0.9))
            ax2.legend(fontsize=8)
        ax2.set_title(f'{label} Actual vs Predicted', fontsize=10, fontweight='bold')

        ax3 = fig_ev.add_subplot(gs_ev[row, 2])
        if len(actuals_eval) > 0 and len(preds_ens) > 0:
            resid_pct = [(a-p)/(max(abs(a),1))*100 for a,p in zip(actuals_eval,preds_ens)]
            ax3.bar(range(len(resid_pct)), resid_pct, color=COLORS[row], alpha=0.75)
            ax3.axhline(y=0, color='red', lw=2, ls='--')
            ax3.set_title(f'{label} Residual %', fontsize=10, fontweight='bold')
            ax3.set_ylabel('Error (%)')
            ax3.text(0.05, 0.95,
                     f'Mean: {np.mean(resid_pct):+.1f}%\nStd: {np.std(resid_pct):.1f}%',
                     transform=ax3.transAxes, fontsize=9, va='top',
                     bbox=dict(boxstyle='round', facecolor='#E8F8F5', alpha=0.9))

        ax4 = fig_ev.add_subplot(gs_ev[row, 3])
        div_f = 1e6 if key == 'severity' else (1e9 if key == 'total_claim' else 1)
        unit  = 'Juta IDR' if key == 'severity' else ('Miliar IDR' if key == 'total_claim' else 'klaim')
        ax4.plot(range(n_s), series/div_f, color='gray', lw=1.5, alpha=0.7, marker='o', ms=4, label='Actual')
        try:
            if mdl['hw'] is not None:
                hw_fv = np.array(mdl['hw'].fittedvalues)
                if use_log: hw_fv = np.expm1(hw_fv)
                ax4.plot(range(n_s), np.maximum(hw_fv,0)/div_f, color=COLORS[1], lw=1.5, alpha=0.7, label='HW fit')
            if mdl['kalman'] is not None:
                kf_fv = mdl['kalman'].fitted_values_
                if use_log: kf_fv = np.expm1(np.array(kf_fv))
                ax4.plot(range(n_s), np.maximum(kf_fv,0)/div_f, color=COLORS[0], lw=1.5, ls='--', alpha=0.7, label='Kalman')
        except: pass
        ax4.axvline(x=n_s-n_test, color='red', lw=2, ls='--', alpha=0.7, label='Val start')
        ax4.set_ylabel(unit)
        ax4.legend(fontsize=7)
        ax4.set_title(f'{label} Historical', fontsize=10, fontweight='bold')

    plt.savefig('model_evaluation_v16.png', dpi=100, bbox_inches='tight', facecolor='white')
    plt.close()
    print("  ✓ model_evaluation_v16.png tersimpan")
except Exception as e:
    print(f"  ⚠ Eval plot: {e}")
    plt.close('all')


# ========================== 8. PREDIKSI AGT–DES 2025 =======================
print("\n[8/10] PREDIKSI AGT–DES 2025 (v16)...")

# ─── Map EXT_COLS_AVAIL (monthly names with 'ext_' prefix) ─────────────────
# to raw df_ext_forecast column names (without 'ext_' prefix)
# EXT_COLS_AVAIL example: ['ext_inflation', 'ext_seasonal', ...]
# df_ext_forecast columns:  ['inflation_idx', 'seasonal_prior', ...]
EXT_COL_MAP = {
    'ext_inflation': 'inflation_idx',
    'ext_seasonal':  'seasonal_prior',
    'ext_ramadan':   'is_ramadan',
    'ext_eid':       'is_eid_fitr',
    'ext_yearend':   'is_yearend',
    'ext_trend':     'linear_trend',
}

# Build forecast ext arrays using correct column names
ext_fc_dict = {}
for i, m_ in enumerate(PRED_MONTHS_2025):
    if EXT_COLS_AVAIL and len(df_ext_forecast) > i:
        row_vals = []
        for col_monthly in EXT_COLS_AVAIL:
            raw_col = EXT_COL_MAP.get(col_monthly, None)
            if raw_col and raw_col in df_ext_forecast.columns:
                row_vals.append(float(df_ext_forecast.iloc[i][raw_col]))
            else:
                row_vals.append(1.0)  # fallback
        ext_fc_dict[m_] = np.array(row_vals, dtype=float)
    else:
        ext_fc_dict[m_] = None

print(f"  ext_fc_dict months: {list(ext_fc_dict.keys())}")
for m_, arr in ext_fc_dict.items():
    if arr is not None:
        print(f"    {m_}: {dict(zip(EXT_COLS_AVAIL, arr.round(4)))}")


def forecast_target_v16(tgt_name, pred_year, pred_months):
    """
    [v16] Complete forecasting pipeline:
    1. Kalman Trend forecast (damped)
    2. Optimal Damped HW forecast
    3. SARIMAX with external regressors
    4. STL + Ridge (with external)
    5. Geometric mean ensemble
    6. Bias correction
    7. YoY blending (adaptive)
    8. ANCHOR / FLOOR protection (KEY FIX)
    9. Seasonal adjustment (external-informed)
    10. Clip to expanded historical range
    """
    mdl     = final_models[tgt_name]
    use_log = mdl['use_log']
    series  = mdl['series_raw']
    n_s     = mdl['n_train']
    w       = mdl['weights']
    bias_cf = mdl['bias_cf']
    ext_all = mdl['ext_all']

    last_train_ym = monthly['year_month'].iloc[-1]
    last_year  = int(str(last_train_ym)[:4])
    last_month = int(str(last_train_ym)[5:7])

    results = {}
    for pred_month in pred_months:
        steps = (pred_year - last_year)*12 + (pred_month - last_month)
        h = max(1, steps)
        
        # Forecast exogenous (for SARIMAX)
        ext_fc = ext_fc_dict.get(pred_month, None)
        
        # ── [v16-A] Kalman Trend ─────────────────────────────────────────────
        try:
            kf = mdl['kalman']
            if kf is not None:
                fc_kf_log_arr = kf.forecast(h)
                fc_kf_log     = float(fc_kf_log_arr[-1])
                fc_kf = max(0, np.expm1(fc_kf_log) if use_log else fc_kf_log)
            else:
                fc_kf = float(series.mean())
        except:
            fc_kf = float(series.mean())

        # ── [v16-B] Damped HW ─────────────────────────────────────────────────
        try:
            if mdl['hw'] is not None:
                fc_hw_arr = mdl['hw'].forecast(h)
                fc_hw_log = float(fc_hw_arr.iloc[-1])
                fc_hw = max(0, np.expm1(fc_hw_log) if use_log else fc_hw_log)
            else:
                fc_hw = float(series.mean())
        except:
            fc_hw = float(series.mean())

        # ── [v16-C] SARIMAX with external regressors ─────────────────────────
        try:
            if mdl['sarima'] is not None:
                if ext_fc is not None and ext_all is not None:
                    exog_fc = np.zeros((h, ext_all.shape[1]))
                    for step_i in range(h):
                        exog_fc[step_i] = ext_fc  # use same month's exog for all steps
                    fc_s_arr = mdl['sarima'].get_forecast(steps=h, exog=exog_fc)
                    fc_s_log = float(fc_s_arr.predicted_mean.iloc[-1])
                else:
                    fc_s_arr = mdl['sarima'].get_forecast(steps=h)
                    fc_s_log = float(fc_s_arr.predicted_mean.iloc[-1])
                fc_sarima = max(0, np.expm1(fc_s_log) if use_log else fc_s_log)
            else:
                fc_sarima = fc_hw
        except:
            fc_sarima = fc_hw

        # ── [v16-D] STL ───────────────────────────────────────────────────────
        try:
            if mdl['stl'] is not None:
                stl_c = mdl['stl']
                t_next = n_s + h - 1
                trend_next    = np.polyval(stl_c['trend_poly'], t_next)
                seasonal_next = stl_c['seasonal_pattern'][t_next % 6]
                
                n_ext_stl = stl_c.get('n_ext', 0)
                if ext_fc is not None and n_ext_stl > 0:
                    X_next = np.array([[t_next,
                                        np.sin(2*np.pi*t_next/6), np.cos(2*np.pi*t_next/6)] +
                                        ext_fc[:n_ext_stl].tolist()])
                else:
                    X_next = np.array([[t_next,
                                        np.sin(2*np.pi*t_next/6), np.cos(2*np.pi*t_next/6)]])
                X_next_sc  = stl_c['scaler'].transform(X_next)
                resid_next = float(stl_c['ridge'].predict(X_next_sc)[0])
                pv_stl     = trend_next + seasonal_next + resid_next
                fc_stl     = max(0, np.expm1(pv_stl) if use_log else pv_stl)
            else:
                fc_stl = float(series.mean())
        except:
            fc_stl = float(series.mean())

        # ── [v16-E] Prophet ───────────────────────────────────────────────────
        fc_prophet = None
        try:
            if mdl['prophet'] is not None and w.get('Prophet', 0) > 0.01:
                model_p = mdl['prophet']
                pred_date = pd.Timestamp(f'{pred_year}-{pred_month:02d}-01')
                future_p  = model_p.make_future_dataframe(periods=h, freq='MS')
                if ext_fc is not None and EXT_COLS_AVAIL:
                    last_ext = ext_all[-1] if ext_all is not None else None
                    for j, col_name in enumerate(EXT_COLS_AVAIL[:3]):
                        if j < len(future_p) and ext_fc is not None:
                            future_p.loc[future_p['ds'] >= pred_date, col_name] = float(ext_fc[j])
                            if last_ext is not None:
                                future_p.loc[future_p['ds'] < pred_date, col_name] = float(last_ext[j])
                fc_p_df   = model_p.predict(future_p)
                fc_prophet = max(0, float(fc_p_df['yhat'].iloc[-1]))
        except:
            fc_prophet = None

        # ── [v16] GEOMETRIC MEAN ENSEMBLE ─────────────────────────────────────
        pred_list_   = []
        weight_list_ = []
        for mn, fc_val in [('Kalman', fc_kf), ('HW', fc_hw), ('SARIMAX', fc_sarima),
                            ('STL', fc_stl), ('Prophet', fc_prophet)]:
            wt = w.get(mn, 0)
            if wt > 0 and fc_val is not None and fc_val > 0:
                pred_list_.append(fc_val)
                weight_list_.append(wt)

        if pred_list_:
            log_preds_ = np.log(np.maximum(pred_list_, 1e-10))
            wts_       = np.array(weight_list_) / sum(weight_list_)
            fc_ens     = float(np.exp(np.dot(wts_, log_preds_)))
        else:
            fc_ens = float(series.mean())
        fc_ens = max(0, fc_ens)

        # ── [v16] BIAS CORRECTION ─────────────────────────────────────────────
        fc_ens = fc_ens * bias_cf

        # ── [v17-FIX-3] ADAPTIVE YoY BLENDING ───────────────────────────────
        yoy_mask_p = (monthly['bulan_masuk'] == pred_month) & (monthly['tahun'] == pred_year - 1)
        if yoy_mask_p.any():
            yoy_actual = float(monthly.loc[yoy_mask_p, mdl['col']].values[0])
            if tgt_name == 'frequency':
                growth = ACTUAL_GROWTH_FREQ
            elif tgt_name == 'severity':
                growth = ACTUAL_GROWTH_SEV
            else:
                growth = ACTUAL_GROWTH_TOTAL
            fc_yoy = yoy_actual * (1 + growth)

            # [v20-FIX-3] YoY weight lebih rendah untuk target yang sudah akurat
            # Freq 3.22% & Sev 3.86% → model sudah bagus → YoY minimal
            model_mape = wf_results[tgt_name]['mean_mape']
            if model_mape < 4.0:
                yoy_weight = 0.10   # model sangat akurat → percaya model
            elif model_mape < 7.0:
                yoy_weight = 0.20   # cukup akurat
            elif model_mape > 10:
                yoy_weight = 0.40   # model kurang akurat → lebih percaya YoY
            else:
                yoy_weight = 0.30
            fc_ens = (1 - yoy_weight) * fc_ens + yoy_weight * fc_yoy
            fc_ens = max(0, fc_ens)

        # ── [v16] EXTERNAL SEASONAL ADJUSTMENT ───────────────────────────────
        # Use prior knowledge seasonal index (from external data)
        ext_seas_idx = 1.0
        if ext_fc is not None and 'ext_seasonal' in EXT_COLS_AVAIL:
            j_seas = EXT_COLS_AVAIL.index('ext_seasonal')
            ext_seas_idx = float(ext_fc[j_seas])
        
        # Dampen the seasonal adjustment (30% weight)
        seasonal_weight = 0.30
        fc_ens = fc_ens * (1 - seasonal_weight + seasonal_weight * ext_seas_idx)

        # ── [v16 KEY FIX] ANCHOR AND FLOOR PROTECTION ────────────────────────
        if tgt_name == 'frequency':
            # [v17-FIX-1] Frequency floor - HARD floor (v16 had soft floor bug)
            freq_floor  = FREQ_RECENT_3M * 0.80
            freq_last   = float(series[-1])
            freq_anchor = freq_last * 0.75
            fc_floor    = max(freq_floor, freq_anchor)

            if fc_ens < fc_floor:
                # [v17] Hard floor - langsung set ke floor (bukan soft recovery yg buggy)
                print(f"    [ANCHOR] Freq {pred_month}: {fc_ens:.0f} → hard floor {fc_floor:.0f}")
                fc_ens = fc_floor

            # Cannot exceed recent high × 1.3
            freq_ceiling = HIST_FREQ_P95 * 1.25
            fc_ens = min(fc_ens, freq_ceiling)

        elif tgt_name == 'severity':
            # [v17-FIX-2] Severity floor saja, TIDAK dikali inflation ganda
            # Model sudah capture trend naik, inflation hanya sebagai safety floor
            sev_floor = SEV_RECENT_6M * 0.88  # slightly tighter floor
            if fc_ens < sev_floor:
                print(f"    [FLOOR] Sev {pred_month}: {fc_ens/1e6:.2f}M → floor {sev_floor/1e6:.2f}M")
                fc_ens = sev_floor
            # [v17] REMOVED: double inflation factor (was causing overestimate)
            # inflation_factor = (1 + 0.007) ** months_ahead  ← DIHAPUS

        else:  # total_claim
            tot_floor = HIST_TOT_P5 * 0.75
            if fc_ens < tot_floor:
                fc_ens = tot_floor

        # ── FINAL CLIPPING ────────────────────────────────────────────────────
        if tgt_name == 'frequency':
            fc_ens = np.clip(fc_ens, HIST_FREQ_P5 * 0.60, HIST_FREQ_P95 * 1.50)
        elif tgt_name == 'severity':
            fc_ens = np.clip(fc_ens, HIST_SEV_P5 * 0.70, HIST_SEV_P95 * 1.50)
        else:
            fc_ens = np.clip(fc_ens, HIST_TOT_P5 * 0.60, HIST_TOT_P95 * 1.60)

        results[pred_month] = fc_ens

    return results


# Generate predictions
print("\n  === Prediksi 2025 (Agt–Des) ===")
all_preds_2025 = {
    'frequency':   forecast_target_v16('frequency',   2025, PRED_MONTHS_2025),
    'severity':    forecast_target_v16('severity',    2025, PRED_MONTHS_2025),
    'total_claim': forecast_target_v16('total_claim', 2025, PRED_MONTHS_2025),
}

print(f"\n  Initial predictions (before F×S enforcement):")
print(f"  {'Bulan':<10} {'Freq':>8} {'Sev (M)':>10} {'Total (B)':>12} {'F×S (B)':>10} {'Ratio':>7}")
print(f"  {'-'*60}")
for m in PRED_MONTHS_2025:
    f_ = all_preds_2025['frequency'][m]
    s_ = all_preds_2025['severity'][m]
    t_ = all_preds_2025['total_claim'][m]
    fs = f_ * s_ * GLOBAL_FS_CORRECTION
    ratio = t_ / (fs + 1e-10)
    print(f"  2025-{m:02d}   {f_:>8,.0f} {s_/1e6:>9.2f}M {t_/1e9:>11.3f}B "
          f"{fs/1e9:>9.3f}B {ratio:>6.2f}")

# ─── [v20-FIX-1] F×S PRIMARY ENFORCEMENT (85% F×S + 15% Model) ──────────────
# Logika: freq WF-MAPE 3.22% dan sev WF-MAPE 3.86% sudah sangat akurat
# → F×S = freq × sev × CF akan lebih akurat dari model total_claim independent
print("\n  [v20] F×S Primary Enforcement (85% F×S + 15% model)...")
print(f"  F×S correction factor: {GLOBAL_FS_CORRECTION:.4f}")
print(f"  Freq MAPE={wf_results['frequency']['mean_mape']:.2f}% + "
      f"Sev MAPE={wf_results['severity']['mean_mape']:.2f}% → F×S reliable\n")

for m in PRED_MONTHS_2025:
    f_ = all_preds_2025['frequency'][m]
    s_ = all_preds_2025['severity'][m]
    t_ = all_preds_2025['total_claim'][m]

    t_fs  = f_ * s_ * GLOBAL_FS_CORRECTION
    ratio = t_ / (t_fs + 1e-10)

    # [v20] F×S dominan di semua kondisi
    if ratio < 0.65 or ratio > 1.55:
        t_new = t_fs                             # Full F×S
        tag = f"FULL F×S     (ratio={ratio:.2f})"
    elif ratio < 0.80 or ratio > 1.30:
        t_new = 0.90 * t_fs + 0.10 * t_         # 90% F×S
        tag = f"90/10 F×S    (ratio={ratio:.2f})"
    else:
        t_new = 0.85 * t_fs + 0.15 * t_         # 85% F×S (normal)
        tag = f"85/15 F×S    (ratio={ratio:.2f})"

    print(f"  2025-{m:02d}: {tag} | "
          f"model={t_/1e9:.3f}B F×S={t_fs/1e9:.3f}B → {t_new/1e9:.3f}B")
    all_preds_2025['total_claim'][m] = max(0, t_new)

# ─── Final Summary ────────────────────────────────────────────────────────────
print(f"\n  Final predictions:")
print(f"  {'Bulan':<10} {'Freq':>8} {'Sev (M)':>10} {'Total (B)':>12} {'F×S (B)':>10} {'Check':>5}")
print(f"  {'-'*60}")
for m in PRED_MONTHS_2025:
    f_ = all_preds_2025['frequency'][m]
    s_ = all_preds_2025['severity'][m]
    t_ = all_preds_2025['total_claim'][m]
    fs = f_ * s_ * GLOBAL_FS_CORRECTION
    ok = '✓' if 0.6 < t_/fs < 1.6 else '⚠'
    print(f"  2025-{m:02d}   {f_:>8,.0f} {s_/1e6:>9.2f}M {t_/1e9:>11.3f}B "
          f"{fs/1e9:>9.3f}B {ok:>5}")


# ─── Build Submission ─────────────────────────────────────────────────────────
rows_ = []
for m in PRED_MONTHS_2025:
    ms = f'2025_{m:02d}'
    rows_.append({'id': f'{ms}_Claim_Frequency', 'value': all_preds_2025['frequency'][m]})
    rows_.append({'id': f'{ms}_Claim_Severity',  'value': all_preds_2025['severity'][m]})
    rows_.append({'id': f'{ms}_Total_Claim',     'value': all_preds_2025['total_claim'][m]})

submission = pd.DataFrame(rows_)
submission['value'] = pd.to_numeric(submission['value'], errors='coerce').fillna(0).clip(0)

sample_ids = df_sample['id'].tolist()
sub_dict   = dict(zip(submission['id'], submission['value']))

id_match = set(submission['id'].tolist()) == set(sample_ids)
if id_match:
    submission = submission.set_index('id').loc[sample_ids].reset_index()
    print("\n  ✓ ID cocok sempurna")
else:
    missing = [s for s in sample_ids if s not in sub_dict]
    if missing:
        print(f"  ⚠ Missing IDs: {missing}")
    submission = pd.DataFrame({
        'id':    sample_ids,
        'value': [sub_dict.get(s, 0.0) for s in sample_ids]
    })

submission['value'] = submission['value'].round(4)
assert submission['value'].sum() > 0, "CRITICAL: All values = 0!"
submission.to_csv('submission.csv', index=False)

print(f"\n  Full submission:")
print(submission.to_string(index=False))


# ─── Forecast Visualization ──────────────────────────────────────────────────
try:
    fig_pred = plt.figure(figsize=(20, 20))
    fig_pred.suptitle(
        'Prediksi Klaim: Agt–Des 2025\n'
        'DSC MCF ITB — v16.0: Kalman + Optimal Damping + External Data\n'
        '[v16] Frequency ANCHOR + Severity FLOOR | Expected Public Score < 4.0',
        fontsize=13, fontweight='bold'
    )
    gs_pred = gridspec.GridSpec(3, 1, figure=fig_pred, hspace=0.55)

    hist12  = monthly.tail(12).copy()
    hist_dt = pd.to_datetime([str(ym) for ym in hist12['year_month']])
    pred_dt = [pd.Timestamp(f'2025-{m:02d}-01') for m in PRED_MONTHS_2025]

    cfg_plot = [
        ('frequency',
         [all_preds_2025['frequency'][m]   for m in PRED_MONTHS_2025],
         COLORS[0], 'Frekuensi Klaim (klaim/bulan)', 1, 'Klaim'),
        ('severity_w',
         [all_preds_2025['severity'][m]    for m in PRED_MONTHS_2025],
         COLORS[1], '[v16] Severitas Wins (Juta IDR/klaim)', 1e6, 'Juta IDR'),
        ('total_claim',
         [all_preds_2025['total_claim'][m] for m in PRED_MONTHS_2025],
         COLORS[2], 'Total Klaim (Miliar IDR/bulan)', 1e9, 'Miliar IDR'),
    ]

    for i, (tc, py25r, color, title, div, unit) in enumerate(cfg_plot):
        ax = fig_pred.add_subplot(gs_pred[i])
        hist_col = tc if tc in hist12.columns else tc.replace('_w', '')
        hy = hist12[hist_col].values / div if hist_col in hist12.columns else np.zeros(len(hist12))
        py25 = [v / div for v in py25r]

        ax.plot(hist_dt, hy, color='#636363', lw=2, marker='o', ms=6, label='Historis')
        ax.fill_between(hist_dt, 0, hy, alpha=0.06, color='gray')
        ax.plot([hist_dt[-1], pred_dt[0]], [hy[-1], py25[0]],
                color=color, lw=1.5, ls=':', alpha=0.5)
        ax.plot(pred_dt, py25, color=color, lw=3, marker='D', ms=10,
                ls='--', label='Prediksi v16 2025', zorder=5)

        # [v16] Show v15 predictions for comparison
        if tc == 'frequency':
            v15_vals = [198.0, 190.5, 226.0, 228.1, 201.5]
            ax.plot(pred_dt, [v/div for v in v15_vals], color='#C73E1D',
                    lw=1.5, ls=':', marker='^', ms=7, alpha=0.7, label='v15 pred (old)')
        
        wf_key  = tc.replace('_w', '')
        wf_mape = wf_results.get(wf_key, {}).get('mean_mape', 10)
        sigma   = np.mean(hy) * wf_mape / 100 if np.mean(hy) > 0 else 1
        ax.fill_between(pred_dt,
                        [max(0, v - sigma) for v in py25],
                        [v + sigma for v in py25],
                        alpha=0.15, color=color, label=f'±{wf_mape:.1f}% CI')

        ax.axvline(x=pd.Timestamp('2025-07-31'), color='gray', ls=':', lw=2, alpha=0.7)
        
        for dt, val in zip(pred_dt, py25):
            ax.annotate(f'{val:.1f}', (dt, val),
                        textcoords='offset points', xytext=(0, 14),
                        ha='center', fontsize=10, color=color, fontweight='bold')

        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_ylabel(unit)
        ax.legend(loc='upper left', fontsize=8)
        ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%b %Y'))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
        ax.set_facecolor('#F9F9F9')

        # [v16] Add anchor/floor line for frequency & severity
        if tc == 'frequency':
            ax.axhline(y=FREQ_RECENT_3M/div, color='green', lw=2, ls='--', alpha=0.6,
                       label=f'Anchor (3m avg={FREQ_RECENT_3M:.0f})')
        elif tc.replace('_w', '') == 'severity':
            ax.axhline(y=SEV_RECENT_6M*0.85/div, color='red', lw=2, ls='--', alpha=0.6,
                       label=f'Severity floor')

    plt.savefig('predictions_forecast_v16.png', dpi=100, bbox_inches='tight', facecolor='white')
    plt.close()
    print("\n  ✓ predictions_forecast_v16.png tersimpan")
except Exception as e:
    print(f"  ⚠ Forecast plot: {e}")
    plt.close('all')


# ========================== 9. VALIDATION SUMMARY ===========================
print("\n[9/10] VALIDASI SUBMISSION...")
print(f"\n  ─── VALIDASI SUBMISSION ─────────────────────────────────────────")
print(f"  Rows  : {len(submission)} (expected {len(sample_ids)})")
print(f"  Sum   : {submission['value'].sum():,.2f}")
print(f"\n  {'ID':<40} {'Value':>15}")
print(f"  {'-'*57}")
for _, row in submission.iterrows():
    print(f"  {row['id']:<40} {row['value']:>15,.2f}")


# ========================== 10. FINAL SUMMARY ==============================
print("\n" + "=" * 80)
print("  RINGKASAN HASIL FINAL v22.0")
print("=" * 80)
print(f"""
  ╔══════════════════════════════════════════════════════════════════════════╗
  ║  ROOT CAUSE FIX: Kalman OOS Evaluation                                  ║
  ╠══════════════════════════════════════════════════════════════════════════╣
  ║  [v22-FIX-1] Kalman OOS pakai phi optimal (bukan default 0.98)         ║
  ║    v21: optimize=False → phi=0.98 default → OOS-MAPE tidak berubah     ║
  ║    v22: simpan phi dari full fit → pakai untuk semua OOS steps          ║
  ║    Ekspektasi: Kalman MAPE turun → bobot Kalman naik                   ║
  ║  [v22-FIX-2] WF loop Kalman phi=0.95 (stable default)                 ║
  ║                                                                          ║
  ║  TETAP DARI v21/v20:                                                     ║
  ║    ✓ Kalman phi minimum 0.85                                            ║
  ║    ✓ Total Claim: 85% F×S + 15% model                                  ║
  ║    ✓ YoY weight adaptif (10% jika MAPE < 4%)                           ║
  ║    ✓ Path: VINCENT (benar)                                              ║
  ╚══════════════════════════════════════════════════════════════════════════╝
""")

# ============================================================
# [v17-NEW] WALK-FORWARD CV DISPLAY (seperti screenshot)
# ============================================================
print("=" * 62)
print("  WALK-FORWARD CV RESULTS (SIMULASI OFFLINE):")
print("=" * 62)
print(f"\n  {'Target':<15} {'WF-MAPE':>10} {'Bias':>10}")
print(f"  {'-'*38}")

tgt_display = [
    ('frequency',   'Frequency'),
    ('severity',    'Severity'),
    ('total_claim', 'Total Claim'),
]
for key, label in tgt_display:
    wf_m  = wf_results[key]['mean_mape']
    bias_ = wf_results[key]['bias'] * 100
    print(f"\n  {label:<15} {wf_m:>9.2f}% {bias_:>+9.1f}%")

print(f"\n{'=' * 62}")
print(f"  📌 Catatan: WF-MAPE adalah estimasi akurasi offline. Semakin kecil semakin baik.")
print(f"  🎯 Target Total Claim < 4.0% untuk bersaing di leaderboard.")
print(f"{'=' * 62}\n")

print(f"""
  ─── Output Files ────────────────────────────────────────────────────────
  ✓ submission.csv              ← UPLOAD KE KAGGLE (15 baris)
  ✓ eda_dashboard_v16.png       ← EDA + Kalman + Optimal Damping
  ✓ feature_importance_v16.png  ← Feature Importance + External
  ✓ model_evaluation_v16.png    ← Model comparison
  ✓ predictions_forecast_v16.png ← Forecast 2025 + v15 comparison

  ✓ v22 PIPELINE SELESAI!
  ✓ Expected Public Score: < 3.5 (stretch: < 3.0)
""")
print("=" * 80)


# ─── SIMULASI SKOR KAGGLE (5-MONTH HOLDOUT) ──────────────────────────────────
def hitung_simulasi_v23():
    """
    Simulasi SAMA dengan kode teman (5-month holdout).
    Potong 5 bulan terakhir → forecast → bandingkan → hitung MAPE.
    Teman v16: Freq=5.87%, Sev=2.44%, Total=7.12%, AVG=5.14%
    """
    print("\n" + "═" * 62)
    print("  🔍 SIMULASI SKOR KAGGLE v23 (5-MONTH HOLDOUT)")
    print("  [Metode sama dengan kode teman untuk perbandingan fair]")
    print("═" * 62)

    n_holdout = 5
    if len(monthly) <= n_holdout * 2:
        print("  ⚠ Data terlalu sedikit untuk simulasi holdout.")
        return

    train_data = monthly.iloc[:-n_holdout].copy()
    test_data  = monthly.iloc[-n_holdout:].copy()

    actuals = {
        'frequency':   test_data['frequency'].values,
        'severity':    test_data['severity_w'].values,
        'total_claim': test_data['total_claim'].values,
    }

    sim_preds = {'frequency': [], 'severity': [], 'total_claim': []}

    for key in ['frequency', 'severity', 'total_claim']:
        mdl      = final_models[key]
        use_log  = mdl['use_log']
        col_name = mdl['col']
        w        = mdl['weights']
        bias_cf  = mdl['bias_cf']

        series_train_raw = train_data[col_name].values
        series_train_log = np.log1p(np.maximum(series_train_raw, 0)) if use_log else series_train_raw.copy()

        fc_kf = np.zeros(n_holdout)
        fc_hw = np.zeros(n_holdout)
        fc_sa = np.zeros(n_holdout)

        # [A] Kalman — pakai phi optimal dari final model
        try:
            kf_sim = KalmanTrendFilter(
                phi=mdl['kalman'].phi,
                q_level=mdl['kalman'].q_level,
                q_trend=mdl['kalman'].q_trend
            )
            kf_sim.fit(series_train_log, optimize=False)
            fc_kf_log = kf_sim.forecast(n_holdout)
            fc_kf = np.maximum(0, np.expm1(fc_kf_log) if use_log else fc_kf_log)
        except:
            fc_kf = np.full(n_holdout, series_train_raw.mean())

        # [B] Damped HW
        try:
            phi_hw = mdl['hw_phi']
            hw_sim = ExponentialSmoothing(
                series_train_log, trend='add', seasonal=None,
                damped_trend=(phi_hw < 1.0), initialization_method='estimated'
            )
            hw_fit_kw = {'optimized': True, 'use_brute': False}
            if phi_hw < 1.0:
                hw_fit_kw['damping_trend'] = phi_hw
            hw_fit = hw_sim.fit(**hw_fit_kw)
            fc_hw_log = hw_fit.forecast(n_holdout).values
            fc_hw = np.maximum(0, np.expm1(fc_hw_log) if use_log else fc_hw_log)
        except:
            fc_hw = np.full(n_holdout, series_train_raw.mean())

        # [C] SARIMAX
        try:
            if mdl['sarima'] is not None:
                ord_ = mdl['sarima'].model.order if hasattr(mdl['sarima'], 'model') else (1, 1, 1)
                sa_m = SARIMAX(series_train_log, order=ord_,
                               enforce_stationarity=False, enforce_invertibility=False)
                sa_fit = sa_m.fit(disp=False, maxiter=50)
                fc_sa_log = sa_fit.forecast(n_holdout).values
                fc_sa = np.maximum(0, np.expm1(fc_sa_log) if use_log else fc_sa_log)
            else:
                fc_sa = fc_hw
        except:
            fc_sa = fc_hw

        # Geometric Mean Ensemble (sama dengan kode teman)
        preds_ens = []
        for h in range(n_holdout):
            pred_list_, weight_list_ = [], []
            for mn, pv_m in [('Kalman', fc_kf[h]), ('HW', fc_hw[h]), ('SARIMAX', fc_sa[h])]:
                wt = w.get(mn, 0)
                if wt > 0 and pv_m > 0:
                    pred_list_.append(pv_m)
                    weight_list_.append(wt)
            if pred_list_:
                pe = float(np.exp(
                    np.dot(np.array(weight_list_) / sum(weight_list_),
                           np.log(np.maximum(pred_list_, 1e-10)))
                ))
            else:
                pe = float(series_train_raw.mean())
            preds_ens.append(max(0, pe * bias_cf))

        sim_preds[key] = preds_ens

    # F×S Enforcement (85% F×S seperti final pipeline v23)
    for idx in range(n_holdout):
        f_ = sim_preds['frequency'][idx]
        s_ = sim_preds['severity'][idx]
        t_ = sim_preds['total_claim'][idx]
        t_fs = f_ * s_ * GLOBAL_FS_CORRECTION
        ratio = t_ / (t_fs + 1e-10)
        if ratio < 0.65 or ratio > 1.55:
            sim_preds['total_claim'][idx] = t_fs
        elif ratio < 0.80 or ratio > 1.30:
            sim_preds['total_claim'][idx] = 0.90 * t_fs + 0.10 * t_
        else:
            sim_preds['total_claim'][idx] = 0.85 * t_fs + 0.15 * t_

    # ─── Tampilkan Hasil ─────────────────────────────────────────────────────
    print(f"\n  {'Holdout':<10} {'Target':<15} {'Aktual':>15} {'Prediksi':>15} {'Error%':>8}")
    print("  " + "-" * 66)

    all_errors   = []
    target_mapes = {}

    for key in ['frequency', 'severity', 'total_claim']:
        y_true = actuals[key]
        y_pred = sim_preds[key]
        errs_target = []

        for idx in range(n_holdout):
            err = abs(y_true[idx] - y_pred[idx]) / max(y_true[idx], 1e-5) * 100
            errs_target.append(err)
            all_errors.append(err)

            if key == 'frequency':
                yt_d = f"{y_true[idx]:,.0f}"
                yp_d = f"{y_pred[idx]:,.0f}"
            elif key == 'severity':
                yt_d = f"{y_true[idx]/1e6:,.2f}M"
                yp_d = f"{y_pred[idx]/1e6:,.2f}M"
            else:
                yt_d = f"{y_true[idx]/1e9:,.3f}B"
                yp_d = f"{y_pred[idx]/1e9:,.3f}B"

            print(f"  Holdout-{idx+1}   {key:<15} {yt_d:>15} {yp_d:>15} {err:>7.2f}%")

        target_mapes[key] = np.mean(errs_target)
        print("  " + "-" * 66)

    final_sim_score = np.mean(all_errors)

    # ─── Perbandingan langsung dengan teman ──────────────────────────────────
    teman = {'frequency': 5.87, 'severity': 2.44, 'total_claim': 7.12}
    teman_avg = np.mean(list(teman.values()))

    print(f"\n  📊 PERBANDINGAN SIMULASI HOLDOUT (apple-to-apple):")
    print(f"  {'Target':<15} {'Teman v16':>10} {'Kita v23':>10} {'Delta':>8}  Hasil")
    print(f"  {'-'*55}")
    for key, label in [('frequency','Frequency'),('severity','Severity'),('total_claim','Total Claim')]:
        our   = target_mapes[key]
        their = teman[key]
        delta = our - their
        result = "✅ LEBIH BAIK" if delta < 0 else "❌ lebih buruk"
        print(f"  {label:<15} {their:>9.2f}% {our:>9.2f}% {delta:>+7.2f}%  {result}")
    print(f"  {'-'*55}")
    delta_avg = final_sim_score - teman_avg
    result_avg = "✅ LEBIH BAIK" if delta_avg < 0 else "❌ lebih buruk"
    print(f"  {'AVG':<15} {teman_avg:>9.2f}% {final_sim_score:>9.2f}% {delta_avg:>+7.2f}%  {result_avg}")

    print("\n  " + "★" * 60)
    print(f"  ★★★  ESTIMASI SKOR KITA  (v23) : {final_sim_score:.4f}%")
    print(f"  ★★★  ESTIMASI SKOR TEMAN (v16) : {teman_avg:.4f}%")
    print(f"  ★★★  SELISIH                   : {delta_avg:+.4f}%  {result_avg}")
    print("  " + "★" * 60)
    print("═" * 62 + "\n")

hitung_simulasi_v23()
print("=" * 80)

  ✓ LightGBM tersedia
  ✓ Prophet tersedia
  DSC MCF ITB — INSURANCE CLAIM PREDICTION v22.0
  ROOT CAUSE FIX: Kalman OOS eval pakai phi optimal (bukan default 0.98)
  BASE: v21 | Target: AVG MAPE < 3.5%

[1/10] LOADING DATA...
  → 'Data_Klaim (1).csv' (encoding=utf-8, sep=',')
  → 'Data_Polis.csv' (encoding=utf-8, sep=',')
  → 'sample_submission.csv' (encoding=utf-8, sep=',')
  ✓ Data Klaim  : 4,627 × 13
  ✓ Data Polis  : 4,096 × 6
  ✓ Sample IDs  : ['2025_08_Claim_Frequency', '2025_08_Claim_Severity', '2025_08_Total_Claim', '2025_09_Claim_Frequency', '2025_09_Claim_Severity', '2025_09_Total_Claim', '2025_10_Claim_Frequency', '2025_10_Claim_Severity', '2025_10_Total_Claim', '2025_11_Claim_Frequency', '2025_11_Claim_Severity', '2025_11_Total_Claim', '2025_12_Claim_Frequency', '2025_12_Claim_Severity', '2025_12_Total_Claim']

[2/10] PRE-PROCESSING...
  ✓ df_paid     : 4,627 baris
  Best winsor.  : target_w99 (|ratio-1|=0.080)
  PAID values   : ['PAID']

[3/10] EXPLORATORY DATA ANALYSIS..

15:17:15 - cmdstanpy - INFO - Chain [1] start processing
15:17:15 - cmdstanpy - INFO - Chain [1] done processing


    [E] Prophet MAPE=6.29%
    MAPEs: Kalman=9.4% HW=inf% SARIMAX=3.2% STL=6.2% Prophet=6.3%
    Weights: Kalman=0.072 HW=0.000 SARIMAX=0.606 STL=0.163 Prophet=0.159
    Bias correction: ×0.9656

  ─── severity ────────────────────────────────────────────
    [A] Kalman Trend Filter...
    [Kalman] phi=0.850 q_l=0.00000 q_t=0.00000
    [Kalman] phi=0.850 q_l=0.00000 q_t=0.00000
    [Kalman] phi=0.850 q_l=0.00000 q_t=0.00000
    [Kalman] phi=0.850 q_l=0.00000 q_t=0.00000
    [Kalman] phi=0.850 q_l=0.00000 q_t=0.00000
    [Kalman] phi=0.850 q_l=0.00000 q_t=0.00000
    [Kalman] phi=0.850 q_l=0.00000 q_t=0.00000
    [A] Kalman phi=0.850 OOS-MAPE=7.96%
    [B] Optimal Damped HW...
    [B] Optimal φ=0.95 (WF-MAPE=inf%)
    [B] Damped HW OOS-MAPE=inf%
    [C] SARIMAX + external regressors...
    [C] Best SARIMAX(2, 0, 1) AIC=-28.52
    [C] SARIMAX MAPE=3.86%
    [D] STL + Ridge...
    [D] STL+Ridge MAPE=4.87%
    [E] Prophet...


15:17:18 - cmdstanpy - INFO - Chain [1] start processing
15:17:18 - cmdstanpy - INFO - Chain [1] done processing


    [E] Prophet MAPE=5.10%
    MAPEs: Kalman=8.0% HW=inf% SARIMAX=3.9% STL=4.9% Prophet=5.1%
    Weights: Kalman=0.096 HW=0.000 SARIMAX=0.411 STL=0.258 Prophet=0.235
    Bias correction: ×1.0076

  ─── total_claim ────────────────────────────────────────────
    [A] Kalman Trend Filter...
    [Kalman] phi=0.850 q_l=0.00000 q_t=0.00000
    [Kalman] phi=0.850 q_l=0.00000 q_t=0.00000
    [Kalman] phi=0.850 q_l=0.00000 q_t=0.00000
    [Kalman] phi=0.850 q_l=0.00000 q_t=0.00000
    [Kalman] phi=0.850 q_l=0.00000 q_t=0.00000
    [Kalman] phi=0.850 q_l=0.00000 q_t=0.00000
    [Kalman] phi=0.850 q_l=0.00000 q_t=0.00000
    [A] Kalman phi=0.850 OOS-MAPE=12.87%
    [B] Optimal Damped HW...
    [B] Optimal φ=0.95 (WF-MAPE=inf%)
    [B] Damped HW OOS-MAPE=inf%
    [C] SARIMAX + external regressors...
    [C] Best SARIMAX(2, 0, 1) AIC=-29.44
    [C] SARIMAX MAPE=5.31%
    [D] STL + Ridge...
    [D] STL+Ridge MAPE=6.20%
    [E] Prophet...


15:17:20 - cmdstanpy - INFO - Chain [1] start processing
15:17:20 - cmdstanpy - INFO - Chain [1] done processing


    [E] Prophet MAPE=9.94%
    MAPEs: Kalman=12.9% HW=inf% SARIMAX=5.3% STL=6.2% Prophet=9.9%
    Weights: Kalman=0.078 HW=0.000 SARIMAX=0.457 STL=0.335 Prophet=0.130
    Bias correction: ×0.9710

  ─────────────────────────────────────────────────────────────────
  frequency      : Best MAPE=3.22%
  severity       : Best MAPE=3.86%
  total_claim    : Best MAPE=5.31%
  AVG MAPE: 4.13%

  [Generating Model Evaluation Dashboard...]
    [Kalman] phi=0.980 q_l=0.01000 q_t=0.00100
    [Kalman] phi=0.980 q_l=0.01000 q_t=0.00100
    [Kalman] phi=0.980 q_l=0.01000 q_t=0.00100
    [Kalman] phi=0.980 q_l=0.01000 q_t=0.00100
    [Kalman] phi=0.980 q_l=0.01000 q_t=0.00100
    [Kalman] phi=0.980 q_l=0.01000 q_t=0.00100
    [Kalman] phi=0.980 q_l=0.01000 q_t=0.00100
    [Kalman] phi=0.980 q_l=0.01000 q_t=0.00100
    [Kalman] phi=0.980 q_l=0.01000 q_t=0.00100
    [Kalman] phi=0.980 q_l=0.01000 q_t=0.00100
    [Kalman] phi=0.980 q_l=0.01000 q_t=0.00100
    [Kalman] phi=0.980 q_l=0.01000 q_t=0.00100
  

In [ ]:
#3.94-8%

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
import re
from itertools import product
warnings.filterwarnings('ignore')

from scipy import stats
from scipy.optimize import minimize, minimize_scalar, differential_evolution
from scipy.linalg import solve
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                               ExtraTreesRegressor)
from sklearn.linear_model import Ridge, HuberRegressor, Lasso, ElasticNet
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.model_selection import TimeSeriesSplit

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import acf, pacf, adfuller
from statsmodels.tsa.exponential_smoothing.ets import ETSModel

try:
    import lightgbm as lgb
    HAS_LGB = True
    print("  ✓ LightGBM tersedia")
except ImportError:
    HAS_LGB = False
    print("  ℹ LightGBM tidak tersedia")

try:
    import xgboost as xgb
    HAS_XGB = True
    print("  ✓ XGBoost tersedia")
except ImportError:
    HAS_XGB = False
    print("  ℹ XGBoost tidak tersedia")

try:
    from prophet import Prophet
    HAS_PROPHET = True
    print("  ✓ Prophet tersedia")
except ImportError:
    HAS_PROPHET = False
    print("  ℹ Prophet tidak tersedia")

plt.rcParams.update({
    'figure.dpi': 100, 'font.family': 'DejaVu Sans',
    'axes.spines.top': False, 'axes.spines.right': False,
})
COLORS = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#3B1F2B', '#44BBA4',
          '#E84855', '#3D405B', '#81B29A', '#F2CC8F']

# ─── Path Data ───────────────────────────────────────────────────────────────
DATA_KLAIM  = r"Data_Klaim (1).csv"
DATA_POLIS  = r"Data_Polis.csv"
DATA_SAMPLE = r"sample_submission.csv"

print("=" * 80)
print("  DSC MCF ITB — INSURANCE CLAIM PREDICTION v24.0")
print("  KEY UPGRADES: LightGBM/XGBoost + Adaptive Weights + Per-Month Bias")
print("  + Conformal PI + DSHW + Regime Detection + Smarter F×S")
print("  Target: AVG MAPE < 3.0%")
print("=" * 80)


# ========================== HELPER FUNCTIONS =================================

def calc_mape(y_true, y_pred, eps=1.0):
    yt = np.maximum(np.array(y_true, dtype=float), eps)
    yp = np.maximum(np.array(y_pred, dtype=float), 0)
    return float(np.mean(np.abs((yt - yp) / yt)) * 100)

def calc_rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.array(y_true) - np.array(y_pred))**2)))

def safe_kde(ax, data, color, linestyle='-', linewidth=2.0):
    d = np.array(data, dtype=float)
    d = d[np.isfinite(d)]
    if len(d) < 2 or len(np.unique(d)) < 2:
        return
    try:
        kde = stats.gaussian_kde(d)
        xr = np.linspace(d.min(), d.max(), 200)
        ax.plot(xr, kde(xr), color=color, linewidth=linewidth, linestyle=linestyle)
    except:
        pass

def _title(ax, t, fs=11):
    ax.set_title(t, fontsize=fs, fontweight='bold')

def _no_data(ax, msg='Tidak tersedia'):
    ax.text(0.5, 0.5, msg, transform=ax.transAxes,
            ha='center', va='center', fontsize=10, color='gray')

def parse_numeric_safe(series):
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors='coerce')
    def _parse_one(val):
        if pd.isna(val): return np.nan
        s = str(val).strip()
        s = re.sub(r'[^\d.,\-]', '', s)
        if not s: return np.nan
        if ',' in s and '.' in s:
            last_dot, last_comma = s.rfind('.'), s.rfind(',')
            if last_comma > last_dot:
                s = s.replace('.', '').replace(',', '.')
            else:
                s = s.replace(',', '')
        elif ',' in s:
            parts = s.split(',')
            if len(parts) == 2 and len(parts[1]) <= 2:
                s = s.replace(',', '.')
            else:
                s = s.replace(',', '')
        elif '.' in s:
            parts = s.split('.')
            if len(parts) > 2:
                s = s.replace('.', '')
        try: return float(s)
        except: return np.nan
    return series.apply(_parse_one)

def parse_date_safe(series):
    if pd.api.types.is_integer_dtype(series) or pd.api.types.is_float_dtype(series):
        return pd.to_datetime(series.astype(str).str.split('.').str[0],
                              format='%Y%m%d', errors='coerce')
    result = pd.to_datetime(series, errors='coerce', dayfirst=True)
    for fmt in ['%d/%m/%Y', '%m/%d/%Y', '%Y/%m/%d', '%d-%m-%Y',
                '%Y%m%d', '%d %B %Y', '%B %d, %Y', '%Y-%m-%d']:
        missing = result.isna()
        if not missing.any(): break
        try:
            result[missing] = pd.to_datetime(series[missing], format=fmt, errors='coerce')
        except: pass
    return result

def find_col(df, keywords, required=False):
    cols_lower = {c.lower().strip(): c for c in df.columns}
    for kw in keywords:
        kw_l = kw.lower()
        if kw_l in cols_lower: return cols_lower[kw_l]
        for cl, corig in cols_lower.items():
            if kw_l in cl or cl in kw_l: return corig
    if required:
        raise KeyError(f"Kolom tidak ditemukan. Keywords: {keywords}")
    return None

def read_csv_robust(path):
    for enc in ['utf-8', 'utf-8-sig', 'latin-1', 'cp1252', 'iso-8859-1']:
        for sep in [',', ';', '\t', '|']:
            try:
                df = pd.read_csv(path, encoding=enc, sep=sep, low_memory=False)
                if df.shape[1] > 1:
                    df.columns = df.columns.str.strip()
                    print(f"  → '{path}' (encoding={enc}, sep='{sep}')")
                    return df
            except: continue
    raise ValueError(f"Tidak bisa membaca: {path}")


# ========================== [v24] REGIME DETECTION ==========================

def detect_regime(series, window=6):
    """
    [v24-J] Detect if series is in HIGH/LOW/NORMAL regime.
    Used for adaptive YoY blending and floor/ceiling adjustments.
    Returns: 'high', 'low', 'normal', 'transition_up', 'transition_down'
    """
    series = np.array(series, dtype=float)
    n = len(series)
    if n < window * 2:
        return 'normal', 1.0

    recent   = series[-window:]
    older    = series[-2*window:-window]
    long_avg = series.mean()

    recent_mean = recent.mean()
    older_mean  = older.mean()
    ratio_vs_long = recent_mean / (long_avg + 1e-10)
    ratio_recent  = recent_mean / (older_mean + 1e-10)

    # Detect regime
    if ratio_recent > 1.15 and ratio_vs_long > 1.10:
        return 'transition_up', float(ratio_recent)
    elif ratio_recent < 0.85 and ratio_vs_long < 0.90:
        return 'transition_down', float(ratio_recent)
    elif ratio_vs_long > 1.20:
        return 'high', float(ratio_vs_long)
    elif ratio_vs_long < 0.80:
        return 'low', float(ratio_vs_long)
    else:
        return 'normal', float(ratio_vs_long)


# ========================== [KALMAN FILTERS] =================================

class SimpleKalmanFilter:
    def __init__(self, q=0.1):
        self.q = q
        self.level_ = None
        self.P_ = None
        self.fitted_values_ = None
        self.resid_ = None

    def fit(self, y, optimize_q=True):
        y = np.array(y, dtype=float)
        n = len(y)
        if optimize_q:
            def neg_ll(log_q):
                q_ = np.exp(log_q[0])
                _, ll = self._run_filter(y, q_)
                return -ll
            try:
                res = minimize(neg_ll, [np.log(0.1)], method='Nelder-Mead',
                               options={'xatol': 1e-6, 'fatol': 1e-6, 'maxiter': 200})
                self.q = np.exp(res.x[0])
            except:
                self.q = 0.1
        levels, Ps, innov = self._run_filter(y, self.q, return_levels=True)
        self.level_      = levels[-1]
        self.P_          = Ps[-1]
        self.fitted_values_ = levels
        self.resid_      = y - np.array(levels)
        self.y_          = y
        return self

    def _run_filter(self, y, q, return_levels=False):
        n = len(y)
        level = y[0]
        P = 1.0
        levels = []
        Ps = []
        log_likelihood = 0.0
        for t in range(n):
            level_pred = level
            P_pred = P + q
            innov = y[t] - level_pred
            S = P_pred + 1.0
            if S > 0:
                log_likelihood += -0.5 * (np.log(2 * np.pi * S) + innov**2 / S)
            K = P_pred / S
            level = level_pred + K * innov
            P = (1 - K) * P_pred
            levels.append(level)
            Ps.append(P)
        if return_levels:
            return levels, Ps, None
        return None, log_likelihood

    def forecast(self, h):
        return np.array([self.level_] * h)


class KalmanTrendFilter:
    """Local Linear Trend Kalman Filter with damped trend."""
    def __init__(self, phi=0.95, q_level=0.01, q_trend=0.001, q_obs=1.0):
        self.phi = phi
        self.q_level = q_level
        self.q_trend = q_trend
        self.q_obs   = q_obs
        self.level_  = None
        self.trend_  = None
        self.fitted_values_ = None

    def fit(self, y, optimize=True):
        y = np.array(y, dtype=float)
        if optimize:
            def neg_ll(params):
                phi = np.clip(params[0], 0.85, 1.00)
                q_l = np.exp(params[1])
                q_t = np.exp(params[2])
                try:
                    ll = self._run_filter_ll(y, phi, q_l, q_t, self.q_obs)
                    return -ll
                except:
                    return 1e10
            try:
                res = minimize(
                    neg_ll,
                    [0.95, np.log(0.01), np.log(0.001)],
                    method='Nelder-Mead',
                    options={'xatol': 1e-5, 'fatol': 1e-5, 'maxiter': 500}
                )
                self.phi     = np.clip(res.x[0], 0.85, 1.00)
                self.q_level = np.exp(res.x[1])
                self.q_trend = np.exp(res.x[2])
            except:
                pass
        levels, trends = self._run_filter(y, self.phi, self.q_level, self.q_trend, self.q_obs)
        self.level_ = levels[-1]
        self.trend_ = trends[-1]
        self.fitted_values_ = np.array(levels)
        self.resid_ = y - self.fitted_values_
        print(f"    [Kalman] phi={self.phi:.3f} q_l={self.q_level:.5f} q_t={self.q_trend:.5f}")
        return self

    def _run_filter_ll(self, y, phi, q_l, q_t, q_obs):
        n = len(y)
        state = np.array([y[0], 0.0])
        P = np.eye(2) * 10.0
        F = np.array([[1, 1], [0, phi]])
        Q = np.diag([q_l, q_t])
        H = np.array([[1, 0]])
        R = np.array([[q_obs]])
        ll = 0.0
        for t in range(n):
            state_pred = F @ state
            P_pred     = F @ P @ F.T + Q
            innov = y[t] - float(H @ state_pred)
            S     = float(H @ P_pred @ H.T + R)
            if S <= 0: return -1e10
            ll += -0.5 * (np.log(2*np.pi*S) + innov**2/S)
            K     = (P_pred @ H.T) / S
            state = state_pred + K.flatten() * innov
            P     = (np.eye(2) - K @ H) @ P_pred
        return ll

    def _run_filter(self, y, phi, q_l, q_t, q_obs):
        n = len(y)
        state = np.array([y[0], 0.0])
        P = np.eye(2) * 10.0
        F = np.array([[1, 1], [0, phi]])
        Q = np.diag([q_l, q_t])
        H = np.array([[1, 0]])
        R = np.array([[q_obs]])
        levels = []
        trends = []
        for t in range(n):
            state_pred = F @ state
            P_pred     = F @ P @ F.T + Q
            innov = y[t] - float(H @ state_pred)
            S     = float(H @ P_pred @ H.T + R)
            K = (P_pred @ H.T) / S
            state = state_pred + K.flatten() * innov
            P = (np.eye(2) - K @ H) @ P_pred
            levels.append(float(state[0]))
            trends.append(float(state[1]))
        return levels, trends

    def forecast(self, h):
        phi   = self.phi
        level = self.level_
        trend = self.trend_
        forecasts = []
        curr_level = level
        curr_trend = trend
        for i in range(h):
            curr_level = curr_level + curr_trend
            curr_trend = phi * curr_trend
            forecasts.append(curr_level)
        return np.array(forecasts)


# ========================== [v24-C] CONFORMAL PREDICTION ====================

class ConformalPrediction:
    """
    [v24-C] Conformal prediction intervals for time series.
    Uses split conformal approach with residuals from calibration set.
    Provides valid coverage guarantees without distributional assumptions.
    """
    def __init__(self, alpha=0.10):
        self.alpha = alpha  # 1-alpha = coverage level (default: 90%)
        self.quantile_ = None
        self.calibration_scores_ = None

    def calibrate(self, y_true, y_pred):
        """Compute nonconformity scores on calibration set."""
        y_true = np.array(y_true, dtype=float)
        y_pred = np.array(y_pred, dtype=float)
        # Use absolute residuals as nonconformity scores
        self.calibration_scores_ = np.abs(y_true - y_pred)
        n = len(self.calibration_scores_)
        # Finite-sample corrected quantile
        level = np.ceil((n + 1) * (1 - self.alpha)) / n
        level = min(level, 1.0)
        self.quantile_ = float(np.quantile(self.calibration_scores_, level))
        return self

    def predict_interval(self, y_pred):
        """Return (lower, upper) conformal prediction interval."""
        if self.quantile_ is None:
            raise ValueError("Call calibrate() first")
        y_pred = np.atleast_1d(np.array(y_pred, dtype=float))
        lower = np.maximum(0, y_pred - self.quantile_)
        upper = y_pred + self.quantile_
        return lower, upper


# ========================== [v24] OPTIMAL DAMPING ===========================

def find_optimal_damping(series, phi_grid=None):
    if phi_grid is None:
        phi_grid = [0.80, 0.85, 0.88, 0.90, 0.92, 0.95, 0.97, 0.98, 0.99, 1.00]
    series = np.array(series, dtype=float)
    n = len(series)
    min_train = max(8, n - 8)
    best_phi  = 0.95
    best_mape = np.inf
    phi_mapes = {}
    for phi in phi_grid:
        wf_mapes = []
        for test_idx in range(min_train, n):
            y_train = series[:test_idx]
            y_true  = series[test_idx]
            if len(y_train) < 6: continue
            try:
                if phi < 1.0:
                    model = ExponentialSmoothing(
                        y_train, trend='add', seasonal=None,
                        damped_trend=True, initialization_method='estimated'
                    )
                    fit = model.fit(damping_trend=phi, optimized=True, use_brute=False)
                else:
                    model = ExponentialSmoothing(
                        y_train, trend='add', seasonal=None,
                        damped_trend=False, initialization_method='estimated'
                    )
                    fit = model.fit(optimized=True, use_brute=False)
                pred = float(fit.forecast(1).iloc[0])
                pred = max(0, pred)
                wf_mapes.append(calc_mape([y_true], [pred]))
            except:
                continue
        if wf_mapes:
            avg_mape = np.mean(wf_mapes)
            phi_mapes[phi] = avg_mape
            if avg_mape < best_mape:
                best_mape = avg_mape
                best_phi  = phi
    return best_phi, best_mape, phi_mapes


# ========================== [v24] EXTERNAL DATA ==============================

def build_external_data(year_start=2024, month_start=1, n_months=None,
                        forecast_months=None, forecast_year=2025):
    all_periods = []
    yr, mo = year_start, month_start
    max_iter = 60
    while max_iter > 0:
        all_periods.append((yr, mo))
        mo += 1
        if mo > 12:
            mo = 1
            yr += 1
        max_iter -= 1
        if n_months and len(all_periods) >= n_months:
            break
    if forecast_months:
        for fm in forecast_months:
            all_periods.append((forecast_year, fm))

    records = []
    base_inflation = 0.007
    ramadan_months  = {(2024, 3): 1, (2025, 3): 1}
    eid_months      = {(2024, 4): 1, (2025, 4): 1}
    adha_months     = {(2024, 6): 0.5, (2025, 6): 0.5}
    monthly_seasonality = {
        1: 0.85, 2: 0.90, 3: 1.00, 4: 1.05, 5: 0.95,
        6: 1.10, 7: 1.00, 8: 0.95, 9: 1.00, 10: 1.05,
        11: 1.10, 12: 1.15,
    }
    for i, (yr, mo) in enumerate(all_periods):
        months_since_start = (yr - year_start) * 12 + (mo - month_start)
        inflation_idx = (1 + base_inflation) ** months_since_start
        record = {
            'year': yr, 'month': mo,
            'time_idx': months_since_start,
            'inflation_idx': inflation_idx,
            'inflation_mom': base_inflation,
            'is_ramadan':   int((yr, mo) in ramadan_months),
            'is_eid_fitr':  int((yr, mo) in eid_months),
            'is_eid_adha':  int((yr, mo) in adha_months),
            'is_yearend':   int(mo in [11, 12]),
            'is_yearstart': int(mo == 1),
            'is_q1':        int(mo in [1, 2, 3]),
            'is_q4':        int(mo in [10, 11, 12]),
            'is_holiday':   int(mo in [1, 4, 6, 7, 12]),
            'seasonal_prior': monthly_seasonality.get(mo, 1.0),
            'linear_trend': months_since_start / 12.0,
            'trend_sq':     (months_since_start / 12.0) ** 2,
            'sin_month': np.sin(2 * np.pi * mo / 12),
            'cos_month': np.cos(2 * np.pi * mo / 12),
            'sin_qtr':   np.sin(2 * np.pi * ((mo-1)//3 + 1) / 4),
            'cos_qtr':   np.cos(2 * np.pi * ((mo-1)//3 + 1) / 4),
        }
        records.append(record)
    return pd.DataFrame(records)


# ========================== [v24-B] LGBM/XGB TIME SERIES MODEL ==============

def build_ts_features(series, ext_vals=None, lookback=6):
    """
    [v24-B] Build supervised learning features from time series.
    Uses lag features + rolling stats + external regressors.
    """
    series = np.array(series, dtype=float)
    n = len(series)
    feats = []
    for t in range(lookback, n):
        row = {}
        # Lag features
        for lag in range(1, lookback + 1):
            row[f'lag_{lag}'] = series[t - lag]
        # Rolling stats
        window3 = series[max(0, t-3):t]
        window6 = series[max(0, t-6):t]
        row['roll_mean_3'] = window3.mean()
        row['roll_std_3']  = window3.std() if len(window3) > 1 else 0
        row['roll_mean_6'] = window6.mean()
        row['roll_std_6']  = window6.std() if len(window6) > 1 else 0
        row['roll_max_3']  = window3.max()
        row['roll_min_3']  = window3.min()
        # Trend features
        row['t_idx'] = t
        row['sin_t'] = np.sin(2 * np.pi * t / 12)
        row['cos_t'] = np.cos(2 * np.pi * t / 12)
        row['sin_t6'] = np.sin(2 * np.pi * t / 6)
        row['cos_t6'] = np.cos(2 * np.pi * t / 6)
        # Momentum
        if t >= 2:
            row['mom_1'] = series[t-1] - series[t-2]
            row['mom_pct'] = (series[t-1] / (series[t-2] + 1e-10)) - 1
        else:
            row['mom_1'] = 0
            row['mom_pct'] = 0
        # External regressors
        if ext_vals is not None:
            for j, v in enumerate(ext_vals[t]):
                row[f'ext_{j}'] = v
        feats.append(row)
    return pd.DataFrame(feats), series[lookback:]


def train_lgbm_model(series, ext_vals=None, lookback=6, use_log=True):
    """
    [v24-B] Train LightGBM model for time series forecasting.
    Uses expanding window cross-validation for OOS MAPE estimation.
    """
    if not HAS_LGB:
        return None, 99.0

    y = np.log1p(np.maximum(series, 0)) if use_log else series.copy()
    X_df, y_arr = build_ts_features(y, ext_vals=ext_vals, lookback=lookback)

    if len(X_df) < 5:
        return None, 99.0

    n_cv = min(4, len(X_df) // 3)
    oos_preds, oos_actuals = [], []

    for i in range(n_cv):
        train_end = len(X_df) - n_cv + i
        if train_end < lookback: continue
        X_tr = X_df.iloc[:train_end]
        y_tr = y_arr[:train_end]
        X_te = X_df.iloc[train_end:train_end+1]
        y_te = y_arr[train_end]

        try:
            m = lgb.LGBMRegressor(
                n_estimators=200, learning_rate=0.05, max_depth=4,
                num_leaves=15, min_child_samples=3,
                subsample=0.8, colsample_bytree=0.8,
                reg_alpha=0.1, reg_lambda=0.1,
                random_state=42, verbose=-1
            )
            m.fit(X_tr, y_tr)
            pred_log = float(m.predict(X_te)[0])
            pred_raw = max(0, np.expm1(pred_log) if use_log else pred_log)
            oos_preds.append(pred_raw)
            oos_actuals.append(np.expm1(y_te) if use_log else y_te)
        except:
            continue

    mape_lgb = calc_mape(oos_actuals, oos_preds) if oos_preds else 99.0

    # Train final model on all data
    try:
        final_model = lgb.LGBMRegressor(
            n_estimators=300, learning_rate=0.03, max_depth=4,
            num_leaves=15, min_child_samples=3,
            subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1,
            random_state=42, verbose=-1
        )
        final_model.fit(X_df, y_arr)
    except:
        return None, 99.0

    return final_model, mape_lgb


def forecast_lgbm(model, series, ext_fc_row=None, h=1, lookback=6, use_log=True):
    """
    [v24-B] Generate h-step ahead forecast from LGBM model.
    Uses recursive strategy for multi-step ahead.
    """
    if model is None:
        return float(np.mean(series))

    y = np.log1p(np.maximum(series, 0)) if use_log else series.copy()
    curr_series = list(y)
    forecasts = []

    for step in range(h):
        if len(curr_series) < lookback:
            forecasts.append(float(np.mean(series)))
            continue

        row = {}
        for lag in range(1, lookback + 1):
            row[f'lag_{lag}'] = curr_series[-lag]
        window3 = curr_series[-3:]
        window6 = curr_series[-6:]
        row['roll_mean_3'] = np.mean(window3)
        row['roll_std_3']  = np.std(window3) if len(window3) > 1 else 0
        row['roll_mean_6'] = np.mean(window6)
        row['roll_std_6']  = np.std(window6) if len(window6) > 1 else 0
        row['roll_max_3']  = np.max(window3)
        row['roll_min_3']  = np.min(window3)
        t = len(curr_series)
        row['t_idx'] = t
        row['sin_t'] = np.sin(2 * np.pi * t / 12)
        row['cos_t'] = np.cos(2 * np.pi * t / 12)
        row['sin_t6'] = np.sin(2 * np.pi * t / 6)
        row['cos_t6'] = np.cos(2 * np.pi * t / 6)
        if len(curr_series) >= 2:
            row['mom_1'] = curr_series[-1] - curr_series[-2]
            row['mom_pct'] = (curr_series[-1] / (curr_series[-2] + 1e-10)) - 1
        else:
            row['mom_1'] = 0
            row['mom_pct'] = 0
        if ext_fc_row is not None:
            for j, v in enumerate(ext_fc_row):
                row[f'ext_{j}'] = v

        X_next = pd.DataFrame([row])
        # Align with training columns
        for col in model.feature_name_:
            if col not in X_next.columns:
                X_next[col] = 0
        X_next = X_next[model.feature_name_]

        try:
            pred_log = float(model.predict(X_next)[0])
            pred_raw = max(0, np.expm1(pred_log) if use_log else pred_log)
        except:
            pred_raw = float(np.mean(series))

        forecasts.append(pred_raw)
        curr_series.append(np.log1p(pred_raw) if use_log else pred_raw)

    return forecasts[-1] if h == 1 else forecasts


# ========================== [v24] ADAPTIVE ENSEMBLE WEIGHTS =================

def compute_adaptive_weights_v24(mapes_dict, wf_results_dict, regime='normal',
                                  method='inv_mape2'):
    """
    [v24] Compute ensemble weights with multiple strategies:
    1. inv_mape2: Inverse MAPE squared (default)
    2. softmax: Softmax of negative MAPE
    3. rank: Rank-based weights
    4. min_mape: Winner-takes-most (80% to best, 20% rest)

    Also applies regime-based adjustments:
    - 'transition_up': Boost Kalman & LightGBM (trend-following)
    - 'transition_down': Boost HW (mean reverting)
    - 'high'/'low': Boost SARIMAX (structural)
    - 'normal': Use pure MAPE-based weights
    """
    valid = {k: v for k, v in mapes_dict.items() if v < 60.0 and v > 0}
    if not valid:
        valid = {'HW': 10.0}

    # Compute base weights
    if method == 'inv_mape2':
        total = sum(1.0 / (v**2 + 1e-6) for v in valid.values())
        weights = {k: (1.0 / (v**2 + 1e-6)) / total for k, v in valid.items()}
    elif method == 'softmax':
        neg_mapes = {k: -v for k, v in valid.items()}
        max_neg = max(neg_mapes.values())
        exp_vals = {k: np.exp(v - max_neg) for k, v in neg_mapes.items()}
        total = sum(exp_vals.values())
        weights = {k: v / total for k, v in exp_vals.items()}
    elif method == 'rank':
        sorted_models = sorted(valid.items(), key=lambda x: x[1])
        n_m = len(sorted_models)
        rank_weights = {k: (n_m - i) for i, (k, _) in enumerate(sorted_models)}
        total = sum(rank_weights.values())
        weights = {k: v / total for k, v in rank_weights.items()}
    elif method == 'min_mape':
        best_k = min(valid, key=valid.get)
        weights = {k: 0.05 for k in valid}
        weights[best_k] = 1.0 - 0.05 * (len(valid) - 1)
        total = sum(weights.values())
        weights = {k: v / total for k, v in weights.items()}
    else:
        weights = {k: 1.0/len(valid) for k in valid}

    # Add zeros for invalid models
    all_keys = list(mapes_dict.keys())
    final_weights = {k: weights.get(k, 0.0) for k in all_keys}

    # Regime adjustments (soft nudge, not override)
    regime_boost = {}
    if regime == 'transition_up':
        regime_boost = {'Kalman': 1.3, 'LightGBM': 1.2, 'XGBoost': 1.2}
    elif regime == 'transition_down':
        regime_boost = {'HW': 1.3, 'SARIMAX': 1.1}
    elif regime in ['high', 'low']:
        regime_boost = {'SARIMAX': 1.2, 'STL': 1.1}

    if regime_boost:
        for k, boost in regime_boost.items():
            if k in final_weights and final_weights[k] > 0:
                final_weights[k] *= boost
        total = sum(final_weights.values())
        if total > 0:
            final_weights = {k: v / total for k, v in final_weights.items()}

    return final_weights


# ========================== [v24-D] ADAPTIVE F×S ENFORCEMENT ================

def adaptive_fs_enforcement(f_pred, s_pred, t_pred, global_cf, monthly_cf,
                              pred_month, wf_mape_freq, wf_mape_sev, wf_mape_total):
    """
    [v24-D] Adaptive F×S enforcement.
    Instead of flat 85%/15%, compute weights based on:
    - How reliable is F model vs S model vs Total model
    - Per-month historical F×S correction factor
    - Uncertainty of each component

    Lower MAPE → more reliable → higher weight in final total
    """
    t_fs_global = f_pred * s_pred * global_cf

    # Get per-month correction factor if available
    month_cf = monthly_cf.get(pred_month, global_cf)
    t_fs_monthly = f_pred * s_pred * month_cf

    # Reliability scores (lower MAPE = higher reliability)
    rel_freq   = 1.0 / (wf_mape_freq**2   + 1e-6)
    rel_sev    = 1.0 / (wf_mape_sev**2    + 1e-6)
    rel_total  = 1.0 / (wf_mape_total**2  + 1e-6)
    rel_fs_comb = rel_freq * rel_sev  # F×S is product of two components

    # F×S vs Total weights
    fs_weight = rel_fs_comb / (rel_fs_comb + rel_total)
    t_weight  = rel_total   / (rel_fs_comb + rel_total)

    # Blend global and per-month F×S
    t_fs_blended = 0.6 * t_fs_global + 0.4 * t_fs_monthly

    # Final blend
    t_final = fs_weight * t_fs_blended + t_weight * t_pred

    # Sanity check ratio
    ratio = t_final / (t_fs_global + 1e-10)
    if ratio < 0.6 or ratio > 1.6:
        t_final = t_fs_global  # fallback to pure F×S
        ratio = 1.0

    return max(0, t_final), float(ratio), float(fs_weight)


# ========================== [v24-G] PER-MONTH BIAS CORRECTION ===============

def compute_per_month_bias(monthly_df, tgt_col, model_preds_insample, months_col='bulan_masuk'):
    """
    [v24-G] Compute per-month bias correction factor.
    For each month (1-12), compute the average residual ratio.
    Returns dict: {month: correction_factor}
    """
    month_bias = {}
    for mo in range(1, 13):
        mask = monthly_df[months_col] == mo
        if mask.sum() == 0:
            month_bias[mo] = 1.0
            continue
        actuals = monthly_df.loc[mask, tgt_col].values
        idxs = monthly_df.index[mask].tolist()
        preds = []
        for idx in idxs:
            pos = monthly_df.index.get_loc(idx)
            if pos < len(model_preds_insample):
                preds.append(model_preds_insample[pos])
            else:
                preds.append(np.nan)
        preds = np.array(preds, dtype=float)
        valid = ~np.isnan(preds) & (preds > 0) & (actuals > 0)
        if valid.sum() >= 1:
            ratios = actuals[valid] / preds[valid]
            month_bias[mo] = float(np.median(ratios))
        else:
            month_bias[mo] = 1.0
    return month_bias


# ========================== LOAD & PREPROCESS ===============================
print("\n[1/10] LOADING DATA...")

df_klaim  = read_csv_robust(DATA_KLAIM)
df_polis  = read_csv_robust(DATA_POLIS)
df_sample = read_csv_robust(DATA_SAMPLE)

print(f"  ✓ Data Klaim  : {df_klaim.shape[0]:,} × {df_klaim.shape[1]}")
print(f"  ✓ Data Polis  : {df_polis.shape[0]:,} × {df_polis.shape[1]}")
print(f"  ✓ Sample IDs  : {df_sample['id'].tolist()}")

print("\n[2/10] PRE-PROCESSING...")

COL_POLIS_K    = find_col(df_klaim, ['nomor polis', 'nopolis', 'polis_id', 'polis'], required=True)
COL_STATUS     = find_col(df_klaim, ['status klaim', 'status claim', 'status'], required=True)
COL_TARGET     = find_col(df_klaim, ['nominal klaim yang disetujui', 'klaim disetujui',
                                     'nominal disetujui', 'approved claim', 'disetujui'])
COL_TGL_MASUK  = find_col(df_klaim, ['tanggal pasien masuk rs', 'tanggal pasien masuk',
                                     'tgl masuk', 'masuk rs', 'admission date', 'masuk'])
COL_TGL_KELUAR = find_col(df_klaim, ['tanggal pasien keluar rs', 'tanggal pasien keluar',
                                     'tgl keluar', 'keluar rs', 'discharge date', 'keluar'])
COL_TGL_BAYAR  = find_col(df_klaim, ['tanggal pembayaran klaim', 'tanggal pembayaran',
                                     'tgl bayar', 'payment date', 'pembayaran'])
COL_ICD        = find_col(df_klaim, ['icd diagnosis', 'icd_diagnosis', 'icd code', 'icd'])
COL_IP_OP      = find_col(df_klaim, ['inpatient/outpatient', 'rawat inap', 'ip_op', 'rawat'])
COL_LOKASI_RS  = find_col(df_klaim, ['lokasi rs', 'lokasi rumah sakit', 'lokasi'])
COL_BIAYA_RS   = find_col(df_klaim, ['nominal biaya rs yang terjadi', 'nominal biaya rs',
                                     'biaya rs', 'biaya'])
COL_REIMBURSE  = find_col(df_klaim, ['reimburse/cashless', 'reimburse', 'cashless'])

COL_POLIS_P   = find_col(df_polis, ['nomor polis', 'nopolis', 'polis_id', 'polis'], required=True)
COL_TGL_LAHIR = find_col(df_polis, ['tanggal lahir', 'tgl lahir', 'birth date', 'dob'])
COL_TGL_EFF   = find_col(df_polis, ['tanggal efektif polis', 'tanggal efektif', 'efektif'])
COL_PLAN      = find_col(df_polis, ['plan code', 'plancode', 'plan'])
COL_GENDER    = find_col(df_polis, ['gender', 'jenis kelamin', 'sex'])

# Parse numeric & dates
df_klaim[COL_TARGET] = parse_numeric_safe(df_klaim[COL_TARGET])
if COL_BIAYA_RS and COL_BIAYA_RS in df_klaim.columns:
    df_klaim[COL_BIAYA_RS] = parse_numeric_safe(df_klaim[COL_BIAYA_RS])

for col in [c for c in [COL_TGL_MASUK, COL_TGL_KELUAR, COL_TGL_BAYAR]
            if c and c in df_klaim.columns]:
    df_klaim[col] = parse_date_safe(df_klaim[col])

for col in [c for c in [COL_TGL_LAHIR, COL_TGL_EFF]
            if c and c in df_polis.columns]:
    df_polis[col] = parse_date_safe(df_polis[col])

df = df_klaim.merge(df_polis, left_on=COL_POLIS_K, right_on=COL_POLIS_P, how='left')
if COL_POLIS_K != COL_POLIS_P and COL_POLIS_P in df.columns:
    df = df.drop(columns=[COL_POLIS_P])

if (COL_TGL_BAYAR and COL_TGL_MASUK and
        COL_TGL_BAYAR in df.columns and COL_TGL_MASUK in df.columns):
    known = df[COL_TGL_BAYAR].notna() & df[COL_TGL_MASUK].notna()
    med_lag = 14
    if known.sum() > 0:
        lag = (df.loc[known, COL_TGL_BAYAR] - df.loc[known, COL_TGL_MASUK]).dt.days
        if not lag.isna().all():
            med_lag = int(np.clip(lag.median(), 0, 90))
    paid_na = df[COL_STATUS].isin(['PAID']) & df[COL_TGL_BAYAR].isna()
    df.loc[paid_na, COL_TGL_BAYAR] = (
        df.loc[paid_na, COL_TGL_MASUK] + pd.Timedelta(days=med_lag))

for col in df.select_dtypes(include=[np.number]).columns:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())
for col in df.select_dtypes(include='object').columns:
    if df[col].isna().any():
        mode_ = df[col].mode()
        df[col] = df[col].fillna(mode_.iloc[0] if not mode_.empty else 'Unknown')

# Filter PAID
status_counts = df[COL_STATUS].value_counts()
PAID_VALUES = [v for v in status_counts.index
               if any(kw in str(v).lower()
                      for kw in ['paid', 'dibayar', 'approved', 'disetujui', 'settled'])]
if not PAID_VALUES:
    PAID_VALUES = [status_counts.index[0]]

df_paid = df[df[COL_STATUS].isin(PAID_VALUES)].copy().reset_index(drop=True)
df_paid[COL_TARGET] = parse_numeric_safe(df_paid[COL_TARGET])

masuk_col = (COL_TGL_MASUK if COL_TGL_MASUK and COL_TGL_MASUK in df_paid.columns
             else COL_TGL_BAYAR)
df_paid[masuk_col] = pd.to_datetime(df_paid[masuk_col], errors='coerce')

valid_dates = df_paid[masuk_col].dropna().sort_values()
if df_paid[masuk_col].isna().any() and len(valid_dates) > 0:
    med_date = valid_dates.iloc[len(valid_dates) // 2]
    df_paid[masuk_col] = df_paid[masuk_col].fillna(med_date)

df_paid['tahun']      = df_paid[masuk_col].dt.year.fillna(2024).astype(int)
df_paid['bulan']      = df_paid[masuk_col].dt.month.fillna(1).astype(int)
df_paid['year_month'] = df_paid[masuk_col].dt.to_period('M')

TARGET_COL = COL_TARGET

# [v24] Adaptive Winsorization + Severity Mixture
# [v24-I] Detect bimodal distribution → use separate winsorization for IP vs OP
for pct in [(0.01, 0.99), (0.02, 0.98), (0.05, 0.95)]:
    col_name = f'target_w{int(pct[1]*100)}'
    df_paid[col_name] = df_paid[TARGET_COL].clip(
        lower=df_paid[TARGET_COL].quantile(pct[0]),
        upper=df_paid[TARGET_COL].quantile(pct[1])
    )

ratios = {}
for win_col in ['target_w99', 'target_w98', 'target_w95']:
    tmp = df_paid.groupby('year_month').agg(
        freq=(TARGET_COL, 'count'),
        sev_w=(win_col, 'mean'),
        total=(TARGET_COL, 'sum')
    )
    ratio_ = (tmp['total'] / (tmp['freq'] * tmp['sev_w'] + 1e-10)).median()
    ratios[win_col] = abs(ratio_ - 1.0)

best_win_col = min(ratios, key=ratios.get)
df_paid['target_wins'] = df_paid[best_win_col].copy()
TARGET_WINS_COL = 'target_wins'

print(f"  ✓ df_paid     : {len(df_paid):,} baris")
print(f"  Best winsor.  : {best_win_col} (|ratio-1|={ratios[best_win_col]:.3f})")
print(f"  PAID values   : {PAID_VALUES}")


# ========================== 3. EDA ==========================================
print("\n[3/10] EXPLORATORY DATA ANALYSIS...")

mf_eda = df_paid.groupby('year_month').agg(
    freq     = (TARGET_COL, 'count'),
    sev_mean = (TARGET_COL, 'mean'),
    sev_wins = ('target_wins', 'mean'),
    sev_med  = (TARGET_COL, 'median'),
    total    = (TARGET_COL, 'sum'),
    p25      = (TARGET_COL, lambda x: x.quantile(.25)),
    p75      = (TARGET_COL, lambda x: x.quantile(.75))
).reset_index()
mf_eda['ym_str'] = mf_eda['year_month'].astype(str)
mf_eda['ym_dt']  = pd.to_datetime(mf_eda['ym_str'])

ratio_train = (mf_eda['total'] / (mf_eda['freq'] * mf_eda['sev_wins'] + 1e-10)).median()
print(f"  F×S ratio (global)   : {ratio_train:.3f}")

start_yr = int(str(mf_eda['year_month'].min())[:4])
start_mo = int(str(mf_eda['year_month'].min())[5:7])
df_ext_train = build_external_data(
    year_start=start_yr, month_start=start_mo, n_months=len(mf_eda)
)
print(f"  External data shape  : {df_ext_train.shape}")

# [v24-J] Regime detection
regime_freq, ratio_freq = detect_regime(mf_eda['freq'].values)
regime_sev,  ratio_sev  = detect_regime(mf_eda['sev_wins'].values)
print(f"  Frequency regime  : {regime_freq} (ratio={ratio_freq:.3f})")
print(f"  Severity  regime  : {regime_sev}  (ratio={ratio_sev:.3f})")

# Minimal EDA plot
try:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('DSC MCF ITB — EDA v24.0', fontsize=13, fontweight='bold')

    # Plot 1: Frequency
    ax = axes[0, 0]
    ax.plot(range(len(mf_eda)), mf_eda['freq'], color=COLORS[0], lw=2, marker='o', ms=5)
    ax.set_title('Frekuensi Klaim'); ax.set_ylabel('Klaim/bulan')

    # Plot 2: Severity
    ax = axes[0, 1]
    ax.plot(range(len(mf_eda)), mf_eda['sev_wins']/1e6, color=COLORS[1], lw=2, marker='s', ms=5)
    ax.set_title('Severitas Wins (Juta IDR)'); ax.set_ylabel('Juta IDR')

    # Plot 3: Total
    ax = axes[0, 2]
    ax.plot(range(len(mf_eda)), mf_eda['total']/1e9, color=COLORS[2], lw=2, marker='^', ms=5)
    ax.set_title('Total Klaim (Miliar IDR)'); ax.set_ylabel('Miliar IDR')

    # Plot 4: ACF
    ax = axes[1, 0]
    try:
        acf_v = acf(mf_eda['freq'].values, nlags=min(len(mf_eda)-2, 12), fft=True)
        ax.bar(range(len(acf_v)), acf_v, color=COLORS[0], alpha=0.7)
        ci = 1.96 / np.sqrt(len(mf_eda))
        ax.axhline(y=ci, color='red', ls='--'); ax.axhline(y=-ci, color='red', ls='--')
    except: pass
    ax.set_title('ACF Frequency')

    # Plot 5: External seasonal
    ax = axes[1, 1]
    ax.bar(range(1, 13), [df_ext_train[df_ext_train['month']==m]['seasonal_prior'].iloc[0]
                           if len(df_ext_train[df_ext_train['month']==m])>0 else 1.0
                           for m in range(1,13)],
           color=COLORS[2], alpha=0.7)
    ax.axhline(y=1.0, color='red', lw=1, ls='--')
    ax.set_title('Prior Seasonal Index'); ax.set_xticks(range(1,13))

    # Plot 6: Regime info
    ax = axes[1, 2]
    ax.text(0.5, 0.7, f'Freq Regime:\n{regime_freq}\n(ratio={ratio_freq:.2f})',
            transform=ax.transAxes, ha='center', fontsize=12, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor=COLORS[0], alpha=0.2))
    ax.text(0.5, 0.3, f'Sev Regime:\n{regime_sev}\n(ratio={ratio_sev:.2f})',
            transform=ax.transAxes, ha='center', fontsize=12, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor=COLORS[1], alpha=0.2))
    ax.axis('off')
    ax.set_title('[v24] Regime Detection')

    plt.tight_layout()
    plt.savefig('eda_dashboard_v24.png', dpi=90, bbox_inches='tight')
    plt.close()
    print("  ✓ eda_dashboard_v24.png tersimpan")
except Exception as e:
    print(f"  ⚠ EDA plot: {e}")
    plt.close('all')


# ========================== 4. FEATURE ENGINEERING ==========================
print("\n[4/10] FEATURE ENGINEERING...")

df_fe = df_paid.copy()
tgl_masuk_s = pd.to_datetime(df_fe[masuk_col], errors='coerce')

if COL_TGL_LAHIR and COL_TGL_LAHIR in df_fe.columns:
    df_fe['usia_saat_klaim'] = (
        (tgl_masuk_s - pd.to_datetime(df_fe[COL_TGL_LAHIR], errors='coerce')).dt.days / 365.25
    ).clip(0, 100).fillna(35)
else:
    df_fe['usia_saat_klaim'] = 35.0

if COL_TGL_EFF and COL_TGL_EFF in df_fe.columns:
    df_fe['tenure_polis_tahun'] = (
        (tgl_masuk_s - pd.to_datetime(df_fe[COL_TGL_EFF], errors='coerce')).dt.days / 365.25
    ).clip(0).fillna(1)
else:
    df_fe['tenure_polis_tahun'] = 1.0

if COL_TGL_KELUAR and COL_TGL_KELUAR in df_fe.columns:
    df_fe['lama_rawat'] = (
        (pd.to_datetime(df_fe[COL_TGL_KELUAR], errors='coerce') - tgl_masuk_s).dt.days
    ).clip(0, 365).fillna(0)
else:
    df_fe['lama_rawat'] = 1.0

df_fe['bulan_masuk']       = tgl_masuk_s.dt.month.fillna(1).astype(int)
df_fe['kuartal_masuk']     = tgl_masuk_s.dt.quarter.fillna(1).astype(int)
df_fe['hari_dalam_minggu'] = tgl_masuk_s.dt.dayofweek.fillna(0).astype(int)
df_fe['is_weekend']        = df_fe['hari_dalam_minggu'].isin([5, 6]).astype(int)
df_fe['tahun']             = tgl_masuk_s.dt.year.fillna(2024).astype(int)
df_fe['day_of_month']      = tgl_masuk_s.dt.day.fillna(15).astype(int)

df_fe['sin_month'] = np.sin(2 * np.pi * df_fe['bulan_masuk'] / 12)
df_fe['cos_month'] = np.cos(2 * np.pi * df_fe['bulan_masuk'] / 12)
df_fe['sin_qtr']   = np.sin(2 * np.pi * df_fe['kuartal_masuk'] / 4)
df_fe['cos_qtr']   = np.cos(2 * np.pi * df_fe['kuartal_masuk'] / 4)

ext_lookup = df_ext_train.set_index(['year', 'month'])
def get_ext_feat(yr, mo, feat):
    try: return float(ext_lookup.loc[(yr, mo), feat])
    except: return 1.0

for feat_ in ['inflation_idx', 'seasonal_prior', 'is_ramadan', 'is_eid_fitr',
              'is_yearend', 'is_yearstart', 'linear_trend']:
    df_fe[f'ext_{feat_}'] = [
        get_ext_feat(int(r['tahun']), int(r['bulan_masuk']), feat_)
        for _, r in df_fe[['tahun', 'bulan_masuk']].iterrows()
    ]

if COL_IP_OP and COL_IP_OP in df_fe.columns:
    df_fe['is_ip'] = df_fe[COL_IP_OP].astype(str).str.lower().str.contains('in').astype(int)
else:
    df_fe['is_ip'] = 0

if COL_GENDER and COL_GENDER in df_fe.columns:
    df_fe['is_female'] = df_fe[COL_GENDER].astype(str).str.upper().str.contains('F').astype(int)
else:
    df_fe['is_female'] = 0

if COL_REIMBURSE and COL_REIMBURSE in df_fe.columns:
    df_fe['is_reimburse'] = df_fe[COL_REIMBURSE].astype(str).str.upper().str.contains('R').astype(int)
else:
    df_fe['is_reimburse'] = 0

global_med = df_fe[TARGET_COL].median()
if pd.isna(global_med) or global_med == 0:
    global_med = 1.0

if COL_ICD and COL_ICD in df_fe.columns:
    df_fe['icd_chapter'] = df_fe[COL_ICD].astype(str).str[0].str.upper().fillna('Z')
    df_fe['icd_3char']   = df_fe[COL_ICD].astype(str).str[:3].str.upper().fillna('Z00')
    med_cost = df_fe.groupby('icd_3char')[TARGET_COL].median()
    freq_icd = df_fe.groupby('icd_3char').size()
    pctiles  = med_cost.quantile([.20, .40, .60, .80])
    def _tier(icd):
        c = med_cost.get(icd, global_med)
        if pd.isna(c): c = global_med
        if c <= pctiles[.20]:   return 0
        elif c <= pctiles[.40]: return 1
        elif c <= pctiles[.60]: return 2
        elif c <= pctiles[.80]: return 3
        return 4
    df_fe['icd_cost_tier']   = df_fe['icd_3char'].map(_tier).fillna(2).astype(int)
    df_fe['icd_freq_rank']   = df_fe['icd_3char'].map(freq_icd).fillna(1).astype(int)
    chapter_sev = {
        'A': 1, 'B': 1, 'C': 5, 'D': 4, 'E': 3, 'F': 2, 'G': 3, 'H': 1,
        'I': 4, 'J': 2, 'K': 2, 'L': 1, 'M': 2, 'N': 2, 'O': 3, 'P': 2,
        'Q': 3, 'R': 1, 'S': 2, 'T': 3, 'Z': 1
    }
    df_fe['icd_disease_sev'] = df_fe['icd_chapter'].map(chapter_sev).fillna(1).astype(int)
    chronic_ch = {'C', 'D', 'E', 'I', 'J', 'K', 'M', 'N'}
    df_fe['is_chronic']      = df_fe['icd_chapter'].isin(chronic_ch).astype(int)
    df_fe['is_catastrophic'] = (df_fe['icd_cost_tier'] == 4).astype(int)
    mean_cost_icd = df_fe.groupby('icd_3char')[TARGET_WINS_COL].mean()
    df_fe['icd_mean_cost']  = df_fe['icd_3char'].map(mean_cost_icd).fillna(global_med)
    df_fe['icd_cost_ratio'] = df_fe['icd_mean_cost'] / (global_med + 1e-10)
else:
    for col_ in ['icd_chapter', 'icd_3char', 'icd_cost_tier', 'icd_disease_sev',
                 'is_chronic', 'is_catastrophic', 'icd_freq_rank', 'icd_mean_cost', 'icd_cost_ratio']:
        df_fe[col_] = 2 if 'cost' in col_ or 'tier' in col_ else (1 if 'sev' in col_ else 0)

if COL_BIAYA_RS and COL_BIAYA_RS in df_fe.columns:
    df_fe['ratio_klaim_biaya'] = (
        df_fe[TARGET_COL] / df_fe[COL_BIAYA_RS].replace(0, np.nan)
    ).clip(0, 5).fillna(0.8)
else:
    df_fe['ratio_klaim_biaya'] = 0.8

if COL_PLAN and COL_PLAN in df_fe.columns:
    plan_sev   = df_fe.groupby(COL_PLAN)[TARGET_COL].median()
    plan_count = df_fe.groupby(COL_PLAN).size()
    df_fe['plan_sev_median'] = df_fe[COL_PLAN].map(plan_sev).fillna(global_med)
    df_fe['plan_count']      = df_fe[COL_PLAN].map(plan_count).fillna(0)
    plan_rank = {'M-001': 3, 'M-002': 2, 'M-003': 1}
    df_fe['plan_rank']       = df_fe[COL_PLAN].map(plan_rank).fillna(1)
else:
    df_fe['plan_sev_median'] = 0
    df_fe['plan_count']      = 0
    df_fe['plan_rank']       = 1

df_fe['usia_x_chronic']  = df_fe['usia_saat_klaim'] * df_fe['is_chronic']
df_fe['usia_x_tier']     = df_fe['usia_saat_klaim'] * df_fe['icd_cost_tier']
df_fe['rawat_x_tier']    = df_fe['lama_rawat']      * df_fe['icd_cost_tier']
df_fe['rawat_x_ip']      = df_fe['lama_rawat']      * df_fe['is_ip']
df_fe['plan_x_tier']     = df_fe['plan_rank']       * df_fe['icd_cost_tier']
df_fe['cost_x_ip']       = df_fe['icd_cost_ratio']  * df_fe['is_ip']
df_fe['cost_x_chronic']  = df_fe['icd_cost_ratio']  * df_fe['is_chronic']
df_fe['sev_adj_infl']    = df_fe[TARGET_WINS_COL] / df_fe['ext_inflation_idx']

print(f"  ✓ Features: {df_fe.shape[1]} kolom | Records: {len(df_fe):,}")


# ========================== 5. AGREGASI BULANAN ==============================
print("\n[5/10] AGREGASI BULANAN...")

monthly = df_fe.groupby(['tahun', 'bulan_masuk']).agg(
    frequency     = (TARGET_COL,           'count'),
    severity      = (TARGET_COL,           'mean'),
    severity_w    = (TARGET_WINS_COL,      'mean'),
    severity_med  = (TARGET_COL,           'median'),
    total_claim   = (TARGET_COL,           'sum'),
    usia_mean     = ('usia_saat_klaim',    'mean'),
    tenure_mean   = ('tenure_polis_tahun', 'mean'),
    rawat_mean    = ('lama_rawat',         'mean'),
    pct_chronic   = ('is_chronic',         'mean'),
    pct_catast    = ('is_catastrophic',    'mean'),
    avg_tier      = ('icd_cost_tier',      'mean'),
    avg_dis_sev   = ('icd_disease_sev',    'mean'),
    pct_ip        = ('is_ip',              'mean'),
    pct_female    = ('is_female',          'mean'),
    plan_sev_m    = ('plan_sev_median',    'mean'),
    plan_rank_m   = ('plan_rank',          'mean'),
    ratio_kb_m    = ('ratio_klaim_biaya',  'mean'),
    icd_cost_r    = ('icd_cost_ratio',     'mean'),
    cost_ip_m     = ('cost_x_ip',          'mean'),
    ext_inflation = ('ext_inflation_idx',  'mean'),
    ext_seasonal  = ('ext_seasonal_prior', 'mean'),
    ext_ramadan   = ('ext_is_ramadan',     'mean'),
    ext_eid       = ('ext_is_eid_fitr',    'mean'),
    ext_yearend   = ('ext_is_yearend',     'mean'),
    ext_trend     = ('ext_linear_trend',   'mean'),
).reset_index()

monthly['year_month'] = pd.to_datetime(
    monthly['tahun'].astype(str) + '-' +
    monthly['bulan_masuk'].astype(str).str.zfill(2)
)
monthly = monthly.sort_values('year_month').reset_index(drop=True)
monthly['fs_wins']       = monthly['frequency'] * monthly['severity_w']
monthly['fs_correction'] = monthly['total_claim'] / (monthly['fs_wins'] + 1e-10)

GLOBAL_FS_CORRECTION = float(monthly['fs_correction'].median())

# [v24-D] Per-month F×S correction factor
MONTHLY_FS_CORRECTION = {}
for mo in range(1, 13):
    mask = monthly['bulan_masuk'] == mo
    if mask.sum() >= 1:
        MONTHLY_FS_CORRECTION[mo] = float(monthly.loc[mask, 'fs_correction'].median())
    else:
        MONTHLY_FS_CORRECTION[mo] = GLOBAL_FS_CORRECTION

print(f"  Global F×S correction: {GLOBAL_FS_CORRECTION:.3f}")
print(f"  Per-month CF range: [{min(MONTHLY_FS_CORRECTION.values()):.3f}, {max(MONTHLY_FS_CORRECTION.values()):.3f}]")

# Lag features
for lag in [1, 2, 3]:
    monthly[f'mlag_freq_{lag}'] = monthly['frequency'].shift(lag)
    monthly[f'mlag_sevw_{lag}'] = monthly['severity_w'].shift(lag)
monthly['mlag_tot_1']  = monthly['total_claim'].shift(1)
monthly['mlag_tot_2']  = monthly['total_claim'].shift(2)
monthly['mlag_corr_1'] = monthly['fs_correction'].shift(1)

monthly['mroll_freq_3m']  = monthly['frequency'].rolling(3, min_periods=1).mean().shift(1)
monthly['mroll_freq_6m']  = monthly['frequency'].rolling(6, min_periods=1).mean().shift(1)
monthly['mroll_sevw_3m']  = monthly['severity_w'].rolling(3, min_periods=1).mean().shift(1)
monthly['mroll_tot_3m']   = monthly['total_claim'].rolling(3, min_periods=1).sum().shift(1)
monthly['mroll_tot_6m']   = monthly['total_claim'].rolling(6, min_periods=1).sum().shift(1)

monthly['yoy_freq']  = monthly['frequency'].shift(12)
monthly['yoy_sevw']  = monthly['severity_w'].shift(12)
monthly['yoy_total'] = monthly['total_claim'].shift(12)
monthly['yoy_growth_freq']  = (monthly['frequency']   / (monthly['yoy_freq'] + 1e-10) - 1).clip(-1, 5)
monthly['yoy_growth_sevw']  = (monthly['severity_w']  / (monthly['yoy_sevw'] + 1e-10) - 1).clip(-1, 5)
monthly['yoy_growth_total'] = (monthly['total_claim'] / (monthly['yoy_total']+ 1e-10) - 1).clip(-1, 5)

monthly['sin_month'] = np.sin(2 * np.pi * monthly['bulan_masuk'] / 12)
monthly['cos_month'] = np.cos(2 * np.pi * monthly['bulan_masuk'] / 12)
monthly['freq_mom']  = monthly['frequency'].pct_change().clip(-2, 2).shift(1).fillna(0)
monthly['sevw_mom']  = monthly['severity_w'].pct_change().clip(-2, 2).shift(1).fillna(0)

all_lag_cols = [c for c in monthly.columns if any(x in c for x in ['mlag', 'mroll', 'yoy', 'mom'])]
monthly[all_lag_cols] = monthly[all_lag_cols].bfill().ffill().fillna(0)

N = len(monthly)
print(f"  ✓ Monthly data: {N} bulan")
print(f"  frequency  : mean={monthly['frequency'].mean():.1f}, last={monthly['frequency'].iloc[-1]:.0f}")
print(f"  severity_w : mean={monthly['severity_w'].mean()/1e6:.2f}M")
print(f"  total_claim: mean={monthly['total_claim'].mean()/1e9:.3f}B IDR")

yoy_mask = monthly['yoy_freq'].notna() & (monthly['yoy_freq'] > 0)
if yoy_mask.sum() >= 3:
    ACTUAL_GROWTH_FREQ  = float(monthly.loc[yoy_mask, 'yoy_growth_freq'].median())
    ACTUAL_GROWTH_SEV   = float(monthly.loc[yoy_mask, 'yoy_growth_sevw'].median())
    ACTUAL_GROWTH_TOTAL = float(monthly.loc[yoy_mask, 'yoy_growth_total'].median())
else:
    ACTUAL_GROWTH_FREQ  = 0.027
    ACTUAL_GROWTH_SEV   = 0.05
    ACTUAL_GROWTH_TOTAL = 0.10

print(f"  YoY growth: Freq={ACTUAL_GROWTH_FREQ*100:+.1f}% Sev={ACTUAL_GROWTH_SEV*100:+.1f}% "
      f"Total={ACTUAL_GROWTH_TOTAL*100:+.1f}%")

HIST_FREQ_P5,  HIST_FREQ_P95  = monthly['frequency'].quantile(0.05), monthly['frequency'].quantile(0.95)
HIST_SEV_P5,   HIST_SEV_P95   = monthly['severity_w'].quantile(0.05), monthly['severity_w'].quantile(0.95)
HIST_TOT_P5,   HIST_TOT_P95   = monthly['total_claim'].quantile(0.05), monthly['total_claim'].quantile(0.95)

FREQ_RECENT_3M = float(monthly['frequency'].tail(3).mean())
FREQ_RECENT_6M = float(monthly['frequency'].tail(6).mean())
SEV_RECENT_3M  = float(monthly['severity_w'].tail(3).mean())
SEV_RECENT_6M  = float(monthly['severity_w'].tail(6).mean())
TOT_RECENT_3M  = float(monthly['total_claim'].tail(3).mean())
TOT_RECENT_6M  = float(monthly['total_claim'].tail(6).mean())

print(f"\n  [v24] Anchors:")
print(f"    Freq 3m={FREQ_RECENT_3M:.1f} 6m={FREQ_RECENT_6M:.1f}")
print(f"    Sev  3m={SEV_RECENT_3M/1e6:.2f}M 6m={SEV_RECENT_6M/1e6:.2f}M")


# ========================== 6. FEATURE IMPORTANCE ===========================
print("\n[6/10] FEATURE IMPORTANCE (v24)...")

FEATS_CANDIDATES = [
    'bulan_masuk', 'tahun', 'sin_month', 'cos_month',
    'usia_mean', 'tenure_mean', 'rawat_mean',
    'pct_chronic', 'pct_catast', 'avg_tier', 'avg_dis_sev', 'pct_ip',
    'pct_female', 'plan_sev_m', 'plan_rank_m', 'ratio_kb_m',
    'icd_cost_r', 'cost_ip_m',
    'mlag_freq_1', 'mlag_freq_2', 'mlag_freq_3',
    'mlag_sevw_1', 'mlag_sevw_2',
    'mlag_tot_1', 'mlag_tot_2', 'mlag_corr_1',
    'mroll_freq_3m', 'mroll_freq_6m',
    'mroll_sevw_3m', 'mroll_tot_3m', 'mroll_tot_6m',
    'freq_mom', 'sevw_mom',
    'yoy_freq', 'yoy_sevw', 'yoy_total',
    'ext_inflation', 'ext_seasonal', 'ext_ramadan', 'ext_eid',
    'ext_yearend', 'ext_trend',
]
FEATS = list(dict.fromkeys([f for f in FEATS_CANDIDATES if f in monthly.columns]))
monthly[FEATS] = monthly[FEATS].fillna(0)

train_data = monthly[
    monthly['frequency'].notna() & monthly['severity_w'].notna() & monthly['total_claim'].notna()
].copy()
train_data[FEATS] = train_data[FEATS].fillna(0)
print(f"  ✓ Features: {len(FEATS)} | Training: {len(train_data)} bulan")


# ========================== 7. PEMODELAN v24 ================================
print("\n[7/10] PEMODELAN v24.0...")
print("  [v24] 7 Models: Kalman + Damped HW + SARIMAX + STL + Prophet + LightGBM + ExtraTrees")
print("  [v24] Adaptive Weights + Per-Month Bias + Conformal PI + Regime Detection")

TARGET_CONFIG = {
    'frequency':   {'col': 'frequency',   'log': False, 'use_exog': True},
    'severity':    {'col': 'severity_w',  'log': True,  'use_exog': True},
    'total_claim': {'col': 'total_claim', 'log': True,  'use_exog': True},
}

EXT_COLS_MONTHLY = ['ext_inflation', 'ext_seasonal', 'ext_ramadan', 'ext_eid',
                    'ext_yearend', 'ext_trend']
EXT_COLS_AVAIL = [c for c in EXT_COLS_MONTHLY if c in monthly.columns]

PRED_MONTHS_2025 = [8, 9, 10, 11, 12]
df_ext_forecast = build_external_data(year_start=2025, month_start=8, n_months=5).head(5)

EXT_COL_MAP = {
    'ext_inflation': 'inflation_idx', 'ext_seasonal': 'seasonal_prior',
    'ext_ramadan': 'is_ramadan', 'ext_eid': 'is_eid_fitr',
    'ext_yearend': 'is_yearend', 'ext_trend': 'linear_trend',
}

def find_best_sarimax(series, exog=None, max_p=2, max_d=1, max_q=2):
    best_aic = np.inf
    best_order = (1, 1, 1)
    for p, d, q in product(range(max_p+1), range(max_d+1), range(max_q+1)):
        if p == 0 and q == 0: continue
        try:
            model = SARIMAX(
                series, exog=exog, order=(p, d, q),
                seasonal_order=(0,0,0,0),
                enforce_stationarity=False, enforce_invertibility=False
            )
            result = model.fit(disp=False, maxiter=100)
            if result.aic < best_aic:
                best_aic = result.aic
                best_order = (p, d, q)
        except: continue
    return best_order, best_aic


# ─── Walk-Forward Cross-Validation v24 ──────────────────────────────────────
print("\n  [Walk-Forward CV - v24]")
MIN_TRAIN_WF = max(12, N - 8)

wf_results = {}

for tgt_name, cfg in TARGET_CONFIG.items():
    col     = cfg['col']
    use_log = cfg['log']
    series  = monthly[col].values.copy()
    mapes_wf, preds_wf, actuals_wf = [], [], []

    for test_idx in range(MIN_TRAIN_WF, N):
        y_train   = series[:test_idx]
        y_true    = series[test_idx]
        ext_train = monthly[EXT_COLS_AVAIL].values[:test_idx] if EXT_COLS_AVAIL else None
        ext_test  = monthly[EXT_COLS_AVAIL].values[test_idx:test_idx+1] if EXT_COLS_AVAIL else None

        if len(y_train) < 6 or np.all(y_train == 0): continue
        y_fit = np.log1p(np.maximum(y_train, 0)) if use_log else y_train.copy()

        # [A] Kalman (phi=0.95 stable)
        try:
            kf = KalmanTrendFilter(phi=0.95)
            kf.fit(y_fit, optimize=False)
            kf_pred_log = float(kf.forecast(1)[0])
            kf_pred = max(0, np.expm1(kf_pred_log) if use_log else kf_pred_log)
        except:
            kf_pred = float(y_train.mean())

        # [B] Damped HW
        try:
            hw_model = ExponentialSmoothing(
                y_fit, trend='add', seasonal=None, damped_trend=True,
                initialization_method='estimated'
            )
            hw_fit = hw_model.fit(optimized=True, use_brute=False)
            hw_pred_log = float(hw_fit.forecast(1).iloc[0])
            hw_pred = max(0, np.expm1(hw_pred_log) if use_log else hw_pred_log)
        except:
            hw_pred = float(y_train.mean())

        # [C] SARIMAX
        try:
            if ext_train is not None and len(ext_train) >= 6:
                sm = SARIMAX(y_fit, exog=ext_train, order=(1,1,1),
                             enforce_stationarity=False, enforce_invertibility=False)
                sf = sm.fit(disp=False, maxiter=100)
                sarima_pred_log = float(sf.forecast(steps=1, exog=ext_test).iloc[-1])
            else:
                sm = SARIMAX(y_fit, order=(1,1,1),
                             enforce_stationarity=False, enforce_invertibility=False)
                sf = sm.fit(disp=False, maxiter=100)
                sarima_pred_log = float(sf.forecast(1).iloc[-1])
            sarima_pred = max(0, np.expm1(sarima_pred_log) if use_log else sarima_pred_log)
        except:
            sarima_pred = hw_pred

        # [v24-B] LightGBM (quick version for WF)
        lgb_pred = float(y_train.mean())
        if HAS_LGB and len(y_train) >= 8:
            try:
                lgb_m, _ = train_lgbm_model(y_train, ext_vals=ext_train, lookback=4, use_log=use_log)
                if lgb_m is not None:
                    lgb_pred = forecast_lgbm(lgb_m, y_train, ext_fc_row=ext_test[0] if ext_test is not None else None,
                                              h=1, lookback=4, use_log=use_log)
            except:
                pass

        # Ensemble
        preds_list = [kf_pred, hw_pred, sarima_pred, lgb_pred]
        ens_pred = float(np.mean([p for p in preds_list if p > 0]))
        ens_pred = max(0, ens_pred)

        mapes_wf.append(calc_mape([y_true], [ens_pred]))
        preds_wf.append(ens_pred)
        actuals_wf.append(y_true)

    wf_results[tgt_name] = {
        'mapes':     mapes_wf,
        'mean_mape': float(np.mean(mapes_wf)) if mapes_wf else 99.0,
        'preds':     preds_wf,
        'actuals':   actuals_wf,
        'bias':      float(np.mean([(a-p)/max(abs(a),1) for a,p in zip(actuals_wf,preds_wf)]))
                     if preds_wf else 0.0
    }
    print(f"  {tgt_name:<15}: WF-MAPE={wf_results[tgt_name]['mean_mape']:.2f}%"
          f"  Bias={wf_results[tgt_name]['bias']*100:+.1f}%  (n={len(mapes_wf)})")


# ─── Build Final Models v24 ──────────────────────────────────────────────────
print("\n  [Building Final Models - v24]")
final_models = {}
final_mapes  = {}

for tgt_name, cfg in TARGET_CONFIG.items():
    col     = cfg['col']
    use_log = cfg['log']
    series_raw = monthly[col].values.copy()
    n_series   = len(series_raw)

    print(f"\n  ─── {tgt_name} ───────────────────────────────────────────────────")

    y_fit   = np.log1p(np.maximum(series_raw, 0)) if use_log else series_raw.copy()
    ext_all = monthly[EXT_COLS_AVAIL].values if EXT_COLS_AVAIL else None

    # Regime detection per target
    regime_tgt, ratio_tgt = detect_regime(series_raw)
    print(f"    Regime: {regime_tgt} (ratio={ratio_tgt:.3f})")

    # [A] Kalman Trend Filter
    kf_model = None
    mape_kf  = 99.0
    try:
        print(f"    [A] Kalman Trend Filter...")
        kf_model = KalmanTrendFilter()
        kf_model.fit(y_fit, optimize=True)
        optimal_phi_kf  = kf_model.phi
        optimal_ql_kf   = kf_model.q_level
        optimal_qt_kf   = kf_model.q_trend

        n_test = min(6, n_series // 3)
        kf_oos_preds = []
        for i_oos in range(n_series - n_test, n_series):
            try:
                kf_tmp = KalmanTrendFilter(phi=optimal_phi_kf, q_level=optimal_ql_kf, q_trend=optimal_qt_kf)
                kf_tmp.fit(y_fit[:i_oos], optimize=False)
                kf_fc_log = float(kf_tmp.forecast(1)[0])
                kf_fc_raw = max(0, np.expm1(kf_fc_log) if use_log else kf_fc_log)
                kf_oos_preds.append(kf_fc_raw)
            except:
                kf_oos_preds.append(float(series_raw[:i_oos].mean()))
        mape_kf = calc_mape(series_raw[-n_test:], kf_oos_preds)
        print(f"    [A] Kalman phi={kf_model.phi:.3f} OOS-MAPE={mape_kf:.2f}%")
    except Exception as e:
        print(f"    [A] Kalman failed: {e}")

    # [B] Optimal Damped HW
    hw_model  = None
    mape_hw   = 99.0
    best_phi_hw = 0.95
    try:
        print(f"    [B] Optimal Damped HW...")
        best_phi_hw, best_mape_phi, phi_mapes_tgt = find_optimal_damping(
            series_raw, phi_grid=[0.80, 0.85, 0.90, 0.95, 0.97, 0.98, 1.00]
        )
        print(f"    [B] Optimal φ={best_phi_hw} (WF-MAPE={best_mape_phi:.2f}%)")
        if best_phi_hw < 1.0:
            hw_m = ExponentialSmoothing(y_fit, trend='add', seasonal=None, damped_trend=True,
                                        initialization_method='estimated')
            hw_model = hw_m.fit(damping_trend=best_phi_hw, optimized=True, use_brute=True)
        else:
            hw_m = ExponentialSmoothing(y_fit, trend='add', seasonal=None, damped_trend=False,
                                        initialization_method='estimated')
            hw_model = hw_m.fit(optimized=True, use_brute=True)
        mape_hw = best_mape_phi
        print(f"    [B] Damped HW OOS-MAPE={mape_hw:.2f}%")
    except Exception as e:
        print(f"    [B] HW failed: {e}")

    # [C] SARIMAX with External Regressors
    sarima_fitted = None
    mape_sarima   = 99.0
    best_sarima_order = (1, 1, 1)
    try:
        print(f"    [C] SARIMAX + external regressors...")
        if ext_all is not None and len(ext_all) >= 8:
            best_ord, best_aic = find_best_sarimax(y_fit, exog=ext_all)
            print(f"    [C] Best SARIMAX{best_ord} AIC={best_aic:.2f}")
            sarima_model  = SARIMAX(y_fit, exog=ext_all, order=best_ord,
                                    enforce_stationarity=False, enforce_invertibility=False)
            sarima_fitted = sarima_model.fit(disp=False, maxiter=100)
            best_sarima_order = best_ord
        else:
            sarima_model  = SARIMAX(y_fit, order=(1,1,1),
                                    enforce_stationarity=False, enforce_invertibility=False)
            sarima_fitted = sarima_model.fit(disp=False, maxiter=100)
        n_test = min(6, n_series // 3)
        s_insample = sarima_fitted.predict(start=n_series-n_test, end=n_series-1)
        s_insample_raw = np.expm1(np.array(s_insample)) if use_log else np.array(s_insample)
        mape_sarima = calc_mape(series_raw[-n_test:], np.maximum(s_insample_raw, 0))
        print(f"    [C] SARIMAX MAPE={mape_sarima:.2f}%")
    except Exception as e:
        print(f"    [C] SARIMAX failed: {e}")

    # [D] STL + Ridge (with external)
    stl_ml_components = None
    mape_stl  = 99.0
    try:
        print(f"    [D] STL + Ridge...")
        if n_series >= 12:
            ts_index  = pd.date_range(start='2024-01-01', periods=n_series, freq='MS')
            ts_series = pd.Series(y_fit, index=ts_index)
            stl_d = STL(ts_series, period=6, robust=True)
            stl_r = stl_d.fit()
            trend_vals = stl_r.trend.values
            t_arr = np.arange(n_series)
            trend_poly = np.polyfit(t_arr, trend_vals, deg=1)
            seasonal_pattern = np.zeros(6)
            for m_ in range(6):
                idxs = [i for i in range(n_series) if i % 6 == m_]
                if idxs:
                    seasonal_pattern[m_] = np.mean(stl_r.seasonal.values[idxs])
            resid = stl_r.resid.values
            if ext_all is not None:
                X_stl = np.column_stack([
                    np.arange(n_series),
                    np.sin(2*np.pi*np.arange(n_series)/6),
                    np.cos(2*np.pi*np.arange(n_series)/6),
                    ext_all[:n_series, :min(3, ext_all.shape[1])]
                ])
            else:
                X_stl = np.column_stack([
                    np.arange(n_series),
                    np.sin(2*np.pi*np.arange(n_series)/6),
                    np.cos(2*np.pi*np.arange(n_series)/6),
                ])
            scaler_stl = RobustScaler()
            X_stl_sc   = scaler_stl.fit_transform(X_stl)
            ridge_resid = Ridge(alpha=5)  # [v24] slightly lower alpha for more flexibility
            ridge_resid.fit(X_stl_sc[:-1], resid[1:])
            stl_ml_components = {
                'trend_poly': trend_poly, 'seasonal_pattern': seasonal_pattern,
                'ridge': ridge_resid, 'scaler': scaler_stl,
                'n_train': n_series, 'use_log': use_log,
                'n_ext': X_stl.shape[1] - 3
            }
            n_test = min(6, n_series // 3)
            stl_preds_test = []
            for i in range(n_series - n_test, n_series):
                t_next = i
                trend_next    = np.polyval(trend_poly, t_next)
                seasonal_next = seasonal_pattern[t_next % 6]
                if ext_all is not None:
                    X_next = np.array([[t_next,
                                        np.sin(2*np.pi*t_next/6), np.cos(2*np.pi*t_next/6)] +
                                        ext_all[i, :min(3, ext_all.shape[1])].tolist()])
                else:
                    X_next = np.array([[t_next,
                                        np.sin(2*np.pi*t_next/6), np.cos(2*np.pi*t_next/6)]])
                X_next_sc  = scaler_stl.transform(X_next)
                resid_next = float(ridge_resid.predict(X_next_sc)[0])
                pv = trend_next + seasonal_next + resid_next
                stl_preds_test.append(max(0, np.expm1(pv) if use_log else pv))
            mape_stl = calc_mape(series_raw[-n_test:], stl_preds_test)
            print(f"    [D] STL+Ridge MAPE={mape_stl:.2f}%")
    except Exception as e:
        print(f"    [D] STL failed: {e}")

    # [E] Prophet
    prophet_fitted = None
    mape_prophet   = 99.0
    try:
        if HAS_PROPHET:
            print(f"    [E] Prophet...")
            series_for_prophet = np.expm1(y_fit) if use_log else series_raw
            dates = pd.date_range(end=pd.Timestamp('2025-07-01'), periods=n_series, freq='MS')
            df_p  = pd.DataFrame({'ds': dates, 'y': np.maximum(series_for_prophet, 0.01)})
            if ext_all is not None:
                for j, col_name in enumerate(EXT_COLS_AVAIL[:3]):
                    df_p[col_name] = ext_all[:n_series, j]
            model_p = Prophet(
                yearly_seasonality=False, weekly_seasonality=False, daily_seasonality=False,
                changepoint_prior_scale=0.30, seasonality_prior_scale=0.01,
                changepoint_range=0.85, interval_width=0.8
            )
            if ext_all is not None:
                for col_name in EXT_COLS_AVAIL[:3]:
                    model_p.add_regressor(col_name)
            model_p.fit(df_p, iter=500)
            future = model_p.make_future_dataframe(periods=0, freq='MS')
            if ext_all is not None:
                for j, col_name in enumerate(EXT_COLS_AVAIL[:3]):
                    future[col_name] = ext_all[-len(future):, j]
            forecast_p = model_p.predict(future)
            prophet_fitted = model_p
            n_test = min(6, n_series // 3)
            p_preds_raw = forecast_p['yhat'].values[-n_test:]
            mape_prophet = calc_mape(series_raw[-n_test:], np.maximum(p_preds_raw, 0))
            print(f"    [E] Prophet MAPE={mape_prophet:.2f}%")
    except Exception as e:
        print(f"    [E] Prophet error: {e}")

    # [F] LightGBM (NEW in v24)
    lgbm_model = None
    mape_lgbm  = 99.0
    if HAS_LGB:
        try:
            print(f"    [F] LightGBM...")
            lgbm_model, mape_lgbm = train_lgbm_model(
                series_raw, ext_vals=ext_all, lookback=6, use_log=use_log
            )
            print(f"    [F] LightGBM OOS-MAPE={mape_lgbm:.2f}%")
        except Exception as e:
            print(f"    [F] LightGBM failed: {e}")

    # [G] Extra Trees (NEW in v24, no LightGBM needed)
    extra_trees_model = None
    mape_et = 99.0
    try:
        print(f"    [G] Extra Trees...")
        if n_series >= 10:
            y_et = np.log1p(np.maximum(series_raw, 0)) if use_log else series_raw.copy()
            X_et_df, y_et_arr = build_ts_features(y_et, ext_vals=ext_all, lookback=4)
            if len(X_et_df) >= 5:
                n_test_et = min(4, len(X_et_df) // 3)
                et_oos_preds, et_oos_actuals = [], []
                for i_et in range(len(X_et_df) - n_test_et, len(X_et_df)):
                    try:
                        et_tmp = ExtraTreesRegressor(n_estimators=100, max_depth=4,
                                                      min_samples_leaf=2, random_state=42, n_jobs=-1)
                        et_tmp.fit(X_et_df.iloc[:i_et], y_et_arr[:i_et])
                        pred_log = float(et_tmp.predict(X_et_df.iloc[i_et:i_et+1])[0])
                        pred_raw = max(0, np.expm1(pred_log) if use_log else pred_log)
                        et_oos_preds.append(pred_raw)
                        et_oos_actuals.append(np.expm1(y_et_arr[i_et]) if use_log else y_et_arr[i_et])
                    except:
                        continue
                mape_et = calc_mape(et_oos_actuals, et_oos_preds) if et_oos_preds else 99.0

                extra_trees_model = ExtraTreesRegressor(n_estimators=200, max_depth=4,
                                                         min_samples_leaf=2, random_state=42, n_jobs=-1)
                extra_trees_model.fit(X_et_df, y_et_arr)
                print(f"    [G] Extra Trees OOS-MAPE={mape_et:.2f}%")
    except Exception as e:
        print(f"    [G] Extra Trees failed: {e}")

    # [v24] Adaptive Ensemble Weights (multi-method)
    mapes_all = {'Kalman': mape_kf, 'HW': mape_hw, 'SARIMAX': mape_sarima,
                 'STL': mape_stl, 'Prophet': mape_prophet,
                 'LightGBM': mape_lgbm, 'ExtraTrees': mape_et}

    # Try 4 weight methods and pick best via holdout MAPE
    # In v24 we just use inv_mape2 + regime adjustment
    weights_e = compute_adaptive_weights_v24(
        mapes_all, wf_results, regime=regime_tgt, method='inv_mape2'
    )

    bias_cf = 1.0 + wf_results[tgt_name]['bias']

    print(f"    MAPEs: {' '.join([f'{k}={v:.1f}%' for k,v in mapes_all.items()])}")
    print(f"    Weights: {' '.join([f'{k}={v:.3f}' for k,v in weights_e.items()])}")
    print(f"    Bias correction: ×{bias_cf:.4f}")

    # [v24-C] Conformal Prediction calibration
    cp = None
    try:
        n_cal = min(6, n_series // 3)
        y_fit_full = y_fit
        cal_actuals, cal_preds = [], []
        for i_cal in range(n_series - n_cal, n_series):
            a = series_raw[i_cal]
            # Quick ensemble pred for calibration
            preds_cal = []
            if kf_model is not None:
                try:
                    kf_c = KalmanTrendFilter(phi=kf_model.phi, q_level=kf_model.q_level, q_trend=kf_model.q_trend)
                    kf_c.fit(y_fit[:i_cal], optimize=False)
                    kf_fc = max(0, np.expm1(float(kf_c.forecast(1)[0])) if use_log else float(kf_c.forecast(1)[0]))
                    preds_cal.append(kf_fc)
                except: pass
            if hw_model is not None:
                try:
                    hw_fc = max(0, np.expm1(float(hw_model.fittedvalues[i_cal])) if use_log else float(hw_model.fittedvalues[i_cal]))
                    preds_cal.append(hw_fc)
                except: pass
            if preds_cal:
                cal_preds.append(float(np.mean(preds_cal)))
                cal_actuals.append(a)
        if len(cal_actuals) >= 3:
            cp = ConformalPrediction(alpha=0.10)
            cp.calibrate(cal_actuals, cal_preds)
    except:
        cp = None

    # [v24-G] Per-month bias correction
    try:
        hw_insample = np.array(hw_model.fittedvalues) if hw_model is not None else np.zeros(n_series)
        hw_insample_raw = np.expm1(hw_insample) if use_log else hw_insample
        hw_insample_raw = np.maximum(hw_insample_raw, 0)
        month_bias_dict = compute_per_month_bias(
            monthly.reset_index(), col, hw_insample_raw,
            months_col='bulan_masuk'
        )
    except:
        month_bias_dict = {m: 1.0 for m in range(1, 13)}

    final_models[tgt_name] = {
        'kalman':      kf_model,
        'hw':          hw_model,
        'hw_phi':      best_phi_hw,
        'sarima':      sarima_fitted,
        'sarima_order': best_sarima_order,
        'stl':         stl_ml_components,
        'prophet':     prophet_fitted,
        'lgbm':        lgbm_model,
        'extra_trees': extra_trees_model,
        'weights':     weights_e,
        'use_log':     use_log,
        'col':         col,
        'n_train':     n_series,
        'series_raw':  series_raw,
        'series_log':  y_fit,
        'bias_cf':     bias_cf,
        'ext_all':     ext_all,
        'regime':      regime_tgt,
        'conformal':   cp,
        'month_bias':  month_bias_dict,
    }
    valid_models = {k: v for k, v in mapes_all.items() if v < 60.0}
    final_mapes[tgt_name] = min(valid_models.values()) if valid_models else 99.0

print(f"\n  ─────────────────────────────────────────────────────────────────────────")
for tgt in ['frequency', 'severity', 'total_claim']:
    print(f"  {tgt:<15}: Best Single Model MAPE={final_mapes[tgt]:.2f}%")
print(f"  AVG MAPE: {np.mean(list(final_mapes.values())):.2f}%")


# ========================== 8. PREDIKSI AGT–DES 2025 =======================
print("\n[8/10] PREDIKSI AGT–DES 2025 (v24)...")

ext_fc_dict = {}
for i, m_ in enumerate(PRED_MONTHS_2025):
    if EXT_COLS_AVAIL and len(df_ext_forecast) > i:
        row_vals = []
        for col_monthly in EXT_COLS_AVAIL:
            raw_col = EXT_COL_MAP.get(col_monthly, None)
            if raw_col and raw_col in df_ext_forecast.columns:
                row_vals.append(float(df_ext_forecast.iloc[i][raw_col]))
            else:
                row_vals.append(1.0)
        ext_fc_dict[m_] = np.array(row_vals, dtype=float)
    else:
        ext_fc_dict[m_] = None


def forecast_target_v24(tgt_name, pred_year, pred_months):
    """
    [v24] Complete forecasting pipeline with all improvements:
    1. 7-model ensemble (Kalman + HW + SARIMAX + STL + Prophet + LightGBM + ExtraTrees)
    2. Adaptive regime-aware weights
    3. Per-month bias correction
    4. Conformal prediction intervals
    5. Smarter YoY blending with regime detection
    6. ANCHOR and FLOOR protection
    7. External seasonal adjustment
    """
    mdl        = final_models[tgt_name]
    use_log    = mdl['use_log']
    series     = mdl['series_raw']
    n_s        = mdl['n_train']
    w          = mdl['weights']
    bias_cf    = mdl['bias_cf']
    ext_all    = mdl['ext_all']
    regime_tgt = mdl['regime']
    cp         = mdl['conformal']
    month_bias = mdl['month_bias']

    last_train_ym = monthly['year_month'].iloc[-1]
    last_year  = int(str(last_train_ym)[:4])
    last_month = int(str(last_train_ym)[5:7])

    results = {}
    for pred_month in pred_months:
        steps = (pred_year - last_year)*12 + (pred_month - last_month)
        h = max(1, steps)

        ext_fc = ext_fc_dict.get(pred_month, None)

        # [A] Kalman Trend
        fc_kf = float(series.mean())
        try:
            kf = mdl['kalman']
            if kf is not None:
                fc_kf_log_arr = kf.forecast(h)
                fc_kf_log     = float(fc_kf_log_arr[-1])
                fc_kf = max(0, np.expm1(fc_kf_log) if use_log else fc_kf_log)
        except: pass

        # [B] Damped HW
        fc_hw = float(series.mean())
        try:
            if mdl['hw'] is not None:
                fc_hw_arr = mdl['hw'].forecast(h)
                fc_hw_log = float(fc_hw_arr.iloc[-1])
                fc_hw = max(0, np.expm1(fc_hw_log) if use_log else fc_hw_log)
        except: pass

        # [C] SARIMAX
        fc_sarima = fc_hw
        try:
            if mdl['sarima'] is not None:
                if ext_fc is not None and ext_all is not None:
                    exog_fc = np.zeros((h, ext_all.shape[1]))
                    for step_i in range(h):
                        exog_fc[step_i] = ext_fc
                    fc_s_arr = mdl['sarima'].get_forecast(steps=h, exog=exog_fc)
                    fc_s_log = float(fc_s_arr.predicted_mean.iloc[-1])
                else:
                    fc_s_arr = mdl['sarima'].get_forecast(steps=h)
                    fc_s_log = float(fc_s_arr.predicted_mean.iloc[-1])
                fc_sarima = max(0, np.expm1(fc_s_log) if use_log else fc_s_log)
        except: pass

        # [D] STL
        fc_stl = float(series.mean())
        try:
            if mdl['stl'] is not None:
                stl_c = mdl['stl']
                t_next = n_s + h - 1
                trend_next    = np.polyval(stl_c['trend_poly'], t_next)
                seasonal_next = stl_c['seasonal_pattern'][t_next % 6]
                n_ext_stl = stl_c.get('n_ext', 0)
                if ext_fc is not None and n_ext_stl > 0:
                    X_next = np.array([[t_next,
                                        np.sin(2*np.pi*t_next/6), np.cos(2*np.pi*t_next/6)] +
                                        ext_fc[:n_ext_stl].tolist()])
                else:
                    X_next = np.array([[t_next,
                                        np.sin(2*np.pi*t_next/6), np.cos(2*np.pi*t_next/6)]])
                X_next_sc  = stl_c['scaler'].transform(X_next)
                resid_next = float(stl_c['ridge'].predict(X_next_sc)[0])
                pv_stl = trend_next + seasonal_next + resid_next
                fc_stl = max(0, np.expm1(pv_stl) if use_log else pv_stl)
        except: pass

        # [E] Prophet
        fc_prophet = None
        try:
            if mdl['prophet'] is not None and w.get('Prophet', 0) > 0.01:
                model_p = mdl['prophet']
                pred_date = pd.Timestamp(f'{pred_year}-{pred_month:02d}-01')
                future_p  = model_p.make_future_dataframe(periods=h, freq='MS')
                if ext_fc is not None and EXT_COLS_AVAIL:
                    last_ext = ext_all[-1] if ext_all is not None else None
                    for j, col_name in enumerate(EXT_COLS_AVAIL[:3]):
                        if ext_fc is not None:
                            future_p.loc[future_p['ds'] >= pred_date, col_name] = float(ext_fc[j])
                            if last_ext is not None:
                                future_p.loc[future_p['ds'] < pred_date, col_name] = float(last_ext[j])
                fc_p_df   = model_p.predict(future_p)
                fc_prophet = max(0, float(fc_p_df['yhat'].iloc[-1]))
        except: pass

        # [F] LightGBM
        fc_lgbm = None
        try:
            if mdl['lgbm'] is not None and w.get('LightGBM', 0) > 0.01:
                fc_lgbm = forecast_lgbm(
                    mdl['lgbm'], series, ext_fc_row=ext_fc,
                    h=h, lookback=6, use_log=use_log
                )
                if isinstance(fc_lgbm, list):
                    fc_lgbm = fc_lgbm[-1] if fc_lgbm else float(series.mean())
                fc_lgbm = max(0, float(fc_lgbm))
        except: pass

        # [G] Extra Trees
        fc_et = None
        try:
            if mdl['extra_trees'] is not None and w.get('ExtraTrees', 0) > 0.01:
                # Recursive prediction using build_ts_features approach
                y_et = np.log1p(np.maximum(series, 0)) if use_log else series.copy()
                et_result = forecast_lgbm.__wrapped__ if hasattr(forecast_lgbm, '__wrapped__') else None
                # Use simple approach: use last 4 lags as features
                lookback_et = 4
                row_et = {}
                for lag in range(1, lookback_et + 1):
                    row_et[f'lag_{lag}'] = y_et[-lag] if len(y_et) >= lag else 0
                window3 = y_et[-3:] if len(y_et) >= 3 else y_et
                window6 = y_et[-6:] if len(y_et) >= 6 else y_et
                row_et['roll_mean_3'] = float(np.mean(window3))
                row_et['roll_std_3']  = float(np.std(window3)) if len(window3) > 1 else 0
                row_et['roll_mean_6'] = float(np.mean(window6))
                row_et['roll_std_6']  = float(np.std(window6)) if len(window6) > 1 else 0
                row_et['roll_max_3']  = float(np.max(window3))
                row_et['roll_min_3']  = float(np.min(window3))
                t_et = n_s + h - 1
                row_et['t_idx']  = t_et
                row_et['sin_t']  = np.sin(2 * np.pi * t_et / 12)
                row_et['cos_t']  = np.cos(2 * np.pi * t_et / 12)
                row_et['sin_t6'] = np.sin(2 * np.pi * t_et / 6)
                row_et['cos_t6'] = np.cos(2 * np.pi * t_et / 6)
                row_et['mom_1']   = float(y_et[-1] - y_et[-2]) if len(y_et) >= 2 else 0
                row_et['mom_pct'] = float(y_et[-1] / (y_et[-2] + 1e-10) - 1) if len(y_et) >= 2 else 0
                if ext_fc is not None:
                    for j_et, v_et in enumerate(ext_fc):
                        row_et[f'ext_{j_et}'] = v_et

                X_et_pred = pd.DataFrame([row_et])
                # align with training features
                feat_names = mdl['extra_trees'].feature_names_in_ if hasattr(mdl['extra_trees'], 'feature_names_in_') else list(row_et.keys())
                for fn in feat_names:
                    if fn not in X_et_pred.columns:
                        X_et_pred[fn] = 0
                X_et_pred = X_et_pred[feat_names]

                pred_et_log = float(mdl['extra_trees'].predict(X_et_pred)[0])
                fc_et = max(0, np.expm1(pred_et_log) if use_log else pred_et_log)
        except: pass

        # ── [v24] GEOMETRIC MEAN ENSEMBLE ────────────────────────────────────
        pred_list_   = []
        weight_list_ = []
        for mn, fc_val in [('Kalman', fc_kf), ('HW', fc_hw), ('SARIMAX', fc_sarima),
                            ('STL', fc_stl), ('Prophet', fc_prophet),
                            ('LightGBM', fc_lgbm), ('ExtraTrees', fc_et)]:
            wt = w.get(mn, 0)
            if wt > 0 and fc_val is not None and fc_val > 0:
                pred_list_.append(fc_val)
                weight_list_.append(wt)

        if pred_list_:
            log_preds_ = np.log(np.maximum(pred_list_, 1e-10))
            wts_       = np.array(weight_list_) / sum(weight_list_)
            fc_ens     = float(np.exp(np.dot(wts_, log_preds_)))
        else:
            fc_ens = float(series.mean())
        fc_ens = max(0, fc_ens)

        # ── [v24] BIAS CORRECTION (global + per-month) ─────────────────────
        per_month_cf = month_bias.get(pred_month, 1.0)
        # Blend global bias_cf and per-month cf
        # Only apply per-month if it's meaningfully different from 1.0
        if abs(per_month_cf - 1.0) > 0.02:
            combined_cf = 0.5 * bias_cf + 0.5 * (bias_cf * per_month_cf)
        else:
            combined_cf = bias_cf
        combined_cf = np.clip(combined_cf, 0.85, 1.20)  # safety clip
        fc_ens = fc_ens * combined_cf

        # ── [v24-J] ADAPTIVE YoY BLENDING (regime-aware) ─────────────────
        yoy_mask_p = (monthly['bulan_masuk'] == pred_month) & (monthly['tahun'] == pred_year - 1)
        if yoy_mask_p.any():
            yoy_actual = float(monthly.loc[yoy_mask_p, mdl['col']].values[0])
            if tgt_name == 'frequency':
                growth = ACTUAL_GROWTH_FREQ
            elif tgt_name == 'severity':
                growth = ACTUAL_GROWTH_SEV
            else:
                growth = ACTUAL_GROWTH_TOTAL
            fc_yoy = yoy_actual * (1 + growth)

            model_mape = wf_results[tgt_name]['mean_mape']
            # [v24] Regime-aware YoY weight
            if regime_tgt in ['transition_up', 'transition_down']:
                yoy_weight = 0.05   # ignore YoY in transition (non-stationary)
            elif model_mape < 3.0:
                yoy_weight = 0.08   # model very accurate
            elif model_mape < 5.0:
                yoy_weight = 0.15
            elif model_mape < 8.0:
                yoy_weight = 0.25
            else:
                yoy_weight = 0.35
            fc_ens = (1 - yoy_weight) * fc_ens + yoy_weight * fc_yoy
            fc_ens = max(0, fc_ens)

        # ── [v24] EXTERNAL SEASONAL ADJUSTMENT ────────────────────────────
        ext_seas_idx = 1.0
        if ext_fc is not None and 'ext_seasonal' in EXT_COLS_AVAIL:
            j_seas = EXT_COLS_AVAIL.index('ext_seasonal')
            ext_seas_idx = float(ext_fc[j_seas])
        seasonal_weight = 0.25  # [v24] reduced from 0.30 to avoid over-adjustment
        fc_ens = fc_ens * (1 - seasonal_weight + seasonal_weight * ext_seas_idx)

        # ── [v24] ANCHOR AND FLOOR PROTECTION ─────────────────────────────
        if tgt_name == 'frequency':
            freq_floor  = FREQ_RECENT_3M * 0.80
            freq_last   = float(series[-1])
            freq_anchor = freq_last * 0.75
            fc_floor    = max(freq_floor, freq_anchor)
            if fc_ens < fc_floor:
                print(f"    [ANCHOR] Freq {pred_month}: {fc_ens:.0f} → hard floor {fc_floor:.0f}")
                fc_ens = fc_floor
            freq_ceiling = HIST_FREQ_P95 * 1.25
            fc_ens = min(fc_ens, freq_ceiling)

        elif tgt_name == 'severity':
            sev_floor = SEV_RECENT_6M * 0.88
            if fc_ens < sev_floor:
                print(f"    [FLOOR] Sev {pred_month}: {fc_ens/1e6:.2f}M → floor {sev_floor/1e6:.2f}M")
                fc_ens = sev_floor

        else:
            tot_floor = HIST_TOT_P5 * 0.75
            if fc_ens < tot_floor:
                fc_ens = tot_floor

        # ── FINAL CLIPPING ─────────────────────────────────────────────────
        if tgt_name == 'frequency':
            fc_ens = np.clip(fc_ens, HIST_FREQ_P5 * 0.60, HIST_FREQ_P95 * 1.50)
        elif tgt_name == 'severity':
            fc_ens = np.clip(fc_ens, HIST_SEV_P5 * 0.70, HIST_SEV_P95 * 1.50)
        else:
            fc_ens = np.clip(fc_ens, HIST_TOT_P5 * 0.60, HIST_TOT_P95 * 1.60)

        results[pred_month] = fc_ens

    return results


# Generate predictions
print("\n  === Prediksi 2025 (Agt–Des) ===")
all_preds_2025 = {
    'frequency':   forecast_target_v24('frequency',   2025, PRED_MONTHS_2025),
    'severity':    forecast_target_v24('severity',    2025, PRED_MONTHS_2025),
    'total_claim': forecast_target_v24('total_claim', 2025, PRED_MONTHS_2025),
}

print(f"\n  Initial predictions (before adaptive F×S enforcement):")
print(f"  {'Bulan':<10} {'Freq':>8} {'Sev (M)':>10} {'Total (B)':>12} {'F×S (B)':>10} {'Ratio':>7}")
print(f"  {'-'*60}")
for m in PRED_MONTHS_2025:
    f_ = all_preds_2025['frequency'][m]
    s_ = all_preds_2025['severity'][m]
    t_ = all_preds_2025['total_claim'][m]
    fs = f_ * s_ * GLOBAL_FS_CORRECTION
    ratio = t_ / (fs + 1e-10)
    print(f"  2025-{m:02d}   {f_:>8,.0f} {s_/1e6:>9.2f}M {t_/1e9:>11.3f}B "
          f"{fs/1e9:>9.3f}B {ratio:>6.2f}")

# ─── [v24-D] Adaptive F×S Enforcement ────────────────────────────────────────
print("\n  [v24] Adaptive F×S Enforcement (per-month correction)...")
wf_mape_freq  = wf_results['frequency']['mean_mape']
wf_mape_sev   = wf_results['severity']['mean_mape']
wf_mape_total = wf_results['total_claim']['mean_mape']

for m in PRED_MONTHS_2025:
    f_ = all_preds_2025['frequency'][m]
    s_ = all_preds_2025['severity'][m]
    t_ = all_preds_2025['total_claim'][m]

    t_new, ratio, fs_weight = adaptive_fs_enforcement(
        f_, s_, t_, GLOBAL_FS_CORRECTION, MONTHLY_FS_CORRECTION,
        m, wf_mape_freq, wf_mape_sev, wf_mape_total
    )

    print(f"  2025-{m:02d}: fs_weight={fs_weight:.2f} ratio={ratio:.2f} | "
          f"model={t_/1e9:.3f}B → final={t_new/1e9:.3f}B")
    all_preds_2025['total_claim'][m] = max(0, t_new)

# ─── Final Summary ─────────────────────────────────────────────────────────────
print(f"\n  Final predictions v24:")
print(f"  {'Bulan':<10} {'Freq':>8} {'Sev (M)':>10} {'Total (B)':>12} {'F×S (B)':>10} {'Check':>5}")
print(f"  {'-'*60}")
for m in PRED_MONTHS_2025:
    f_ = all_preds_2025['frequency'][m]
    s_ = all_preds_2025['severity'][m]
    t_ = all_preds_2025['total_claim'][m]
    fs = f_ * s_ * GLOBAL_FS_CORRECTION
    ok = '✓' if 0.6 < t_/fs < 1.6 else '⚠'
    print(f"  2025-{m:02d}   {f_:>8,.0f} {s_/1e6:>9.2f}M {t_/1e9:>11.3f}B "
          f"{fs/1e9:>9.3f}B {ok:>5}")


# ─── Build Submission ──────────────────────────────────────────────────────────
rows_ = []
for m in PRED_MONTHS_2025:
    ms = f'2025_{m:02d}'
    rows_.append({'id': f'{ms}_Claim_Frequency', 'value': all_preds_2025['frequency'][m]})
    rows_.append({'id': f'{ms}_Claim_Severity',  'value': all_preds_2025['severity'][m]})
    rows_.append({'id': f'{ms}_Total_Claim',     'value': all_preds_2025['total_claim'][m]})

submission = pd.DataFrame(rows_)
submission['value'] = pd.to_numeric(submission['value'], errors='coerce').fillna(0).clip(0)

sample_ids = df_sample['id'].tolist()
sub_dict   = dict(zip(submission['id'], submission['value']))

id_match = set(submission['id'].tolist()) == set(sample_ids)
if id_match:
    submission = submission.set_index('id').loc[sample_ids].reset_index()
    print("\n  ✓ ID cocok sempurna")
else:
    missing = [s for s in sample_ids if s not in sub_dict]
    if missing: print(f"  ⚠ Missing IDs: {missing}")
    submission = pd.DataFrame({
        'id':    sample_ids,
        'value': [sub_dict.get(s, 0.0) for s in sample_ids]
    })

submission['value'] = submission['value'].round(4)
assert submission['value'].sum() > 0, "CRITICAL: All values = 0!"
submission.to_csv('submission.csv', index=False)

print(f"\n  Full submission:")
print(submission.to_string(index=False))


# ========================== 9. VALIDATION ====================================
print("\n[9/10] VALIDASI SUBMISSION...")
print(f"  Rows  : {len(submission)} (expected {len(sample_ids)})")
print(f"  Sum   : {submission['value'].sum():,.2f}")
print(f"\n  {'ID':<40} {'Value':>15}")
print(f"  {'-'*57}")
for _, row in submission.iterrows():
    print(f"  {row['id']:<40} {row['value']:>15,.2f}")


# ========================== 10. SIMULASI SKOR v24 ===========================
def hitung_simulasi_v24():
    """
    [v24] 5-month holdout simulation.
    Full 7-model ensemble with all v24 improvements.
    """
    print("\n" + "═" * 65)
    print("  🔍 SIMULASI SKOR KAGGLE v24 (5-MONTH HOLDOUT)")
    print("  [v24] LightGBM + ExtraTrees + Adaptive Weights + Per-Month Bias")
    print("═" * 65)

    n_holdout = 5
    if len(monthly) <= n_holdout * 2:
        print("  ⚠ Data terlalu sedikit untuk simulasi holdout.")
        return

    train_data_sim = monthly.iloc[:-n_holdout].copy()
    test_data_sim  = monthly.iloc[-n_holdout:].copy()

    actuals = {
        'frequency':   test_data_sim['frequency'].values,
        'severity':    test_data_sim['severity_w'].values,
        'total_claim': test_data_sim['total_claim'].values,
    }

    sim_preds = {'frequency': [], 'severity': [], 'total_claim': []}

    for key in ['frequency', 'severity', 'total_claim']:
        mdl      = final_models[key]
        use_log  = mdl['use_log']
        col_name = mdl['col']
        w        = mdl['weights']
        bias_cf  = mdl['bias_cf']

        series_train_raw = train_data_sim[col_name].values
        series_train_log = np.log1p(np.maximum(series_train_raw, 0)) if use_log else series_train_raw.copy()

        fc_kf = np.zeros(n_holdout)
        fc_hw = np.zeros(n_holdout)
        fc_sa = np.zeros(n_holdout)
        fc_lgb = np.full(n_holdout, series_train_raw.mean())

        # Kalman
        try:
            kf_sim = KalmanTrendFilter(
                phi=mdl['kalman'].phi,
                q_level=mdl['kalman'].q_level,
                q_trend=mdl['kalman'].q_trend
            )
            kf_sim.fit(series_train_log, optimize=False)
            fc_kf_log = kf_sim.forecast(n_holdout)
            fc_kf = np.maximum(0, np.expm1(fc_kf_log) if use_log else fc_kf_log)
        except:
            fc_kf = np.full(n_holdout, series_train_raw.mean())

        # HW
        try:
            phi_hw = mdl['hw_phi']
            hw_sim = ExponentialSmoothing(
                series_train_log, trend='add', seasonal=None,
                damped_trend=(phi_hw < 1.0), initialization_method='estimated'
            )
            hw_fit_kw = {'optimized': True, 'use_brute': False}
            if phi_hw < 1.0:
                hw_fit_kw['damping_trend'] = phi_hw
            hw_fit = hw_sim.fit(**hw_fit_kw)
            fc_hw_log = hw_fit.forecast(n_holdout).values
            fc_hw = np.maximum(0, np.expm1(fc_hw_log) if use_log else fc_hw_log)
        except:
            fc_hw = np.full(n_holdout, series_train_raw.mean())

        # SARIMAX
        try:
            if mdl['sarima'] is not None:
                ord_ = mdl['sarima_order']
                sa_m = SARIMAX(series_train_log, order=ord_,
                               enforce_stationarity=False, enforce_invertibility=False)
                sa_fit = sa_m.fit(disp=False, maxiter=50)
                fc_sa_log = sa_fit.forecast(n_holdout).values
                fc_sa = np.maximum(0, np.expm1(fc_sa_log) if use_log else fc_sa_log)
            else:
                fc_sa = fc_hw
        except:
            fc_sa = fc_hw

        # LightGBM
        if HAS_LGB and mdl['lgbm'] is not None:
            try:
                ext_train_sim = train_data_sim[EXT_COLS_AVAIL].values if EXT_COLS_AVAIL else None
                lgb_sim, _ = train_lgbm_model(series_train_raw, ext_vals=ext_train_sim,
                                               lookback=6, use_log=use_log)
                if lgb_sim is not None:
                    for h_idx in range(n_holdout):
                        ext_row = ext_train_sim[-1] if ext_train_sim is not None else None
                        pred_lgb = forecast_lgbm(lgb_sim, series_train_raw,
                                                  ext_fc_row=ext_row, h=h_idx+1,
                                                  lookback=6, use_log=use_log)
                        if isinstance(pred_lgb, list):
                            fc_lgb[h_idx] = pred_lgb[-1] if pred_lgb else series_train_raw.mean()
                        else:
                            fc_lgb[h_idx] = float(pred_lgb)
            except:
                pass

        # Geometric ensemble
        preds_ens = []
        for h_idx in range(n_holdout):
            pred_list_, weight_list_ = [], []
            for mn, pv_m in [('Kalman', fc_kf[h_idx]), ('HW', fc_hw[h_idx]),
                              ('SARIMAX', fc_sa[h_idx]), ('LightGBM', fc_lgb[h_idx])]:
                wt = w.get(mn, 0)
                if wt > 0 and pv_m > 0:
                    pred_list_.append(pv_m)
                    weight_list_.append(wt)
            if pred_list_:
                pe = float(np.exp(
                    np.dot(np.array(weight_list_) / sum(weight_list_),
                           np.log(np.maximum(pred_list_, 1e-10)))
                ))
            else:
                pe = float(series_train_raw.mean())
            preds_ens.append(max(0, pe * bias_cf))
        sim_preds[key] = preds_ens

    # Adaptive F×S
    for idx in range(n_holdout):
        f_ = sim_preds['frequency'][idx]
        s_ = sim_preds['severity'][idx]
        t_ = sim_preds['total_claim'][idx]
        pred_m = test_data_sim['bulan_masuk'].values[idx]
        t_new, ratio, fs_weight = adaptive_fs_enforcement(
            f_, s_, t_, GLOBAL_FS_CORRECTION, MONTHLY_FS_CORRECTION,
            pred_m, wf_mape_freq, wf_mape_sev, wf_mape_total
        )
        sim_preds['total_claim'][idx] = t_new

    # ─── Print results ──────────────────────────────────────────────────────
    print(f"\n  {'Holdout':<10} {'Target':<15} {'Aktual':>15} {'Prediksi':>15} {'Error%':>8}")
    print("  " + "-" * 66)

    all_errors   = []
    target_mapes = {}

    for key in ['frequency', 'severity', 'total_claim']:
        y_true = actuals[key]
        y_pred = sim_preds[key]
        errs_target = []
        for idx in range(n_holdout):
            err = abs(y_true[idx] - y_pred[idx]) / max(y_true[idx], 1e-5) * 100
            errs_target.append(err)
            all_errors.append(err)
            if key == 'frequency':
                yt_d = f"{y_true[idx]:,.0f}"
                yp_d = f"{y_pred[idx]:,.0f}"
            elif key == 'severity':
                yt_d = f"{y_true[idx]/1e6:,.2f}M"
                yp_d = f"{y_pred[idx]/1e6:,.2f}M"
            else:
                yt_d = f"{y_true[idx]/1e9:,.3f}B"
                yp_d = f"{y_pred[idx]/1e9:,.3f}B"
            print(f"  Holdout-{idx+1}   {key:<15} {yt_d:>15} {yp_d:>15} {err:>7.2f}%")
        target_mapes[key] = np.mean(errs_target)
        print("  " + "-" * 66)

    final_sim_score = np.mean(all_errors)

    # Bandingkan dengan teman v16
    teman = {'frequency': 5.87, 'severity': 2.44, 'total_claim': 7.12}
    teman_avg = np.mean(list(teman.values()))

    print(f"\n  📊 PERBANDINGAN SIMULASI HOLDOUT:")
    print(f"  {'Target':<15} {'Teman v16':>10} {'Kita v24':>10} {'Delta':>8}  Hasil")
    print(f"  {'-'*55}")
    for key, label in [('frequency','Frequency'),('severity','Severity'),('total_claim','Total Claim')]:
        our   = target_mapes[key]
        their = teman[key]
        delta = our - their
        result = "✅ LEBIH BAIK" if delta < 0 else "❌ lebih buruk"
        print(f"  {label:<15} {their:>9.2f}% {our:>9.2f}% {delta:>+7.2f}%  {result}")
    print(f"  {'-'*55}")
    delta_avg = final_sim_score - teman_avg
    result_avg = "✅ LEBIH BAIK" if delta_avg < 0 else "❌ lebih buruk"
    print(f"  {'AVG':<15} {teman_avg:>9.2f}% {final_sim_score:>9.2f}% {delta_avg:>+7.2f}%  {result_avg}")

    print("\n  " + "★" * 60)
    print(f"  ★★★  ESTIMASI SKOR KITA  (v24) : {final_sim_score:.4f}%")
    print(f"  ★★★  ESTIMASI SKOR TEMAN (v16) : {teman_avg:.4f}%")
    print(f"  ★★★  SELISIH                   : {delta_avg:+.4f}%  {result_avg}")
    print("  " + "★" * 60)

    # Forecast plot
    try:
        fig_pred = plt.figure(figsize=(18, 14))
        fig_pred.suptitle(
            'Prediksi Klaim: Agt–Des 2025 — v24.0\n'
            '[v24] 7-Model Ensemble: Kalman+HW+SARIMAX+STL+Prophet+LightGBM+ExtraTrees\n'
            'Adaptive Weights + Per-Month Bias + Conformal PI + Regime Detection',
            fontsize=11, fontweight='bold'
        )
        gs_pred = gridspec.GridSpec(3, 1, figure=fig_pred, hspace=0.55)

        hist12  = monthly.tail(12).copy()
        hist_dt = pd.to_datetime([str(ym) for ym in hist12['year_month']])
        pred_dt = [pd.Timestamp(f'2025-{m:02d}-01') for m in PRED_MONTHS_2025]

        cfg_plot = [
            ('frequency',  [all_preds_2025['frequency'][m]   for m in PRED_MONTHS_2025],
             COLORS[0], 'Frekuensi Klaim (klaim/bulan)', 1, 'Klaim'),
            ('severity_w', [all_preds_2025['severity'][m]    for m in PRED_MONTHS_2025],
             COLORS[1], 'Severitas Wins (Juta IDR)', 1e6, 'Juta IDR'),
            ('total_claim', [all_preds_2025['total_claim'][m] for m in PRED_MONTHS_2025],
             COLORS[2], 'Total Klaim (Miliar IDR)', 1e9, 'Miliar IDR'),
        ]

        for i, (tc, py25r, color, title, div, unit) in enumerate(cfg_plot):
            ax = fig_pred.add_subplot(gs_pred[i])
            hist_col = tc if tc in hist12.columns else tc.replace('_w', '')
            hy = hist12[hist_col].values / div if hist_col in hist12.columns else np.zeros(len(hist12))
            py25 = [v / div for v in py25r]

            ax.plot(hist_dt, hy, color='#636363', lw=2, marker='o', ms=6, label='Historis')
            ax.fill_between(hist_dt, 0, hy, alpha=0.06, color='gray')
            ax.plot([hist_dt[-1], pred_dt[0]], [hy[-1], py25[0]],
                    color=color, lw=1.5, ls=':', alpha=0.5)
            ax.plot(pred_dt, py25, color=color, lw=3, marker='D', ms=10,
                    ls='--', label='Prediksi v24 2025', zorder=5)

            wf_key  = tc.replace('_w', '')
            wf_mape = wf_results.get(wf_key, {}).get('mean_mape', 10)
            sigma   = np.mean(hy) * wf_mape / 100 if np.mean(hy) > 0 else 1
            ax.fill_between(pred_dt,
                            [max(0, v - sigma) for v in py25],
                            [v + sigma for v in py25],
                            alpha=0.12, color=color, label=f'±{wf_mape:.1f}% CI')

            # [v24] Conformal PI if available
            cp_obj = final_models.get(wf_key, {}).get('conformal', None)
            if cp_obj is not None:
                lo, hi = cp_obj.predict_interval(np.array(py25r) / div)
                ax.fill_between(pred_dt, lo, hi, alpha=0.08, color='purple',
                                label='Conformal 90% PI')

            for dt, val in zip(pred_dt, py25):
                ax.annotate(f'{val:.1f}', (dt, val),
                            textcoords='offset points', xytext=(0, 14),
                            ha='center', fontsize=9, color=color, fontweight='bold')

            ax.set_title(title, fontsize=11, fontweight='bold')
            ax.set_ylabel(unit)
            ax.legend(loc='upper left', fontsize=8)
            ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%b %Y'))
            plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

        plt.savefig('predictions_forecast_v24.png', dpi=90, bbox_inches='tight', facecolor='white')
        plt.close()
        print("\n  ✓ predictions_forecast_v24.png tersimpan")
    except Exception as e:
        print(f"  ⚠ Forecast plot: {e}")
        plt.close('all')

    print("═" * 65 + "\n")

hitung_simulasi_v24()


# ========================== FINAL SUMMARY ===================================
print("\n" + "=" * 80)
print("  RINGKASAN HASIL FINAL v24.0")
print("=" * 80)

print("=" * 80)

  ✓ LightGBM tersedia
  ℹ XGBoost tidak tersedia
  ✓ Prophet tersedia
  DSC MCF ITB — INSURANCE CLAIM PREDICTION v24.0
  KEY UPGRADES: LightGBM/XGBoost + Adaptive Weights + Per-Month Bias
  + Conformal PI + DSHW + Regime Detection + Smarter F×S
  Target: AVG MAPE < 3.0%

[1/10] LOADING DATA...
  → 'Data_Klaim (1).csv' (encoding=utf-8, sep=',')
  → 'Data_Polis.csv' (encoding=utf-8, sep=',')
  → 'sample_submission.csv' (encoding=utf-8, sep=',')
  ✓ Data Klaim  : 4,627 × 13
  ✓ Data Polis  : 4,096 × 6
  ✓ Sample IDs  : ['2025_08_Claim_Frequency', '2025_08_Claim_Severity', '2025_08_Total_Claim', '2025_09_Claim_Frequency', '2025_09_Claim_Severity', '2025_09_Total_Claim', '2025_10_Claim_Frequency', '2025_10_Claim_Severity', '2025_10_Total_Claim', '2025_11_Claim_Frequency', '2025_11_Claim_Severity', '2025_11_Total_Claim', '2025_12_Claim_Frequency', '2025_12_Claim_Severity', '2025_12_Total_Claim']

[2/10] PRE-PROCESSING...
  ✓ df_paid     : 4,627 baris
  Best winsor.  : target_w99 (|ratio-1|=0

17:37:05 - cmdstanpy - INFO - Chain [1] start processing
17:37:05 - cmdstanpy - INFO - Chain [1] done processing


    [E] Prophet MAPE=6.29%
    [F] LightGBM...
    [F] LightGBM OOS-MAPE=6.54%
    [G] Extra Trees...
    [G] Extra Trees OOS-MAPE=7.57%
    MAPEs: Kalman=9.4% HW=inf% SARIMAX=3.2% STL=7.0% Prophet=6.3% LightGBM=6.5% ExtraTrees=7.6%
    Weights: Kalman=0.059 HW=0.000 SARIMAX=0.497 STL=0.104 Prophet=0.130 LightGBM=0.120 ExtraTrees=0.090
    Bias correction: ×0.9666
    [Kalman] phi=0.850 q_l=1520.07320 q_t=0.00000
    [Kalman] phi=0.850 q_l=1520.07320 q_t=0.00000
    [Kalman] phi=0.850 q_l=1520.07320 q_t=0.00000
    [Kalman] phi=0.850 q_l=1520.07320 q_t=0.00000
    [Kalman] phi=0.850 q_l=1520.07320 q_t=0.00000
    [Kalman] phi=0.850 q_l=1520.07320 q_t=0.00000

  ─── severity ───────────────────────────────────────────────────
    Regime: normal (ratio=1.016)
    [A] Kalman Trend Filter...
    [Kalman] phi=0.850 q_l=0.00000 q_t=0.00000
    [Kalman] phi=0.850 q_l=0.00000 q_t=0.00000
    [Kalman] phi=0.850 q_l=0.00000 q_t=0.00000
    [Kalman] phi=0.850 q_l=0.00000 q_t=0.00000
    [Kalman] 

17:37:08 - cmdstanpy - INFO - Chain [1] start processing
17:37:08 - cmdstanpy - INFO - Chain [1] done processing


    [E] Prophet MAPE=5.10%
    [F] LightGBM...
    [F] LightGBM OOS-MAPE=5.88%
    [G] Extra Trees...
    [G] Extra Trees OOS-MAPE=5.47%
    MAPEs: Kalman=8.0% HW=inf% SARIMAX=3.9% STL=4.8% Prophet=5.1% LightGBM=5.9% ExtraTrees=5.5%
    Weights: Kalman=0.070 HW=0.000 SARIMAX=0.296 STL=0.189 Prophet=0.170 LightGBM=0.128 ExtraTrees=0.147
    Bias correction: ×1.0113
    [Kalman] phi=0.850 q_l=0.00000 q_t=0.00000
    [Kalman] phi=0.850 q_l=0.00000 q_t=0.00000
    [Kalman] phi=0.850 q_l=0.00000 q_t=0.00000
    [Kalman] phi=0.850 q_l=0.00000 q_t=0.00000
    [Kalman] phi=0.850 q_l=0.00000 q_t=0.00000
    [Kalman] phi=0.850 q_l=0.00000 q_t=0.00000

  ─── total_claim ───────────────────────────────────────────────────
    Regime: normal (ratio=1.015)
    [A] Kalman Trend Filter...
    [Kalman] phi=0.850 q_l=0.00000 q_t=0.00000
    [Kalman] phi=0.850 q_l=0.00000 q_t=0.00000
    [Kalman] phi=0.850 q_l=0.00000 q_t=0.00000
    [Kalman] phi=0.850 q_l=0.00000 q_t=0.00000
    [Kalman] phi=0.850 q_l=0

17:37:11 - cmdstanpy - INFO - Chain [1] start processing
17:37:11 - cmdstanpy - INFO - Chain [1] done processing


    [E] Prophet MAPE=9.94%
    [F] LightGBM...
    [F] LightGBM OOS-MAPE=6.79%
    [G] Extra Trees...
    [G] Extra Trees OOS-MAPE=7.04%
    MAPEs: Kalman=12.9% HW=inf% SARIMAX=5.3% STL=6.6% Prophet=9.9% LightGBM=6.8% ExtraTrees=7.0%
    Weights: Kalman=0.052 HW=0.000 SARIMAX=0.304 STL=0.198 Prophet=0.087 LightGBM=0.186 ExtraTrees=0.173
    Bias correction: ×0.9652
    [Kalman] phi=0.850 q_l=0.00000 q_t=0.00000
    [Kalman] phi=0.850 q_l=0.00000 q_t=0.00000
    [Kalman] phi=0.850 q_l=0.00000 q_t=0.00000
    [Kalman] phi=0.850 q_l=0.00000 q_t=0.00000
    [Kalman] phi=0.850 q_l=0.00000 q_t=0.00000
    [Kalman] phi=0.850 q_l=0.00000 q_t=0.00000

  ─────────────────────────────────────────────────────────────────────────
  frequency      : Best Single Model MAPE=3.22%
  severity       : Best Single Model MAPE=3.86%
  total_claim    : Best Single Model MAPE=5.31%
  AVG MAPE: 4.13%

[8/10] PREDIKSI AGT–DES 2025 (v24)...

  === Prediksi 2025 (Agt–Des) ===
    [FLOOR] Sev 10: 44.15M → floor 45

In [34]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
import re
from itertools import product
warnings.filterwarnings('ignore')

from scipy import stats
from scipy.optimize import minimize
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import RobustScaler

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import acf

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False

try:
    from prophet import Prophet
    HAS_PROPHET = True
except ImportError:
    HAS_PROPHET = False

COLORS = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#44BBA4', '#E84855']

DATA_KLAIM  = r"Data_Klaim (1).csv"
DATA_POLIS  = r"Data_Polis.csv"
DATA_SAMPLE = r"sample_submission.csv"

print("=" * 70)
print("  DSC MCF ITB v25.0 — CLEAN PIPELINE, NO LEAKAGE")
print("  Target: Close gap simulasi vs Kaggle")
print("=" * 70)


# ── HELPERS ──────────────────────────────────────────────────────────────────

def calc_mape(y_true, y_pred, eps=1.0):
    yt = np.maximum(np.array(y_true, dtype=float), eps)
    yp = np.maximum(np.array(y_pred, dtype=float), 0)
    return float(np.mean(np.abs((yt - yp) / yt)) * 100)

def parse_numeric(series):
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors='coerce')
    def _p(val):
        if pd.isna(val): return np.nan
        s = re.sub(r'[^\d.,\-]', '', str(val).strip())
        if not s: return np.nan
        if ',' in s and '.' in s:
            if s.rfind(',') > s.rfind('.'): s = s.replace('.','').replace(',','.')
            else: s = s.replace(',','')
        elif ',' in s:
            parts = s.split(',')
            s = s.replace(',','.') if len(parts)==2 and len(parts[1])<=2 else s.replace(',','')
        elif s.count('.') > 1:
            s = s.replace('.','')
        try: return float(s)
        except: return np.nan
    return series.apply(_p)

def parse_date(series):
    if pd.api.types.is_integer_dtype(series) or pd.api.types.is_float_dtype(series):
        return pd.to_datetime(series.astype(str).str.split('.').str[0], format='%Y%m%d', errors='coerce')
    result = pd.to_datetime(series, errors='coerce', dayfirst=True)
    for fmt in ['%d/%m/%Y','%m/%d/%Y','%Y/%m/%d','%d-%m-%Y','%Y%m%d','%Y-%m-%d']:
        miss = result.isna()
        if not miss.any(): break
        try: result[miss] = pd.to_datetime(series[miss], format=fmt, errors='coerce')
        except: pass
    return result

def find_col(df, kws, required=False):
    cl = {c.lower().strip(): c for c in df.columns}
    for k in kws:
        kl = k.lower()
        if kl in cl: return cl[kl]
        for col_l, col_o in cl.items():
            if kl in col_l or col_l in kl: return col_o
    if required: raise KeyError(f"Not found: {kws}")
    return None

def read_csv(path):
    for enc in ['utf-8','utf-8-sig','latin-1','cp1252']:
        for sep in [',',';','\t','|']:
            try:
                df = pd.read_csv(path, encoding=enc, sep=sep, low_memory=False)
                if df.shape[1] > 1:
                    df.columns = df.columns.str.strip()
                    return df
            except: continue
    raise ValueError(f"Cannot read: {path}")


# ── KALMAN TREND FILTER ───────────────────────────────────────────────────────

class KalmanTrend:
    def __init__(self, phi=0.95, q_l=0.01, q_t=0.001):
        self.phi, self.q_l, self.q_t = phi, q_l, q_t
        self.level_ = self.trend_ = None
        self.fitted_ = None

    def _ll(self, y, phi, q_l, q_t):
        state = np.array([y[0], 0.0])
        P = np.eye(2) * 10.0
        F = np.array([[1,1],[0,phi]])
        Q = np.diag([q_l, q_t])
        H = np.array([[1,0]]); R = np.array([[1.0]])
        ll = 0.0
        for t in range(len(y)):
            sp = F @ state; Pp = F @ P @ F.T + Q
            inn = y[t] - float(H @ sp)
            S = float(H @ Pp @ H.T + R)
            if S <= 0: return -1e10
            ll += -0.5*(np.log(2*np.pi*S) + inn**2/S)
            K = (Pp @ H.T)/S
            state = sp + K.flatten()*inn
            P = (np.eye(2) - K@H) @ Pp
        return ll

    def fit(self, y, optimize=True):
        y = np.array(y, dtype=float)
        if optimize:
            def neg(p):
                ph = np.clip(p[0], 0.85, 1.0)
                try: return -self._ll(y, ph, np.exp(p[1]), np.exp(p[2]))
                except: return 1e10
            try:
                res = minimize(neg, [0.95, np.log(0.01), np.log(0.001)],
                               method='Nelder-Mead',
                               options={'xatol':1e-5,'fatol':1e-5,'maxiter':400})
                self.phi = np.clip(res.x[0], 0.85, 1.0)
                self.q_l = np.exp(res.x[1])
                self.q_t = np.exp(res.x[2])
            except: pass

        state = np.array([y[0], 0.0]); P = np.eye(2)*10.0
        F = np.array([[1,1],[0,self.phi]]); Q = np.diag([self.q_l, self.q_t])
        H = np.array([[1,0]]); R = np.array([[1.0]])
        levels = []
        for t in range(len(y)):
            sp = F@state; Pp = F@P@F.T+Q
            inn = y[t]-float(H@sp); S = float(H@Pp@H.T+R)
            K = (Pp@H.T)/S; state = sp+K.flatten()*inn; P=(np.eye(2)-K@H)@Pp
            levels.append(float(state[0]))
        self.level_ = float(state[0]); self.trend_ = float(state[1])
        self.fitted_ = np.array(levels)
        return self

    def forecast(self, h):
        lvl, trd = self.level_, self.trend_
        out = []
        for _ in range(h):
            lvl = lvl + trd; trd = self.phi * trd; out.append(lvl)
        return np.array(out)


# ── OPTIMAL DAMPING ───────────────────────────────────────────────────────────

def best_damping(series, phis=None):
    if phis is None:
        phis = [0.80, 0.85, 0.90, 0.95, 0.97, 0.98, 1.00]
    series = np.array(series, dtype=float)
    n = len(series)
    min_tr = max(8, n-6)
    best_phi, best_mape = 0.95, np.inf
    mape_map = {}
    for phi in phis:
        errs = []
        for ti in range(min_tr, n):
            try:
                if phi < 1.0:
                    m = ExponentialSmoothing(series[:ti], trend='add', damped_trend=True,
                                            initialization_method='estimated')
                    f = m.fit(damping_trend=phi, optimized=True, use_brute=False)
                else:
                    m = ExponentialSmoothing(series[:ti], trend='add', damped_trend=False,
                                            initialization_method='estimated')
                    f = m.fit(optimized=True, use_brute=False)
                p = max(0, float(f.forecast(1).iloc[0]))
                errs.append(calc_mape([series[ti]], [p]))
            except: continue
        if errs:
            avg = np.mean(errs); mape_map[phi] = avg
            if avg < best_mape: best_mape=avg; best_phi=phi
    return best_phi, best_mape


# ── EXTERNAL DATA (minimal, low weight) ──────────────────────────────────────

SEASONAL_IDX = {1:.85,2:.90,3:1.00,4:1.05,5:.95,6:1.10,7:1.00,8:.95,9:1.00,10:1.05,11:1.10,12:1.15}
RAMADAN = {(2024,3),(2025,3)}; EID = {(2024,4),(2025,4)}

def ext_row(yr, mo, base_yr=2024, base_mo=1):
    t = (yr - base_yr)*12 + (mo - base_mo)
    return {
        'infl': (1.007)**t,
        'seas': SEASONAL_IDX.get(mo, 1.0),
        'ram':  int((yr,mo) in RAMADAN),
        'eid':  int((yr,mo) in EID),
        'ye':   int(mo in [11,12]),
        'trend': t/12.0,
        'sin_m': np.sin(2*np.pi*mo/12),
        'cos_m': np.cos(2*np.pi*mo/12),
    }


# ── SARIMAX ORDER SEARCH ──────────────────────────────────────────────────────

def sarimax_best_order(y, exog=None):
    best_aic, best_ord = np.inf, (1,1,1)
    for p,d,q in product(range(3), range(2), range(3)):
        if p==0 and q==0: continue
        try:
            m = SARIMAX(y, exog=exog, order=(p,d,q),
                        enforce_stationarity=False, enforce_invertibility=False)
            r = m.fit(disp=False, maxiter=80)
            if r.aic < best_aic: best_aic=r.aic; best_ord=(p,d,q)
        except: continue
    return best_ord


# ── WALK-FORWARD CV (strictly no leakage) ────────────────────────────────────

def walk_forward_cv(series, ext_series=None, min_train=10, use_log=True):
    """
    Pure walk-forward CV: model refitted on expanding window each step.
    ext_series: array of shape (n, n_ext_features)
    Returns: mapes list, preds list, actuals list
    """
    series = np.array(series, dtype=float)
    n = len(series)
    mapes, preds, actuals = [], [], []

    for ti in range(min_train, n):
        y_tr = series[:ti]
        y_te = series[ti]
        ext_tr = ext_series[:ti] if ext_series is not None else None
        ext_te = ext_series[ti:ti+1] if ext_series is not None else None

        if len(y_tr) < 6: continue
        y_fit = np.log1p(np.maximum(y_tr, 0)) if use_log else y_tr.copy()

        model_preds = []

        # Kalman
        try:
            kf = KalmanTrend(phi=0.95); kf.fit(y_fit, optimize=False)
            p = float(kf.forecast(1)[0])
            p = max(0, np.expm1(p) if use_log else p)
            model_preds.append(p)
        except: model_preds.append(float(y_tr.mean()))

        # HW
        try:
            phi_hw, _ = best_damping(y_tr if not use_log else np.expm1(y_fit),
                                     phis=[0.90, 0.95, 0.98, 1.0])
            m = ExponentialSmoothing(y_fit, trend='add', damped_trend=(phi_hw<1.0),
                                     initialization_method='estimated')
            kw = {'optimized':True,'use_brute':False}
            if phi_hw < 1.0: kw['damping_trend'] = phi_hw
            p = max(0, float(m.fit(**kw).forecast(1).iloc[0]))
            p = max(0, np.expm1(p) if use_log else p)
            model_preds.append(p)
        except: model_preds.append(float(y_tr.mean()))

        # SARIMAX
        try:
            if ext_tr is not None:
                m = SARIMAX(y_fit, exog=ext_tr, order=(1,1,1),
                            enforce_stationarity=False, enforce_invertibility=False)
                f = m.fit(disp=False, maxiter=80)
                p = float(f.forecast(1, exog=ext_te).iloc[-1])
            else:
                m = SARIMAX(y_fit, order=(1,1,1),
                            enforce_stationarity=False, enforce_invertibility=False)
                p = float(m.fit(disp=False, maxiter=80).forecast(1).iloc[-1])
            p = max(0, np.expm1(p) if use_log else p)
            model_preds.append(p)
        except: model_preds.append(float(y_tr.mean()))

        ens = float(np.mean([p for p in model_preds if p > 0]))
        mapes.append(calc_mape([y_te], [ens]))
        preds.append(ens)
        actuals.append(y_te)

    return mapes, preds, actuals


# ═══════════════════════════════════════════════════════════════════════════════
print("\n[1] LOADING DATA...")
df_klaim  = read_csv(DATA_KLAIM)
df_polis  = read_csv(DATA_POLIS)
df_sample = read_csv(DATA_SAMPLE)
print(f"  Klaim: {df_klaim.shape} | Polis: {df_polis.shape}")

print("\n[2] PREPROCESSING...")
CK = find_col(df_klaim, ['nomor polis','nopolis','polis_id','polis'], required=True)
CS = find_col(df_klaim, ['status klaim','status claim','status'], required=True)
CT = find_col(df_klaim, ['nominal klaim yang disetujui','klaim disetujui','nominal disetujui','disetujui'])
CM = find_col(df_klaim, ['tanggal pasien masuk rs','tgl masuk','masuk rs','masuk'])
CL = find_col(df_klaim, ['tanggal pasien keluar rs','tgl keluar','keluar rs','keluar'])
CP_K = find_col(df_polis, ['nomor polis','nopolis','polis_id','polis'], required=True)
C_LAHIR = find_col(df_polis, ['tanggal lahir','tgl lahir','birth date','dob'])
C_EFF   = find_col(df_polis, ['tanggal efektif polis','tanggal efektif','efektif'])
C_PLAN  = find_col(df_polis, ['plan code','plancode','plan'])
C_ICD   = find_col(df_klaim, ['icd diagnosis','icd_diagnosis','icd code','icd'])
C_IPOP  = find_col(df_klaim, ['inpatient/outpatient','rawat inap','ip_op','rawat'])
C_BIAYA = find_col(df_klaim, ['nominal biaya rs yang terjadi','nominal biaya rs','biaya rs','biaya'])

df_klaim[CT] = parse_numeric(df_klaim[CT])
if C_BIAYA: df_klaim[C_BIAYA] = parse_numeric(df_klaim[C_BIAYA])
for c in [CM, CL]:
    if c: df_klaim[c] = parse_date(df_klaim[c])
for c in [C_LAHIR, C_EFF]:
    if c: df_polis[c] = parse_date(df_polis[c])

df = df_klaim.merge(df_polis, left_on=CK, right_on=CP_K, how='left')
if CK != CP_K and CP_K in df.columns: df = df.drop(columns=[CP_K])

# Impute missing dates
if CM:
    med_lag = 14
    if CL and CL in df.columns:
        known = df[CM].notna() & df[CL].notna()
        if known.sum() > 0:
            lag = (df.loc[known,CL] - df.loc[known,CM]).dt.days
            med_lag = int(np.clip(lag.median(), 0, 90)) if not lag.isna().all() else 14

for col in df.select_dtypes(include=[np.number]).columns:
    if df[col].isna().any(): df[col] = df[col].fillna(df[col].median())
for col in df.select_dtypes(include='object').columns:
    if df[col].isna().any():
        mode_ = df[col].mode()
        df[col] = df[col].fillna(mode_.iloc[0] if not mode_.empty else 'Unknown')

# PAID filter
sc = df[CS].value_counts()
paid_vals = [v for v in sc.index if any(k in str(v).lower() for k in ['paid','dibayar','approved','disetujui','settled'])]
if not paid_vals: paid_vals = [sc.index[0]]
df_paid = df[df[CS].isin(paid_vals)].copy().reset_index(drop=True)
df_paid[CT] = parse_numeric(df_paid[CT])

masuk_col = CM if CM and CM in df_paid.columns else CL
df_paid[masuk_col] = pd.to_datetime(df_paid[masuk_col], errors='coerce')
vd = df_paid[masuk_col].dropna().sort_values()
if df_paid[masuk_col].isna().any() and len(vd):
    df_paid[masuk_col] = df_paid[masuk_col].fillna(vd.iloc[len(vd)//2])

df_paid['tahun']  = df_paid[masuk_col].dt.year.fillna(2024).astype(int)
df_paid['bulan']  = df_paid[masuk_col].dt.month.fillna(1).astype(int)
df_paid['ym']     = df_paid[masuk_col].dt.to_period('M')

# [v25 KEY] Winsorization ditetapkan sekali dari full data (acceptable - ini preprocessing)
# Yang penting: FS correction TIDAK menggunakan future data saat inference
TARGET = CT
pct_lo, pct_hi = 0.02, 0.98
lo_v = df_paid[TARGET].quantile(pct_lo)
hi_v = df_paid[TARGET].quantile(pct_hi)
df_paid['target_w'] = df_paid[TARGET].clip(lower=lo_v, upper=hi_v)

print(f"  ✓ df_paid: {len(df_paid):,} baris | Paid: {paid_vals}")
print(f"  Winsor [{pct_lo:.0%},{pct_hi:.0%}]: [{lo_v:,.0f}, {hi_v:,.0f}]")


print("\n[3] FEATURE ENGINEERING...")
tgl = pd.to_datetime(df_paid[masuk_col], errors='coerce')
df_paid['usia'] = ((tgl - pd.to_datetime(df_paid[C_LAHIR], errors='coerce')).dt.days/365.25
                   ).clip(0,100).fillna(35) if C_LAHIR and C_LAHIR in df_paid.columns else 35.0
df_paid['lama_rawat'] = ((pd.to_datetime(df_paid[CL], errors='coerce') - tgl).dt.days
                         ).clip(0,365).fillna(0) if CL and CL in df_paid.columns else 1.0
df_paid['is_ip'] = (df_paid[C_IPOP].astype(str).str.lower().str.contains('in').astype(int)
                    if C_IPOP and C_IPOP in df_paid.columns else 0)

if C_ICD and C_ICD in df_paid.columns:
    df_paid['icd_ch'] = df_paid[C_ICD].astype(str).str[0].str.upper().fillna('Z')
    df_paid['icd_3']  = df_paid[C_ICD].astype(str).str[:3].str.upper().fillna('Z00')
    gmed = df_paid[TARGET].median()
    med_icd = df_paid.groupby('icd_3')[TARGET].median()
    pct_icd = med_icd.quantile([.20,.40,.60,.80])
    def _tier(x):
        v = med_icd.get(x, gmed)
        v = gmed if pd.isna(v) else v
        if v<=pct_icd[.20]: return 0
        elif v<=pct_icd[.40]: return 1
        elif v<=pct_icd[.60]: return 2
        elif v<=pct_icd[.80]: return 3
        return 4
    df_paid['icd_tier'] = df_paid['icd_3'].map(_tier).fillna(2).astype(int)
    chronic = {'C','D','E','I','J','K','M','N'}
    df_paid['is_chronic'] = df_paid['icd_ch'].isin(chronic).astype(int)
else:
    df_paid['icd_tier'] = 2; df_paid['is_chronic'] = 0

if C_PLAN and C_PLAN in df_paid.columns:
    plan_sev = df_paid.groupby(C_PLAN)[TARGET].median()
    df_paid['plan_sev'] = df_paid[C_PLAN].map(plan_sev).fillna(df_paid[TARGET].median())
    plan_rank = {'M-001':3,'M-002':2,'M-003':1}
    df_paid['plan_rank'] = df_paid[C_PLAN].map(plan_rank).fillna(1)
else:
    df_paid['plan_sev'] = df_paid[TARGET].median(); df_paid['plan_rank'] = 1

if C_BIAYA and C_BIAYA in df_paid.columns:
    df_paid['ratio_kb'] = (df_paid[TARGET]/df_paid[C_BIAYA].replace(0,np.nan)).clip(0,5).fillna(0.8)
else:
    df_paid['ratio_kb'] = 0.8


print("\n[4] AGREGASI BULANAN...")
monthly = df_paid.groupby(['tahun','bulan']).agg(
    frequency  = (TARGET, 'count'),
    severity   = (TARGET, 'mean'),
    severity_w = ('target_w', 'mean'),
    severity_m = (TARGET, 'median'),
    total      = (TARGET, 'sum'),
    usia_m     = ('usia', 'mean'),
    rawat_m    = ('lama_rawat', 'mean'),
    pct_ip     = ('is_ip', 'mean'),
    pct_chr    = ('is_chronic', 'mean'),
    avg_tier   = ('icd_tier', 'mean'),
    plan_sev_m = ('plan_sev', 'mean'),
    plan_rnk_m = ('plan_rank', 'mean'),
    ratio_kb_m = ('ratio_kb', 'mean'),
).reset_index()
monthly['ym'] = pd.to_datetime(monthly['tahun'].astype(str)+'-'+monthly['bulan'].astype(str).str.zfill(2))
monthly = monthly.sort_values('ym').reset_index(drop=True)

# Lag features
for lag in [1,2,3]:
    monthly[f'lag_f{lag}'] = monthly['frequency'].shift(lag)
    monthly[f'lag_s{lag}'] = monthly['severity_w'].shift(lag)
monthly['lag_t1'] = monthly['total'].shift(1)
monthly['roll_f3'] = monthly['frequency'].rolling(3,min_periods=1).mean().shift(1)
monthly['roll_f6'] = monthly['frequency'].rolling(6,min_periods=1).mean().shift(1)
monthly['roll_s3'] = monthly['severity_w'].rolling(3,min_periods=1).mean().shift(1)
monthly['sin_m'] = np.sin(2*np.pi*monthly['bulan']/12)
monthly['cos_m'] = np.cos(2*np.pi*monthly['bulan']/12)
monthly['yoy_f'] = monthly['frequency'].shift(12)
monthly['yoy_s'] = monthly['severity_w'].shift(12)

lag_cols = [c for c in monthly.columns if c.startswith(('lag_','roll_','yoy_'))]
monthly[lag_cols] = monthly[lag_cols].bfill().ffill().fillna(0)

N = len(monthly)
print(f"  Monthly data: {N} bulan")
print(f"  Freq: mean={monthly['frequency'].mean():.0f} last={monthly['frequency'].iloc[-1]:.0f}")
print(f"  Sev_w: mean={monthly['severity_w'].mean()/1e6:.2f}M")

# [v25 KEY] FS correction dari SEMUA data training (tidak recalibrate per forecast step)
monthly['fs_w'] = monthly['frequency'] * monthly['severity_w']
monthly['fs_cf'] = monthly['total'] / (monthly['fs_w'] + 1e-10)
GLOBAL_CF = float(monthly['fs_cf'].median())
print(f"  Global F×S CF: {GLOBAL_CF:.4f}")

# YoY growth (hanya dari months yang punya YoY data)
yoy_m = monthly['yoy_f'].notna() & (monthly['yoy_f'] > 0)
yoy_s = monthly['yoy_s'].notna() & (monthly['yoy_s'] > 0)
YOY_FREQ = float((monthly.loc[yoy_m,'frequency']/monthly.loc[yoy_m,'yoy_f'] - 1).median()) if yoy_m.sum()>=2 else 0.03
YOY_SEV  = float((monthly.loc[yoy_s,'severity_w']/monthly.loc[yoy_s,'yoy_s'] - 1).median()) if yoy_s.sum()>=2 else 0.05
print(f"  YoY: Freq={YOY_FREQ*100:+.1f}% Sev={YOY_SEV*100:+.1f}%")

# Anchors
FREQ_3M = float(monthly['frequency'].tail(3).mean())
FREQ_6M = float(monthly['frequency'].tail(6).mean())
SEV_3M  = float(monthly['severity_w'].tail(3).mean())
SEV_6M  = float(monthly['severity_w'].tail(6).mean())
F5,F95  = monthly['frequency'].quantile(0.05), monthly['frequency'].quantile(0.95)
S5,S95  = monthly['severity_w'].quantile(0.05), monthly['severity_w'].quantile(0.95)
T5,T95  = monthly['total'].quantile(0.05), monthly['total'].quantile(0.95)


# ── EXTERNAL FEATURES ARRAY ──────────────────────────────────────────────────
base_yr = int(monthly['tahun'].iloc[0]); base_mo = int(monthly['bulan'].iloc[0])
EXT_KEYS = ['infl','seas','ram','eid','ye','trend','sin_m','cos_m']
ext_all = np.array([
    [ext_row(int(r.tahun), int(r.bulan), base_yr, base_mo)[k] for k in EXT_KEYS]
    for _, r in monthly.iterrows()
], dtype=float)


print("\n[5] WALK-FORWARD CV (no leakage)...")
MIN_TR = max(10, N-8)

wf = {}
for tgt, col, use_log in [('frequency','frequency',False),
                            ('severity','severity_w',True),
                            ('total','total',True)]:
    ser = monthly[col].values.copy()
    m_list, p_list, a_list = walk_forward_cv(ser, ext_series=ext_all,
                                               min_train=MIN_TR, use_log=use_log)
    bias = float(np.mean([(a-p)/max(abs(a),1) for a,p in zip(a_list,p_list)])) if p_list else 0.0
    wf[tgt] = {'mapes': m_list, 'mean': float(np.mean(m_list)) if m_list else 99.0,
               'bias': bias, 'preds': p_list, 'actuals': a_list}
    print(f"  {tgt:<12}: WF-MAPE={wf[tgt]['mean']:.2f}%  Bias={bias*100:+.1f}%  n={len(m_list)}")


print("\n[6] BUILDING FINAL MODELS...")

PRED_MONTHS = [8,9,10,11,12]
PRED_YEAR   = 2025

last_ym  = monthly['ym'].iloc[-1]
last_yr  = int(str(last_ym)[:4])
last_mo  = int(str(last_ym)[5:7])

ext_fc_arr = {}
for mo in PRED_MONTHS:
    steps = (PRED_YEAR - last_yr)*12 + (mo - last_mo)
    ext_fc_arr[mo] = np.array([ext_row(PRED_YEAR, mo, base_yr, base_mo)[k] for k in EXT_KEYS])

final = {}

for tgt, col, use_log in [('frequency','frequency',False),
                            ('severity','severity_w',True),
                            ('total','total',True)]:
    print(f"\n  ── {tgt} ──────────────────────────────────────────────")
    ser = monthly[col].values.copy()
    n   = len(ser)
    yf  = np.log1p(np.maximum(ser, 0)) if use_log else ser.copy()

    # [A] Kalman
    mape_kf = 99.0
    kf_model = None
    try:
        kf_model = KalmanTrend(); kf_model.fit(yf, optimize=True)
        opt_phi, opt_ql, opt_qt = kf_model.phi, kf_model.q_l, kf_model.q_t
        n_test = min(5, n//3)
        kf_oos = []
        for i in range(n-n_test, n):
            kft = KalmanTrend(phi=opt_phi, q_l=opt_ql, q_t=opt_qt)
            kft.fit(yf[:i], optimize=False)
            p = float(kft.forecast(1)[0])
            kf_oos.append(max(0, np.expm1(p) if use_log else p))
        mape_kf = calc_mape(ser[-n_test:], kf_oos)
        print(f"  [A] Kalman phi={kf_model.phi:.3f} MAPE={mape_kf:.2f}%")
    except Exception as e:
        print(f"  [A] Kalman fail: {e}")

    # [B] Damped HW
    mape_hw = 99.0; hw_model = None; best_phi = 0.95
    try:
        best_phi, best_mape_hw = best_damping(ser)
        print(f"  [B] HW phi={best_phi} MAPE={best_mape_hw:.2f}%")
        m = ExponentialSmoothing(yf, trend='add', damped_trend=(best_phi<1.0),
                                  initialization_method='estimated')
        kw = {'optimized':True,'use_brute':True}
        if best_phi < 1.0: kw['damping_trend'] = best_phi
        hw_model = m.fit(**kw)
        mape_hw = best_mape_hw
    except Exception as e:
        print(f"  [B] HW fail: {e}")

    # [C] SARIMAX + ext
    mape_sa = 99.0; sa_model = None; best_ord = (1,1,1)
    try:
        best_ord = sarimax_best_order(yf, exog=ext_all)
        m = SARIMAX(yf, exog=ext_all, order=best_ord,
                    enforce_stationarity=False, enforce_invertibility=False)
        sa_model = m.fit(disp=False, maxiter=100)
        n_test = min(5, n//3)
        p_sa = np.expm1(sa_model.predict(n-n_test, n-1)) if use_log else sa_model.predict(n-n_test, n-1)
        mape_sa = calc_mape(ser[-n_test:], np.maximum(p_sa, 0))
        print(f"  [C] SARIMAX{best_ord} MAPE={mape_sa:.2f}%")
    except Exception as e:
        print(f"  [C] SARIMAX fail: {e}")

    # [D] STL + Ridge
    mape_stl = 99.0; stl_info = None
    try:
        if n >= 12:
            ts = pd.Series(yf, index=pd.date_range('2024-01', periods=n, freq='MS'))
            stl_r = STL(ts, period=6, robust=True).fit()
            tp = np.polyfit(np.arange(n), stl_r.trend.values, 1)
            sp = np.zeros(6)
            for m_ in range(6):
                idxs = [i for i in range(n) if i%6==m_]
                if idxs: sp[m_] = np.mean(stl_r.seasonal.values[idxs])
            X = np.column_stack([np.arange(n),
                                  np.sin(2*np.pi*np.arange(n)/6),
                                  np.cos(2*np.pi*np.arange(n)/6),
                                  ext_all[:,:3]])
            sc = RobustScaler(); Xs = sc.fit_transform(X)
            ridge = Ridge(alpha=5); ridge.fit(Xs[:-1], stl_r.resid.values[1:])
            stl_info = {'tp':tp,'sp':sp,'ridge':ridge,'sc':sc,'n':n}
            n_test = min(5, n//3)
            stl_oos = []
            for i in range(n-n_test, n):
                tn = np.polyval(tp, i) + sp[i%6]
                Xi = np.array([[i, np.sin(2*np.pi*i/6), np.cos(2*np.pi*i/6)] + list(ext_all[i,:3])])
                rn = float(ridge.predict(sc.transform(Xi))[0])
                pv = tn + rn
                stl_oos.append(max(0, np.expm1(pv) if use_log else pv))
            mape_stl = calc_mape(ser[-n_test:], stl_oos)
            print(f"  [D] STL+Ridge MAPE={mape_stl:.2f}%")
    except Exception as e:
        print(f"  [D] STL fail: {e}")

    # [E] Prophet
    mape_ph = 99.0; ph_model = None
    try:
        if HAS_PROPHET:
            ser_ph = np.expm1(yf) if use_log else ser
            dates = pd.date_range(end=pd.Timestamp('2025-07-01'), periods=n, freq='MS')
            df_ph = pd.DataFrame({'ds':dates,'y':np.maximum(ser_ph,0.01)})
            pm = Prophet(yearly_seasonality=False, weekly_seasonality=False,
                         daily_seasonality=False, changepoint_prior_scale=0.25,
                         changepoint_range=0.85)
            pm.fit(df_ph, iter=300)
            ph_model = pm
            n_test = min(5, n//3)
            fut = pm.make_future_dataframe(periods=0, freq='MS')
            fp = pm.predict(fut)['yhat'].values[-n_test:]
            mape_ph = calc_mape(ser[-n_test:], np.maximum(fp, 0))
            print(f"  [E] Prophet MAPE={mape_ph:.2f}%")
    except Exception as e:
        print(f"  [E] Prophet fail: {e}")

    # [v25] Inverse-MAPE^2 weights (simple, no regime distortion)
    mapes = {'Kalman':mape_kf,'HW':mape_hw,'SARIMAX':mape_sa,'STL':mape_stl,'Prophet':mape_ph}
    valid = {k:v for k,v in mapes.items() if v < 50.0}
    if not valid: valid = {'HW': mape_hw if mape_hw < 99 else 10.0}
    total_inv = sum(1/(v**2+1e-6) for v in valid.values())
    weights = {k:(1/(mapes[k]**2+1e-6))/total_inv if k in valid else 0.0 for k in mapes}

    # [v25] Bias correction dari WF results (strict OOS only)
    bias_cf = float(np.clip(1.0 + wf[tgt]['bias'], 0.90, 1.15))

    print(f"  Weights: {' '.join(f'{k}={v:.3f}' for k,v in weights.items())}")
    print(f"  Bias CF: {bias_cf:.4f}")

    final[tgt] = {
        'kalman': kf_model, 'hw': hw_model, 'hw_phi': best_phi,
        'sarima': sa_model, 'sarima_ord': best_ord, 'stl': stl_info,
        'prophet': ph_model, 'weights': weights,
        'use_log': use_log, 'col': col, 'ser': ser, 'yf': yf,
        'n': n, 'bias_cf': bias_cf, 'ext_all': ext_all,
    }

print(f"\n  Summary WF-MAPE: Freq={wf['frequency']['mean']:.2f}% "
      f"Sev={wf['severity']['mean']:.2f}% Total={wf['total']['mean']:.2f}%")


# ── FORECAST FUNCTION ─────────────────────────────────────────────────────────

def forecast_month(tgt, pred_yr, pred_mo):
    mdl = final[tgt]
    use_log = mdl['use_log']
    ser     = mdl['ser']
    n       = mdl['n']
    w       = mdl['weights']
    ext_fc  = ext_fc_arr.get(pred_mo, ext_fc_arr[PRED_MONTHS[0]])
    steps   = (pred_yr - last_yr)*12 + (pred_mo - last_mo)
    h       = max(1, steps)

    fc = {}

    # Kalman
    try:
        kf = mdl['kalman']
        if kf:
            p = float(kf.forecast(h)[-1])
            fc['Kalman'] = max(0, np.expm1(p) if use_log else p)
    except: pass

    # HW
    try:
        if mdl['hw']:
            p = float(mdl['hw'].forecast(h).iloc[-1])
            fc['HW'] = max(0, np.expm1(p) if use_log else p)
    except: pass

    # SARIMAX
    try:
        if mdl['sarima']:
            ext_fc_mat = np.tile(ext_fc, (h,1))
            p = float(mdl['sarima'].get_forecast(h, exog=ext_fc_mat).predicted_mean.iloc[-1])
            fc['SARIMAX'] = max(0, np.expm1(p) if use_log else p)
    except: pass

    # STL
    try:
        if mdl['stl']:
            stl_c = mdl['stl']
            tn_idx = n + h - 1
            tn = np.polyval(stl_c['tp'], tn_idx) + stl_c['sp'][tn_idx%6]
            Xi = np.array([[tn_idx, np.sin(2*np.pi*tn_idx/6), np.cos(2*np.pi*tn_idx/6)] + list(ext_fc[:3])])
            rn = float(stl_c['ridge'].predict(stl_c['sc'].transform(Xi))[0])
            p  = tn + rn
            fc['STL'] = max(0, np.expm1(p) if use_log else p)
    except: pass

    # Prophet
    try:
        if mdl['prophet'] and w.get('Prophet',0) > 0.05:
            pm = mdl['prophet']
            fut = pm.make_future_dataframe(periods=h, freq='MS')
            p_val = float(pm.predict(fut)['yhat'].iloc[-1])
            fc['Prophet'] = max(0, p_val)
    except: pass

    # Geometric mean ensemble
    pl, wl = [], []
    for k, v in fc.items():
        wt = w.get(k, 0)
        if wt > 0 and v > 0:
            pl.append(v); wl.append(wt)

    if pl:
        wl = np.array(wl) / sum(wl)
        ens = float(np.exp(np.dot(wl, np.log(np.maximum(pl, 1e-10)))))
    else:
        ens = float(ser.mean())
    ens = max(0, ens)

    # [v25] Bias correction (global only, no per-month leakage)
    ens = ens * mdl['bias_cf']

    # [v25] YoY blending — SANGAT terbatas (max 15%)
    # Hanya jika model tidak akurat DAN YoY data tersedia
    yoy_mask = (monthly['bulan'] == pred_mo) & (monthly['tahun'] == pred_yr - 1)
    if yoy_mask.any() and wf[tgt]['mean'] > 5.0:
        yoy_val = float(monthly.loc[yoy_mask, mdl['col']].values[0])
        growth = YOY_FREQ if tgt=='frequency' else (YOY_SEV if tgt=='severity' else 0.08)
        fc_yoy = yoy_val * (1 + growth)
        yoy_w = min(0.15, (wf[tgt]['mean'] - 5.0) * 0.02)  # max 15%
        ens = (1-yoy_w)*ens + yoy_w*fc_yoy

    # [v25] External seasonal — MINIMAL WEIGHT (0.15)
    seas_idx = SEASONAL_IDX.get(pred_mo, 1.0)
    ens = ens * (0.85 + 0.15 * seas_idx)  # max ±15% adjustment

    return ens


print("\n[7] PREDIKSI 2025 (Agt-Des)...")

raw = {
    'frequency': {m: forecast_month('frequency', PRED_YEAR, m) for m in PRED_MONTHS},
    'severity':  {m: forecast_month('severity',  PRED_YEAR, m) for m in PRED_MONTHS},
    'total':     {m: forecast_month('total',      PRED_YEAR, m) for m in PRED_MONTHS},
}

# Anchor & floor protection
for m in PRED_MONTHS:
    # Frequency floor
    f_floor = max(FREQ_3M * 0.80, float(monthly['frequency'].iloc[-1]) * 0.75)
    raw['frequency'][m] = np.clip(max(raw['frequency'][m], f_floor), F5*0.6, F95*1.5)

    # Severity floor
    s_floor = SEV_6M * 0.88
    raw['severity'][m] = np.clip(max(raw['severity'][m], s_floor), S5*0.7, S95*1.5)

    # Total floor
    raw['total'][m] = np.clip(raw['total'][m], T5*0.6, T95*1.6)


# [v25] F×S enforcement — menggunakan GLOBAL_CF yang konsisten
# Reliability-based weight: lower WF MAPE = more reliable F×S vs direct total
rel_f  = 1.0 / (wf['frequency']['mean']**2 + 1e-6)
rel_s  = 1.0 / (wf['severity']['mean']**2  + 1e-6)
rel_t  = 1.0 / (wf['total']['mean']**2     + 1e-6)
fs_w   = (rel_f * rel_s) / (rel_f * rel_s + rel_t)
fs_w   = float(np.clip(fs_w, 0.50, 0.90))  # min 50% F×S, max 90%
print(f"\n  F×S weight: {fs_w:.2f} (freq MAPE={wf['frequency']['mean']:.1f}% sev MAPE={wf['severity']['mean']:.1f}%)")

print(f"\n  {'Mo':<5} {'Freq':>8} {'Sev(M)':>9} {'Total(B)':>10} {'F×S(B)':>9} {'w':>5} {'Final(B)':>10}")
print(f"  {'-'*60}")
for m in PRED_MONTHS:
    f_ = raw['frequency'][m]; s_ = raw['severity'][m]; t_ = raw['total'][m]
    t_fs = f_ * s_ * GLOBAL_CF
    t_final = fs_w * t_fs + (1 - fs_w) * t_
    t_final = np.clip(max(t_final, T5*0.6), 0, T95*1.6)
    raw['total'][m] = t_final
    print(f"  {m:<5} {f_:>8,.0f} {s_/1e6:>8.2f}M {t_/1e9:>9.3f}B {t_fs/1e9:>8.3f}B {fs_w:>4.2f} {t_final/1e9:>9.3f}B")


print("\n[8] BUILD SUBMISSION...")
rows = []
for m in PRED_MONTHS:
    ms = f'2025_{m:02d}'
    rows.append({'id':f'{ms}_Claim_Frequency','value': raw['frequency'][m]})
    rows.append({'id':f'{ms}_Claim_Severity', 'value': raw['severity'][m]})
    rows.append({'id':f'{ms}_Total_Claim',    'value': raw['total'][m]})

sub = pd.DataFrame(rows)
sub['value'] = pd.to_numeric(sub['value'], errors='coerce').fillna(0).clip(0)
sample_ids = df_sample['id'].tolist()
sub_dict   = dict(zip(sub['id'], sub['value']))
sub = pd.DataFrame({'id': sample_ids, 'value': [sub_dict.get(s, 0.0) for s in sample_ids]})
sub['value'] = sub['value'].round(4)
assert sub['value'].sum() > 0
sub.to_csv('submission.csv', index=False)
print(sub.to_string(index=False))


# ── SIMULASI 5-MONTH HOLDOUT (proper, no leakage) ────────────────────────────

def simulate_holdout():
    print("\n" + "═"*65)
    print("  SIMULASI 5-MONTH HOLDOUT v25 (strict no leakage)")
    print("═"*65)

    n_hold = 5
    if N <= n_hold * 2:
        print("  Data terlalu sedikit."); return

    tr = monthly.iloc[:-n_hold].copy().reset_index(drop=True)
    te = monthly.iloc[-n_hold:].copy().reset_index(drop=True)
    n_tr = len(tr)

    # [v25] Recalculate CF dari training only (ini yang benar!)
    tr['fs_w'] = tr['frequency'] * tr['severity_w']
    tr['cf']   = tr['total'] / (tr['fs_w'] + 1e-10)
    cf_train   = float(tr['cf'].median())

    ext_tr = np.array([[ext_row(int(r.tahun),int(r.bulan),base_yr,base_mo)[k] for k in EXT_KEYS]
                        for _,r in tr.iterrows()])
    ext_te = np.array([[ext_row(int(r.tahun),int(r.bulan),base_yr,base_mo)[k] for k in EXT_KEYS]
                        for _,r in te.iterrows()])

    sim_pred = {'frequency':[], 'severity':[], 'total':[]}

    for key, col, use_log in [('frequency','frequency',False),
                                ('severity','severity_w',True),
                                ('total','total',True)]:
        ser_tr = tr[col].values.copy()
        yf_tr  = np.log1p(np.maximum(ser_tr,0)) if use_log else ser_tr.copy()

        # Refit models on training only
        fc_kf_h = np.full(n_hold, ser_tr.mean())
        fc_hw_h = np.full(n_hold, ser_tr.mean())
        fc_sa_h = np.full(n_hold, ser_tr.mean())

        try:
            kft = KalmanTrend(phi=final[key]['kalman'].phi if final[key]['kalman'] else 0.95,
                              q_l=final[key]['kalman'].q_l if final[key]['kalman'] else 0.01,
                              q_t=final[key]['kalman'].q_t if final[key]['kalman'] else 0.001)
            kft.fit(yf_tr, optimize=False)
            fc_log = kft.forecast(n_hold)
            fc_kf_h = np.maximum(0, np.expm1(fc_log) if use_log else fc_log)
        except: pass

        try:
            phi_h = final[key]['hw_phi']
            m = ExponentialSmoothing(yf_tr, trend='add', damped_trend=(phi_h<1.0),
                                      initialization_method='estimated')
            kw = {'optimized':True,'use_brute':False}
            if phi_h < 1.0: kw['damping_trend'] = phi_h
            fh = m.fit(**kw).forecast(n_hold).values
            fc_hw_h = np.maximum(0, np.expm1(fh) if use_log else fh)
        except: pass

        try:
            ord_ = final[key]['sarima_ord']
            m = SARIMAX(yf_tr, exog=ext_tr, order=ord_,
                        enforce_stationarity=False, enforce_invertibility=False)
            fs = m.fit(disp=False, maxiter=50)
            fh = fs.forecast(n_hold, exog=ext_te).values
            fc_sa_h = np.maximum(0, np.expm1(fh) if use_log else fh)
        except: pass

        w = final[key]['weights']
        bias_cf = final[key]['bias_cf']

        for h_i in range(n_hold):
            pl, wl = [], []
            for mn, pv in [('Kalman',fc_kf_h[h_i]),('HW',fc_hw_h[h_i]),('SARIMAX',fc_sa_h[h_i])]:
                wt = w.get(mn, 0)
                if wt > 0 and pv > 0: pl.append(pv); wl.append(wt)
            if pl:
                wl = np.array(wl)/sum(wl)
                pe = float(np.exp(np.dot(wl, np.log(np.maximum(pl, 1e-10)))))
            else:
                pe = float(ser_tr.mean())
            # [v25] No external seasonal inflation here — minimal adjustments only
            pe = max(0, pe * bias_cf)
            sim_pred[key].append(pe)

    # F×S with training CF
    for i in range(n_hold):
        f_ = sim_pred['frequency'][i]
        s_ = sim_pred['severity'][i]
        t_ = sim_pred['total'][i]
        t_fs = f_ * s_ * cf_train
        sim_pred['total'][i] = fs_w * t_fs + (1-fs_w) * t_

    actuals = {
        'frequency': te['frequency'].values,
        'severity':  te['severity_w'].values,
        'total':     te['total'].values,
    }

    print(f"\n  {'Hold':<7} {'Target':<12} {'Aktual':>14} {'Prediksi':>14} {'Err%':>8}")
    print("  " + "-"*60)
    all_errs = []; tgt_mapes = {}
    for key in ['frequency','severity','total']:
        errs = []
        for i in range(n_hold):
            a, p = actuals[key][i], sim_pred[key][i]
            err = abs(a-p)/max(abs(a),1e-5)*100
            errs.append(err); all_errs.append(err)
            if key=='frequency':   a_d=f"{a:,.0f}"; p_d=f"{p:,.0f}"
            elif key=='severity':  a_d=f"{a/1e6:,.2f}M"; p_d=f"{p/1e6:,.2f}M"
            else:                   a_d=f"{a/1e9:,.3f}B"; p_d=f"{p/1e9:,.3f}B"
            print(f"  H-{i+1:<5} {key:<12} {a_d:>14} {p_d:>14} {err:>7.2f}%")
        tgt_mapes[key] = np.mean(errs)
        print("  " + "-"*60)

    avg = np.mean(all_errs)
    teman = {'frequency':5.87,'severity':2.44,'total':7.12}
    teman_avg = np.mean(list(teman.values()))

    print(f"\n  {'Target':<12} {'Teman v16':>10} {'Kita v25':>10} {'Delta':>8}")
    print("  " + "-"*45)
    for k,l in [('frequency','Frequency'),('severity','Severity'),('total','Total')]:
        d = tgt_mapes[k]-teman[k]
        r = "✅" if d<0 else "❌"
        print(f"  {l:<12} {teman[k]:>9.2f}% {tgt_mapes[k]:>9.2f}% {d:>+7.2f}% {r}")
    print("  " + "-"*45)
    d_avg = avg-teman_avg
    r_avg = "✅" if d_avg<0 else "❌"
    print(f"  {'AVG':<12} {teman_avg:>9.2f}% {avg:>9.2f}% {d_avg:>+7.2f}% {r_avg}")

    print(f"\n  ★ ESTIMASI SKOR v25 : {avg:.4f}%")
    print(f"  ★ ESTIMASI SKOR v16 : {teman_avg:.4f}%")
    print(f"  ★ SELISIH           : {d_avg:+.4f}% {r_avg}")
    print("  NOTE: Gap simulasi vs Kaggle idealnya < 1.5%. Jika masih > 2%,")
    print("        berarti ada regime change di bulan prediksi vs training.")
    print("═"*65)

    # Simple forecast visualization
    try:
        fig, axes = plt.subplots(1, 3, figsize=(16, 5))
        fig.suptitle('v25 — 5-Month Holdout Simulation', fontweight='bold')
        for i, (key, col, div, unit) in enumerate([
            ('frequency','frequency',1,'klaim'),
            ('severity','severity_w',1e6,'Juta IDR'),
            ('total','total',1e9,'Miliar IDR'),
        ]):
            ax = axes[i]
            hist_y = monthly[col].values / div
            ax.plot(range(N), hist_y, color='gray', lw=1.5, marker='o', ms=3, alpha=0.7)
            n_val = N - n_hold
            ax.plot(range(n_val, N), actuals[key]/div, color='black', lw=2.5,
                    marker='o', ms=6, label='Actual')
            ax.plot(range(n_val, N), np.array(sim_pred[key])/div, color=COLORS[i],
                    lw=2.5, marker='D', ms=6, ls='--', label=f'Pred (MAPE={tgt_mapes[key]:.1f}%)')
            ax.axvline(n_val, color='red', ls='--', lw=1.5, alpha=0.7)
            ax.set_title(key, fontweight='bold'); ax.set_ylabel(unit); ax.legend(fontsize=8)
        plt.tight_layout()
        plt.savefig('holdout_v25.png', dpi=90, bbox_inches='tight')
        plt.close()
        print("  ✓ holdout_v25.png tersimpan")
    except: plt.close('all')


simulate_holdout()

print("\n" + "="*70)
print("  v25 SELESAI — submission.csv siap upload")
print(f"  WF-MAPE: Freq={wf['frequency']['mean']:.2f}% Sev={wf['severity']['mean']:.2f}% Total={wf['total']['mean']:.2f}%")
print("="*70)

  DSC MCF ITB v25.0 — CLEAN PIPELINE, NO LEAKAGE
  Target: Close gap simulasi vs Kaggle

[1] LOADING DATA...
  Klaim: (4627, 13) | Polis: (4096, 6)

[2] PREPROCESSING...
  ✓ df_paid: 4,627 baris | Paid: ['PAID']
  Winsor [2%,98%]: [288,552, 446,068,114]

[3] FEATURE ENGINEERING...

[4] AGREGASI BULANAN...
  Monthly data: 19 bulan
  Freq: mean=244 last=264
  Sev_w: mean=48.53M
  Global F×S CF: 1.1268
  YoY: Freq=-13.0% Sev=-11.5%

[5] WALK-FORWARD CV (no leakage)...
  frequency   : WF-MAPE=6.69%  Bias=-3.5%  n=8
  severity    : WF-MAPE=7.40%  Bias=-0.4%  n=8
  total       : WF-MAPE=13.87%  Bias=-3.8%  n=8

[6] BUILDING FINAL MODELS...

  ── frequency ──────────────────────────────────────────────
  [A] Kalman phi=0.850 MAPE=8.79%
  [B] HW phi=0.95 MAPE=inf%


17:47:53 - cmdstanpy - INFO - Chain [1] start processing
17:47:53 - cmdstanpy - INFO - Chain [1] done processing


  [C] SARIMAX(0, 1, 2) MAPE=0.82%
  [D] STL+Ridge MAPE=6.39%
  [E] Prophet MAPE=6.12%
  Weights: Kalman=0.008 HW=0.000 SARIMAX=0.959 STL=0.016 Prophet=0.017
  Bias CF: 0.9648

  ── severity ──────────────────────────────────────────────
  [A] Kalman phi=0.850 MAPE=6.36%
  [B] HW phi=0.95 MAPE=inf%


17:47:56 - cmdstanpy - INFO - Chain [1] start processing
17:47:56 - cmdstanpy - INFO - Chain [1] done processing


  [C] SARIMAX(0, 0, 2) MAPE=2.58%
  [D] STL+Ridge MAPE=5.63%
  [E] Prophet MAPE=2.44%
  Weights: Kalman=0.066 HW=0.000 SARIMAX=0.400 STL=0.084 Prophet=0.450
  Bias CF: 0.9956

  ── total ──────────────────────────────────────────────
  [A] Kalman phi=0.850 MAPE=8.53%
  [B] HW phi=0.95 MAPE=inf%


17:47:58 - cmdstanpy - INFO - Chain [1] start processing
17:47:58 - cmdstanpy - INFO - Chain [1] done processing


  [C] SARIMAX(2, 0, 1) MAPE=2.21%
  [D] STL+Ridge MAPE=5.59%
  [E] Prophet MAPE=7.65%
  Weights: Kalman=0.051 HW=0.000 SARIMAX=0.765 STL=0.119 Prophet=0.064
  Bias CF: 0.9622

  Summary WF-MAPE: Freq=6.69% Sev=7.40% Total=13.87%

[7] PREDIKSI 2025 (Agt-Des)...

  F×S weight: 0.50 (freq MAPE=6.7% sev MAPE=7.4%)

  Mo        Freq    Sev(M)   Total(B)    F×S(B)     w   Final(B)
  ------------------------------------------------------------
  8          218    48.83M    13.345B   11.989B 0.50    12.667B
  9          241    48.76M    12.834B   13.231B 0.50    13.033B
  10         225    48.60M    11.841B   12.328B 0.50    12.084B
  11         238    48.76M    12.160B   13.081B 0.50    12.620B
  12         215    49.64M    11.743B   12.049B 0.50    11.896B

[8] BUILD SUBMISSION...
                     id        value
2025_08_Claim_Frequency 2.178871e+02
 2025_08_Claim_Severity 4.883287e+07
    2025_08_Total_Claim 1.266710e+10
2025_09_Claim_Frequency 2.408366e+02
 2025_09_Claim_Severity 4.875

In [54]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from statsmodels.tsa.api import ExponentialSmoothing

DATA_KLAIM  = r"Data_Klaim (1).csv"
DATA_SAMPLE = r"sample_submission.csv"

def get_col(df, keywords):
    cols = {c.lower().strip(): c for c in df.columns}
    for kw in keywords:
        for cl, c_orig in cols.items():
            if kw.lower() in cl: return c_orig
    return None

def process_data():
    df = pd.read_csv(DATA_KLAIM, encoding='latin-1', sep=None, engine='python')
    
    c_target = get_col(df, ['nominal klaim yang disetujui', 'approved'])
    c_status = get_col(df, ['status klaim', 'status'])
    c_masuk  = get_col(df, ['tanggal pasien masuk', 'tgl masuk'])
    c_keluar = get_col(df, ['tanggal pasien keluar', 'tgl keluar'])
    c_bayar  = get_col(df, ['tanggal pembayaran', 'tgl bayar'])
    
    df[c_target] = pd.to_numeric(df[c_target].astype(str).str.replace(r'[^\d.-]', '', regex=True), errors='coerce')
    df = df[df[c_status].astype(str).str.contains('PAID|DIBAYAR|APPROV|SETTLED', case=False, na=False)].copy()
    
    df['tgl_keluar'] = pd.to_datetime(df[c_keluar], errors='coerce', dayfirst=True)
    df['tgl_bayar']  = pd.to_datetime(df[c_bayar], errors='coerce', dayfirst=True)
    df['tgl_masuk']  = pd.to_datetime(df[c_masuk], errors='coerce', dayfirst=True)
    
    # Imputasi tanggal bayar kosong dengan tanggal keluar + median lag (jeda standar asuransi)
    null_bayar = df['tgl_bayar'].isna()
    median_lag = (df['tgl_bayar'] - df['tgl_keluar']).dt.days.median()
    median_lag = median_lag if pd.notna(median_lag) else 14
    df.loc[null_bayar, 'tgl_bayar'] = df.loc[null_bayar, 'tgl_keluar'] + pd.Timedelta(days=median_lag)
    df['tgl_analisis'] = df['tgl_bayar'].fillna(df['tgl_masuk'])
    
    df = df.dropna(subset=['tgl_analisis', c_target])
    df['year_month'] = df['tgl_analisis'].dt.to_period('M')
    df['bulan'] = df['tgl_analisis'].dt.month
    df['tahun'] = df['tgl_analisis'].dt.year
    
    # ─── PENERAPAN EDA ───
    # 1. Buang data bulan Agustus yang tidak lengkap (hanya 50%)
    df = df[~((df['bulan'] == 8) & (df['tahun'] == 2025))]
    
    # 2. Capping Anomali Januari di persentil 95
    p95 = df[c_target].quantile(0.95)
    jan_mask = df['bulan'] == 1
    df.loc[jan_mask, c_target] = df.loc[jan_mask, c_target].clip(upper=p95)
    
    monthly = df.groupby('year_month').agg(
        frequency=(c_target, 'count'),
        severity=(c_target, 'mean'),
        total_claim=(c_target, 'sum')
    ).reset_index().sort_values('year_month').reset_index(drop=True)
    
    monthly['fs_calc'] = monthly['frequency'] * monthly['severity']
    monthly['cf_ratio'] = monthly['total_claim'] / (monthly['fs_calc'] + 1e-9)
    
    return monthly

def forecast_hw(series, h):
    y = np.array(series, dtype=float)
    try:
        model = ExponentialSmoothing(y, trend='add', damped_trend=True, initialization_method="estimated")
        return np.maximum(0, np.array(model.fit(optimized=True).forecast(h)))
    except:
        return np.full(h, np.mean(y[-3:]))

def run_simulation(df, n_holdout=5):
    if len(df) <= n_holdout * 2: 
        print("Data tidak cukup untuk simulasi.")
        return
    
    train = df.iloc[:-n_holdout]
    test = df.iloc[-n_holdout:]
    
    p_freq = forecast_hw(train['frequency'], n_holdout)
    p_sev = forecast_hw(train['severity'], n_holdout)
    cf = train['cf_ratio'].tail(6).median()
    p_tot = p_freq * p_sev * cf
    
    errs = []
    print(f"\n{'Holdout':<10} {'Target':<15} {'Aktual':>15} {'Prediksi':>15} {'Error%':>8}")
    print("-" * 70)
    for i in range(n_holdout):
        e_f = abs(test['frequency'].iloc[i] - p_freq[i]) / max(test['frequency'].iloc[i], 1) * 100
        e_s = abs(test['severity'].iloc[i] - p_sev[i]) / max(test['severity'].iloc[i], 1) * 100
        e_t = abs(test['total_claim'].iloc[i] - p_tot[i]) / max(test['total_claim'].iloc[i], 1) * 100
        errs.extend([e_f, e_s, e_t])
        
        print(f"Bulan-{i+1:<4} {'frequency':<15} {test['frequency'].iloc[i]:>15.0f} {p_freq[i]:>15.0f} {e_f:>7.2f}%")
        print(f"Bulan-{i+1:<4} {'severity':<15} {test['severity'].iloc[i]/1e6:>14.2f}M {p_sev[i]/1e6:>14.2f}M {e_s:>7.2f}%")
        print(f"Bulan-{i+1:<4} {'total_claim':<15} {test['total_claim'].iloc[i]/1e9:>14.3f}B {p_tot[i]/1e9:>14.3f}B {e_t:>7.2f}%")
        print("-" * 70)
        
    print(f"ESTIMASI SKOR LOKAL (Holt-Winters + EDA) : {np.mean(errs):.4f}%\n")

def create_submission(df):
    h = 5
    pred_months = [8, 9, 10, 11, 12]
    
    f_pred = forecast_hw(df['frequency'], h)
    s_pred = forecast_hw(df['severity'], h)
    cf = df['cf_ratio'].tail(6).median()
    t_pred = f_pred * s_pred * cf
    
    rows = []
    for i, m in enumerate(pred_months):
        ms = f'2025_{m:02d}'
        rows.append({'id': f'{ms}_Claim_Frequency', 'value': f_pred[i]})
        rows.append({'id': f'{ms}_Claim_Severity', 'value': s_pred[i]})
        rows.append({'id': f'{ms}_Total_Claim', 'value': t_pred[i]})
        
    sub_df = pd.DataFrame(rows)
    sample = pd.read_csv(DATA_SAMPLE, encoding='latin-1', sep=None, engine='python')
    sub_dict = dict(zip(sub_df['id'], sub_df['value']))
    
    final_sub = pd.DataFrame({
        'id': sample['id'],
        'value': [sub_dict.get(sid, 0.0) for sid in sample['id']]
    })
    
    final_sub['value'] = final_sub['value'].round(4)
    final_sub.to_csv('submission.csv', index=False)
    print("File 'submission.csv' berhasil dibuat dan siap untuk di-submit.")

if __name__ == "__main__":
    print("Memproses data dan menerapkan EDA...")
    df_monthly = process_data()
    run_simulation(df_monthly)
    create_submission(df_monthly)

Memproses data dan menerapkan EDA...

Holdout    Target                   Aktual        Prediksi   Error%
----------------------------------------------------------------------
Bulan-1    frequency                   233             217    6.69%
Bulan-1    severity                 49.61M          64.73M   30.47%
Bulan-1    total_claim             11.559B         14.072B   21.74%
----------------------------------------------------------------------
Bulan-2    frequency                    99             217  119.68%
Bulan-2    severity                 53.20M          65.75M   23.58%
Bulan-2    total_claim              5.267B         14.299B  171.47%
----------------------------------------------------------------------
Bulan-3    frequency                    96             218  126.60%
Bulan-3    severity                 64.81M          66.76M    3.01%
Bulan-3    total_claim              6.222B         14.522B  133.42%
---------------------------------------------------------------------